[Mohit Saharan](https://linkedin.com/in/msaharan), 20260512, P22, v2

Apache 2.0 License (see github.com/msaharan/dsaiengineering/LICENSE)

# Tactical Asset Allocation with TabPFN, TabICL, and XGBoost - 3

This notebook follows up on the [P21 tactical asset-allocation workflow](https://github.com/msaharan/dsaiengineering/blob/main/blog/20260511-tabpfn-tabicl-tactical-asset-allocation-2.md) by addressing the highest-value shortcomings left open in that post. P21 made the universe broader, moved to an identity-ablated main feature set, changed the target to next-open-to-next-open, and added richer portfolio diagnostics. The useful next step is not another headline backtest; it is making the testbench more explicit where P21 was incomplete.

The default workflow keeps the 25-ETF universe and identity-ablated main feature policy, but adds three concrete upgrades. First, it fixes the liquidity/spread/impact proxy path by constructing rolling dollar-volume and high-low range features directly from the downloaded OHLCV data; unlike P21, these liquidity proxies are also part of the predictive feature matrix and are disclosed as a feature-set change. Second, it keeps the old alternative-objective audit but also adds objective-specific reruns, calibration-window diagnostics, chronological fixed-model CV evidence, monthly rank checks, and score-to-portfolio diagnostics for benchmark-relative and multi-horizon labels. Third, it adds GPU-accelerated XGBoost and direct TabPFN feature-policy sensitivity, so the feature-policy question is no longer limited to lightweight CPU Logistic Regression diagnostics.

Current run scope: TabPFN and GPU XGBoost remain enabled for the main one-month target; TabICL code remains guarded with `RUN_DIRECT_TABICL = False` because the prior full Kaggle run hit RAM limits. Objective-specific reruns are configured as focused diagnostics, not replacements for the main allocation experiment. The default objective-specific rerun uses fixed and target-specific searched GPU-accelerated XGBoost plus direct TabPFN; CPU Logistic Regression is retained only as an optional disabled benchmark path. TabICL remains a known unresolved empirical comparison in this run by design. This v2 notebook separates headline allocation outputs from feature-sensitivity diagnostics and replaces overlapping multi-horizon portfolio compounding with horizon-aware score-to-return diagnostics.

The objective remains narrow: given features available at a monthly signal date, score which assets are more likely to rank in the top group by next-month return within the configured universe. This is an educational research workflow, not investment advice and not a claim that any score is suitable for live trading without separate production-grade data, execution, risk, and compliance review. Known caveats for this run are written into `method_contract.md`, `leakage_checks.csv`, and `run_closeout.md`.

The notebook writes reviewable CSV, TXT, and PNG artifacts to `tabpfn_tabicl_tactical_asset_allocation_20260512_v2_outputs/` and creates a ZIP archive at the end. Use those files for post-run review because notebook editor output can be truncated.


## How to Run This Notebook on Kaggle

1. Import this notebook into Kaggle.
2. Enable GPU acceleration. The intended environment is Kaggle with two T4 GPUs.
3. Turn on Internet access. The notebook downloads packages, public market data, public macro data, and model checkpoints.
4. Add and attach a Kaggle secret named `TABPFN_TOKEN` after accepting the Prior Labs TabPFN-2.6 license. The setup cell pins the local package line used by the prior run, requests `ModelVersion.V2_6`, and performs a small model-load preflight when TabPFN paths are enabled, so stale or unauthorized tokens fail before expensive model sections.
5. `HF_TOKEN` is not required for the default run because TabICL is disabled. Add it only if you later re-enable TabICL and need Hugging Face checkpoint access.

## 0. Imports, Secrets, and Configuration

- Quickly run complete notebook to test if everything works: `FAST_MODE = True`
- Run full notebook in production mode: `FAST_MODE = False` and `MEMORY_EFFICIENT_MODE = False`
- Run full notebook in production mode if you enounter insufficient memory errors: `FAST_MODE = False` and `MEMORY_EFFICIENT_MODE = True`

In [1]:
%%time
# Kaggle / Colab setup.
# Run this cell once. If imports still fail, restart the notebook session and continue below.

import subprocess
import sys

PYPI_PACKAGES = [
    "xgboost>=2.0.0",
    "rich",
    "yfinance>=0.2.40",
    "tabpfn==7.1.1",
    "tabicl",
    "cupy-cuda12x",
    "tqdm",
    "scipy",
]

subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PYPI_PACKAGES], check=True)

import gc
import importlib.metadata as importlib_metadata
import json
import math
import os
from pathlib import Path
import platform
import subprocess
import time
import warnings
from traceback import format_exception_only
from urllib.parse import urlencode

import cupy as cp
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
import sklearn
import torch
import yfinance as yf

from IPython.display import display
from rich.console import Console
from scipy.stats import loguniform, randint, spearmanr, uniform
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, precision_recall_curve, roc_auc_score
from sklearn.model_selection import ParameterSampler
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="The `cv='prefit'` option is deprecated", category=FutureWarning)

console = Console()
SEED = 42
np.random.seed(SEED)

DATA_START_DATE = "2006-01-01"
DATA_END_DATE = "2026-05-12"  # yfinance end date is exclusive.
FRED_DOWNLOAD_START_DATE = "1990-01-01"

ASSET_TICKERS = {
    "SPY": "US large-cap equity",
    "QQQ": "US growth equity",
    "DIA": "US large-cap blue-chip equity",
    "IWM": "US small-cap equity",
    "EFA": "Developed ex-US equity",
    "EEM": "Emerging-market equity",
    "TLT": "Long-duration Treasury",
    "IEF": "Intermediate Treasury",
    "SHY": "Short-duration Treasury",
    "LQD": "Investment-grade credit",
    "HYG": "High-yield credit",
    "GLD": "Gold",
    "SLV": "Silver",
    "DBC": "Broad commodities",
    "VNQ": "US REIT",
    "IYR": "US real estate",
    "XLB": "US materials sector",
    "XLE": "US energy sector",
    "XLF": "US financials sector",
    "XLI": "US industrials sector",
    "XLK": "US technology sector",
    "XLP": "US consumer staples sector",
    "XLU": "US utilities sector",
    "XLV": "US healthcare sector",
    "XLY": "US consumer discretionary sector",
}
ASSET_GROUPS = {
    "SPY": "equity_us",
    "QQQ": "equity_us",
    "DIA": "equity_us",
    "IWM": "equity_us",
    "EFA": "equity_developed",
    "EEM": "equity_em",
    "TLT": "rates",
    "IEF": "rates",
    "SHY": "rates",
    "LQD": "credit",
    "HYG": "credit",
    "GLD": "commodity",
    "SLV": "commodity",
    "DBC": "commodity",
    "VNQ": "real_estate",
    "IYR": "real_estate",
    "XLB": "sector_us",
    "XLE": "sector_us",
    "XLF": "sector_us",
    "XLI": "sector_us",
    "XLK": "sector_us",
    "XLP": "sector_us",
    "XLU": "sector_us",
    "XLV": "sector_us",
    "XLY": "sector_us",
}
ASSET_RISK_BUCKET = {
    "SPY": 3,
    "QQQ": 4,
    "DIA": 3,
    "IWM": 4,
    "EFA": 4,
    "EEM": 4,
    "TLT": 2,
    "IEF": 1,
    "SHY": 1,
    "LQD": 2,
    "HYG": 3,
    "GLD": 3,
    "SLV": 4,
    "DBC": 4,
    "VNQ": 4,
    "IYR": 4,
    "XLB": 4,
    "XLE": 4,
    "XLF": 4,
    "XLI": 4,
    "XLK": 4,
    "XLP": 3,
    "XLU": 3,
    "XLV": 3,
    "XLY": 4,
}

FAST_MODE = True
if FAST_MODE:
    ASSET_TICKERS = {ticker: ASSET_TICKERS[ticker] for ticker in ["SPY", "QQQ", "IWM", "TLT", "IEF", "GLD", "EFA", "EEM", "XLE", "XLF"]}

PORTFOLIO_TOP_K = 5
if FAST_MODE:
    PORTFOLIO_TOP_K = 3
TARGET_TOP_K = min(PORTFOLIO_TOP_K, len(ASSET_TICKERS))
TRANSACTION_COST_BPS = 5.0
TRANSACTION_COST_SENSITIVITY_BPS = [0.0, 5.0, 10.0, 25.0, 50.0]
TURNOVER_CONSTRAINT_CAPS = [1.0, 0.5]
TAX_DRAG_SENSITIVITY_BPS_PER_TURNOVER = [0.0, 10.0, 25.0, 50.0]
MARKET_IMPACT_NOTIONAL_USD = 1_000_000.0
RISK_WEIGHTED_TOP_K_MAX_WEIGHT = 0.35
EX_ANTE_RISK_LOOKBACK_MONTHS = 36
EX_ANTE_RISK_MIN_MONTHS = 18
EXECUTION_RETURN_MODE = "next_open_to_next_open"  # options: close_to_close, next_open_to_next_open
FEATURE_SET_VARIANT = "identity_ablated"  # main run options: full, ticker_ablated, identity_ablated
FEATURE_VARIANT_SENSITIVITY_ENABLED = True
FEATURE_SENSITIVITY_VARIANTS = ["full", "ticker_ablated", "identity_ablated", "metadata_only", "strict_time_series"]
MODEL_FAMILY_FEATURE_SENSITIVITY_ENABLED = True
MODEL_FAMILY_FEATURE_SENSITIVITY_VARIANTS = list(FEATURE_SENSITIVITY_VARIANTS)
RUN_FEATURE_SENSITIVITY_LOGISTIC = False
RUN_FEATURE_SENSITIVITY_TABPFN = True
RUN_FEATURE_SENSITIVITY_XGBOOST_LIGHT = True
FEATURE_SENSITIVITY_XGBOOST_N_ESTIMATORS = 600
MULTI_HORIZON_MONTHS = [3, 6]
BENCHMARK_ASSET = "SPY"
OBJECTIVE_SPECIFIC_RETRAINING_ENABLED = True
OBJECTIVE_SPECIFIC_TARGETS = ["benchmark_outperform_1m", "top_k_next_3m", "top_k_next_6m"]
RUN_OBJECTIVE_SPECIFIC_LOGISTIC = False
RUN_OBJECTIVE_SPECIFIC_TABPFN = True
RUN_OBJECTIVE_SPECIFIC_XGBOOST_LIGHT = True
RUN_OBJECTIVE_SPECIFIC_XGBOOST_SEARCH = True
OBJECTIVE_SPECIFIC_XGBOOST_N_ESTIMATORS = 600
OBJECTIVE_SPECIFIC_XGBOOST_SEARCH_ITERATIONS = 48
MONTHS_PER_YEAR = 12

CONTEXT_END_DATE = "2011-12-31"
TUNING_END_DATE = "2017-12-31"
CALIBRATION_END_DATE = "2019-12-31"
HOLDOUT_START_DATE = "2020-01-01"

FEATURE_MAX_MISSING_RATE = 0.35
FEATURE_MIN_SELECTION_OBSERVATIONS = 36
RETURN_WINDOWS_DAYS = [21, 63, 126, 252]
VOL_WINDOWS_DAYS = [21, 63, 126]
DRAWDOWN_WINDOWS_DAYS = [63, 126, 252]
TREND_WINDOWS_DAYS = [63, 126, 200]
CROSS_ASSET_TICKERS_FOR_FEATURES = list(ASSET_TICKERS)

N_TFM_ESTIMATORS = 8
MEMORY_EFFICIENT_MODE = False
MEMORY_EFFICIENT_N_TFM_ESTIMATORS = 4
SAVE_FULL_PREDICTION_SCORES = True
XGBOOST_TUNING_ITERATIONS = 160
CLASSICAL_TUNING_CV_SPLITS = 6
MIN_CV_TRAIN_POSITIVES = 30
MIN_CV_VALIDATION_POSITIVES = 8
BOOTSTRAP_ITERATIONS = 400
BOOTSTRAP_CONFIDENCE_LEVEL = 0.95
PREDICTION_CHUNK_SIZE = 8192
TABPFN_AUTH_PREFLIGHT_ENABLED = True
XGBOOST_SEARCH_EARLY_STOPPING_ENABLED = True
XGBOOST_SEARCH_EARLY_STOPPING_ROUNDS = 75

FAST_MODE_N_TFM_ESTIMATORS = 2
FAST_MODE_XGBOOST_TUNING_ITERATIONS = 10
FAST_MODE_CLASSICAL_TUNING_CV_SPLITS = 2
FAST_MODE_BOOTSTRAP_ITERATIONS = 50
FAST_MODE_OBJECTIVE_SPECIFIC_XGBOOST_SEARCH_ITERATIONS = 6
FAST_MODE_MIN_CV_TRAIN_POSITIVES = 8
FAST_MODE_MIN_CV_VALIDATION_POSITIVES = 2

if MEMORY_EFFICIENT_MODE and not FAST_MODE:
    N_TFM_ESTIMATORS = min(N_TFM_ESTIMATORS, MEMORY_EFFICIENT_N_TFM_ESTIMATORS)

if FAST_MODE:
    N_TFM_ESTIMATORS = min(N_TFM_ESTIMATORS, FAST_MODE_N_TFM_ESTIMATORS)
    XGBOOST_TUNING_ITERATIONS = min(XGBOOST_TUNING_ITERATIONS, FAST_MODE_XGBOOST_TUNING_ITERATIONS)
    CLASSICAL_TUNING_CV_SPLITS = min(CLASSICAL_TUNING_CV_SPLITS, FAST_MODE_CLASSICAL_TUNING_CV_SPLITS)
    BOOTSTRAP_ITERATIONS = min(BOOTSTRAP_ITERATIONS, FAST_MODE_BOOTSTRAP_ITERATIONS)
    OBJECTIVE_SPECIFIC_XGBOOST_SEARCH_ITERATIONS = min(OBJECTIVE_SPECIFIC_XGBOOST_SEARCH_ITERATIONS, FAST_MODE_OBJECTIVE_SPECIFIC_XGBOOST_SEARCH_ITERATIONS)
    MIN_CV_TRAIN_POSITIVES = min(MIN_CV_TRAIN_POSITIVES, FAST_MODE_MIN_CV_TRAIN_POSITIVES)
    MIN_CV_VALIDATION_POSITIVES = min(MIN_CV_VALIDATION_POSITIVES, FAST_MODE_MIN_CV_VALIDATION_POSITIVES)

RUN_GPU_XGBOOST = True
RUN_DIRECT_TABPFN = True
# Temporarily disabled so the full notebook can complete while TabICL RAM usage is investigated.
RUN_DIRECT_TABICL = False
RUN_LOGISTIC_REGRESSION_CPU_BENCHMARK = False
EVALUATE_CALIBRATION_VARIANTS = True

XGBOOST_TREE_METHOD = "hist"
XGBOOST_DEVICE = "cuda"
TABICL_CHECKPOINT_VERSION = "tabicl-classifier-v2-20260212.ckpt"
TABPFN_PACKAGE_REQUIREMENT = "tabpfn==7.1.1"
TABPFN_MODEL_VERSION = "V2_6"
TABPFN_MODEL_NOTE = "TabPFN 2.6 explicitly selected via ModelVersion.V2_6 to keep P22 comparable with P21 and the P22 v1 run"

FRED_SERIES = {
    "DGS10": "treasury_10y_yield",
    "DGS2": "treasury_2y_yield",
    "T10Y2Y": "yield_curve_10y_2y",
    "BAMLH0A0HYM2": "high_yield_oas",
    "BAMLC0A0CM": "investment_grade_oas",
    "DFF": "fed_funds_rate",
    "DTB3": "t_bill_3m",
}
FRED_MIN_SELECTION_OBSERVATIONS = FEATURE_MIN_SELECTION_OBSERVATIONS
VIX_CBOE_CSV_URL = "https://cdn.cboe.com/api/global/us_indices/daily_prices/VIX_History.csv"

ARTIFACT_DIR = Path("tabpfn_tabicl_tactical_asset_allocation_20260512_v2_outputs")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
artifact_manifest_rows = []
model_errors = []
model_rows = []
predictions = {}
prediction_frames = []
cuda_memory_snapshots = []

PUBLICATION_MODEL_ORDER = [
    "Rule[12M momentum top-k]",
    "Rule[6M momentum top-k]",
    "Rule[Low volatility top-k]",
    "XGBoost[GPU allocation scorer]",
    "TabPFN[Direct allocation scorer]",
    "TabICL[Direct allocation scorer]",
]
CALIBRATION_MODEL_ORDER = [
    "XGBoost[GPU allocation scorer] Calibration Base",
    "XGBoost[GPU allocation scorer] Calibrated",
    "TabPFN[Direct allocation scorer] Calibration Base",
    "TabICL[Direct allocation scorer] Calibration Base",
]


def register_artifact(path, kind, description=""):
    path = Path(path)
    artifact_manifest_rows.append(
        {
            "path": str(path),
            "filename": path.name,
            "kind": kind,
            "description": description,
            "exists": path.exists(),
            "size_bytes": path.stat().st_size if path.exists() else np.nan,
        }
    )
    return path


def save_artifact_table(table, filename, index=False, description=""):
    output_path = ARTIFACT_DIR / filename
    table.to_csv(output_path, index=index)
    register_artifact(output_path, "table", description)
    print(f"Saved {output_path}")
    return output_path


def save_text_artifact(filename, text, description=""):
    output_path = ARTIFACT_DIR / filename
    output_path.write_text(str(text), encoding="utf-8")
    register_artifact(output_path, "text", description)
    print(f"Saved {output_path}")
    return output_path


def installed_version(*package_names):
    for package_name in package_names:
        try:
            return importlib_metadata.version(package_name)
        except importlib_metadata.PackageNotFoundError:
            continue
    return "not installed"


def safe_string(value):
    if value is None:
        return ""
    if isinstance(value, (list, tuple)):
        return ", ".join(str(item) for item in value)
    return str(value)


def json_ready(value):
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if isinstance(value, np.ndarray):
        return json_ready(value.tolist())
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    return value


def artifact_json(value):
    return json.dumps(json_ready(value), sort_keys=True)


def short_error(exc):
    return "".join(format_exception_only(type(exc), exc)).strip().replace("\n", " ")[:500]


def git_commit_or_unavailable(path):
    path = Path(path)
    if not path.exists():
        return "not available"
    try:
        result = subprocess.run(["git", "-C", str(path), "rev-parse", "HEAD"], capture_output=True, text=True, check=True, timeout=5)
        return result.stdout.strip()
    except Exception:
        return "not a git checkout or unavailable"


def source_provenance_frame():
    cwd = Path.cwd().resolve()
    candidates = {
        "notebook_working_directory": cwd,
        "parent_directory": cwd.parent,
        "tabpfn_sibling_repo": cwd.parent / "tabpfn",
        "tabicl_sibling_repo": cwd.parent / "tabicl",
    }
    return pd.DataFrame([{"Source": name, "Path": str(path), "Git Commit": git_commit_or_unavailable(path)} for name, path in candidates.items()])


def reset_cuda_peak_memory_stats():
    if not torch.cuda.is_available():
        return
    for device_index in range(torch.cuda.device_count()):
        torch.cuda.reset_peak_memory_stats(device_index)


def cuda_memory_snapshot(stage):
    if not torch.cuda.is_available():
        return pd.DataFrame()
    rows = []
    for device_index in range(torch.cuda.device_count()):
        free_bytes, total_bytes = torch.cuda.mem_get_info(device_index)
        rows.append(
            {
                "stage": stage,
                "device_index": device_index,
                "device_name": torch.cuda.get_device_name(device_index),
                "allocated_mb": torch.cuda.memory_allocated(device_index) / 1024**2,
                "reserved_mb": torch.cuda.memory_reserved(device_index) / 1024**2,
                "max_allocated_mb": torch.cuda.max_memory_allocated(device_index) / 1024**2,
                "free_mb": free_bytes / 1024**2,
                "total_mb": total_bytes / 1024**2,
            }
        )
    return pd.DataFrame(rows)


def record_cuda_memory(stage):
    snapshot = cuda_memory_snapshot(stage)
    if len(snapshot) > 0:
        cuda_memory_snapshots.append(snapshot)
    return snapshot


def cleanup_runtime_memory(stage="cleanup"):
    gc.collect()
    try:
        cp.get_default_memory_pool().free_all_blocks()
        cp.get_default_pinned_memory_pool().free_all_blocks()
    except Exception:
        pass
    if torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        except Exception as exc:
            print(f"CUDA cleanup warning at {stage}: {short_error(exc)}")
    return record_cuda_memory(stage)


CUDA_DEVICE_COUNT = torch.cuda.device_count()
if CUDA_DEVICE_COUNT < 1:
    print("CUDA was not detected. Data construction can still run, but default model sections are designed for a GPU runtime.")
    XGBOOST_DEVICE = "cpu"
    RUN_DIRECT_TABPFN = False
    RUN_DIRECT_TABICL = False

TABICL_DEVICE = "cuda:0" if CUDA_DEVICE_COUNT else "cpu"
TABPFN_DEVICE = [f"cuda:{idx}" for idx in range(CUDA_DEVICE_COUNT)] if CUDA_DEVICE_COUNT else "cpu"


secret_load_diagnostics = []


def load_secret(name, aliases=None):
    names = [name] + [alias for alias in (aliases or []) if alias != name]
    for candidate in names:
        value = os.environ.get(candidate)
        if value:
            value = str(value).strip().strip('"').strip("'")
            os.environ[name] = value
            secret_load_diagnostics.append({"Secret": name, "Candidate": candidate, "Source": "environment", "Available": True, "Error": ""})
            return value
    try:
        from kaggle_secrets import UserSecretsClient

        client = UserSecretsClient()
        for candidate in names:
            try:
                value = client.get_secret(candidate)
                if value:
                    value = str(value).strip().strip('"').strip("'")
                    os.environ[name] = value
                    secret_load_diagnostics.append({"Secret": name, "Candidate": candidate, "Source": "kaggle_secrets", "Available": True, "Error": ""})
                    return value
                secret_load_diagnostics.append({"Secret": name, "Candidate": candidate, "Source": "kaggle_secrets", "Available": False, "Error": "empty value"})
            except Exception as exc:
                secret_load_diagnostics.append({"Secret": name, "Candidate": candidate, "Source": "kaggle_secrets", "Available": False, "Error": short_error(exc)})
    except Exception as exc:
        secret_load_diagnostics.append({"Secret": name, "Candidate": ",".join(names), "Source": "kaggle_secrets", "Available": False, "Error": short_error(exc)})
    try:
        from google.colab import userdata

        for candidate in names:
            try:
                value = userdata.get(candidate)
                if value:
                    value = str(value).strip().strip('"').strip("'")
                    os.environ[name] = value
                    secret_load_diagnostics.append({"Secret": name, "Candidate": candidate, "Source": "colab_userdata", "Available": True, "Error": ""})
                    return value
                secret_load_diagnostics.append({"Secret": name, "Candidate": candidate, "Source": "colab_userdata", "Available": False, "Error": "empty value"})
            except Exception as exc:
                secret_load_diagnostics.append({"Secret": name, "Candidate": candidate, "Source": "colab_userdata", "Available": False, "Error": short_error(exc)})
    except Exception as exc:
        secret_load_diagnostics.append({"Secret": name, "Candidate": ",".join(names), "Source": "colab_userdata", "Available": False, "Error": short_error(exc)})
    return None


hf_token = load_secret("HF_TOKEN")
tabpfn_token = load_secret("TABPFN_TOKEN", aliases=["PRIORLABS_API_KEY", "TABPFN_API_KEY"])
if tabpfn_token:
    os.environ["TABPFN_TOKEN"] = tabpfn_token
    os.environ.setdefault("PRIORLABS_API_KEY", tabpfn_token)
else:
    os.environ["TABPFN_NO_BROWSER"] = "1"
_ = os.environ.setdefault("TABPFN_DISABLE_TELEMETRY", "1")

tabpfn_paths_enabled = bool(
    RUN_DIRECT_TABPFN
    or (MODEL_FAMILY_FEATURE_SENSITIVITY_ENABLED and RUN_FEATURE_SENSITIVITY_TABPFN)
    or (OBJECTIVE_SPECIFIC_RETRAINING_ENABLED and RUN_OBJECTIVE_SPECIFIC_TABPFN)
)
secret_load_summary = pd.DataFrame(secret_load_diagnostics)
if len(secret_load_summary) > 0:
    display(secret_load_summary)
    save_artifact_table(secret_load_summary, "secret_load_summary.csv", description="Secret availability diagnostics without exposing secret values.")


def make_tabpfn_classifier(n_estimators=N_TFM_ESTIMATORS):
    from tabpfn import TabPFNClassifier
    from tabpfn.constants import ModelVersion

    model_version = getattr(ModelVersion, TABPFN_MODEL_VERSION)
    return TabPFNClassifier.create_default_for_version(
        model_version,
        n_estimators=n_estimators,
        device=TABPFN_DEVICE,
        random_state=SEED,
    )


def validate_tabpfn_authentication():
    if not tabpfn_paths_enabled or not TABPFN_AUTH_PREFLIGHT_ENABLED:
        return pd.DataFrame([{"Check": "TabPFN authentication preflight", "Status": "skipped", "Evidence": f"tabpfn_paths_enabled={tabpfn_paths_enabled}; TABPFN_AUTH_PREFLIGHT_ENABLED={TABPFN_AUTH_PREFLIGHT_ENABLED}"}])
    if not tabpfn_token:
        return pd.DataFrame([{"Check": "TabPFN authentication preflight", "Status": "fail", "Evidence": "TABPFN_TOKEN is not visible to the Python process"}])
    try:
        probe_X = np.asarray([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 1.0]], dtype=np.float32)
        probe_y = np.asarray([0, 1, 0, 1], dtype=np.int32)
        probe = make_tabpfn_classifier(n_estimators=1)
        probe.fit(probe_X, probe_y)
        _ = probe.predict_proba(probe_X[:2])
        del probe
        cleanup_runtime_memory("after_tabpfn_auth_preflight")
        return pd.DataFrame([{"Check": "TabPFN authentication preflight", "Status": "pass", "Evidence": f"TABPFN_TOKEN loaded; token_length={len(tabpfn_token)}; {TABPFN_MODEL_VERSION} tiny model-load/predict succeeded"}])
    except Exception as exc:
        cleanup_runtime_memory("after_tabpfn_auth_preflight_error")
        return pd.DataFrame([{"Check": "TabPFN authentication preflight", "Status": "fail", "Evidence": short_error(exc)}])


tabpfn_authentication_summary = validate_tabpfn_authentication()
display(tabpfn_authentication_summary)
save_artifact_table(tabpfn_authentication_summary, "tabpfn_authentication_summary.csv", description="TabPFN token and model-load preflight status without exposing token values.")
if tabpfn_paths_enabled and (len(tabpfn_authentication_summary) == 0 or tabpfn_authentication_summary.iloc[0]["Status"] == "fail"):
    raise RuntimeError(
        "TabPFN paths are enabled, but the TabPFN token/model-load preflight failed. "
        "A non-empty TABPFN_TOKEN only proves that a secret string exists; it does not prove that the token is authorized for the current TabPFN model weights. "
        "Open https://ux.priorlabs.ai, accept the license on the Licenses tab, copy a fresh API key from the account page, update the Kaggle secret TABPFN_TOKEN, restart the session, and rerun from this setup cell. "
        "If you want to continue without TabPFN, set RUN_DIRECT_TABPFN=False, RUN_FEATURE_SENSITIVITY_TABPFN=False, and RUN_OBJECTIVE_SPECIFIC_TABPFN=False."
    )

environment_summary = pd.DataFrame(
    [
        {"Component": "Python", "Version": platform.python_version()},
        {"Component": "CUDA devices", "Version": CUDA_DEVICE_COUNT},
        {"Component": "pandas", "Version": installed_version("pandas")},
        {"Component": "numpy", "Version": installed_version("numpy")},
        {"Component": "scikit-learn", "Version": sklearn.__version__},
        {"Component": "xgboost", "Version": installed_version("xgboost")},
        {"Component": "torch", "Version": torch.__version__},
        {"Component": "cupy", "Version": cp.__version__},
        {"Component": "yfinance", "Version": installed_version("yfinance")},
        {"Component": "tabpfn", "Version": installed_version("tabpfn")},
        {"Component": "tabicl", "Version": installed_version("tabicl")},
    ]
)

display(environment_summary)
save_artifact_table(environment_summary, "environment_summary.csv")
source_provenance_summary = source_provenance_frame()
display(source_provenance_summary)
save_artifact_table(source_provenance_summary, "source_provenance_summary.csv")
initial_cuda_memory = record_cuda_memory("after_imports_and_configuration")
if len(initial_cuda_memory) > 0:
    display(initial_cuda_memory.round(2))
    save_artifact_table(initial_cuda_memory, "cuda_memory_after_imports.csv")

configuration_summary = pd.DataFrame(
    [
        {"Parameter": "DATA_START_DATE", "Value": DATA_START_DATE},
        {"Parameter": "DATA_END_DATE", "Value": DATA_END_DATE},
        {"Parameter": "FRED_DOWNLOAD_START_DATE", "Value": FRED_DOWNLOAD_START_DATE},
        {"Parameter": "FRED_MIN_SELECTION_OBSERVATIONS", "Value": FRED_MIN_SELECTION_OBSERVATIONS},
        {"Parameter": "ASSET_TICKERS", "Value": safe_string(list(ASSET_TICKERS))},
        {"Parameter": "ASSET_COUNT", "Value": len(ASSET_TICKERS)},
        {"Parameter": "TARGET_TOP_K", "Value": TARGET_TOP_K},
        {"Parameter": "PORTFOLIO_TOP_K", "Value": PORTFOLIO_TOP_K},
        {"Parameter": "TRANSACTION_COST_BPS", "Value": TRANSACTION_COST_BPS},
        {"Parameter": "TRANSACTION_COST_SENSITIVITY_BPS", "Value": safe_string(TRANSACTION_COST_SENSITIVITY_BPS)},
        {"Parameter": "TURNOVER_CONSTRAINT_CAPS", "Value": safe_string(TURNOVER_CONSTRAINT_CAPS)},
        {"Parameter": "TAX_DRAG_SENSITIVITY_BPS_PER_TURNOVER", "Value": safe_string(TAX_DRAG_SENSITIVITY_BPS_PER_TURNOVER)},
        {"Parameter": "MARKET_IMPACT_NOTIONAL_USD", "Value": MARKET_IMPACT_NOTIONAL_USD},
        {"Parameter": "RISK_WEIGHTED_TOP_K_MAX_WEIGHT", "Value": RISK_WEIGHTED_TOP_K_MAX_WEIGHT},
        {"Parameter": "EX_ANTE_RISK_LOOKBACK_MONTHS", "Value": EX_ANTE_RISK_LOOKBACK_MONTHS},
        {"Parameter": "EX_ANTE_RISK_MIN_MONTHS", "Value": EX_ANTE_RISK_MIN_MONTHS},
        {"Parameter": "EXECUTION_RETURN_MODE", "Value": EXECUTION_RETURN_MODE},
        {"Parameter": "FEATURE_SET_VARIANT", "Value": FEATURE_SET_VARIANT},
        {"Parameter": "FEATURE_VARIANT_SENSITIVITY_ENABLED", "Value": FEATURE_VARIANT_SENSITIVITY_ENABLED},
        {"Parameter": "FEATURE_SENSITIVITY_VARIANTS", "Value": safe_string(FEATURE_SENSITIVITY_VARIANTS)},
        {"Parameter": "MODEL_FAMILY_FEATURE_SENSITIVITY_ENABLED", "Value": MODEL_FAMILY_FEATURE_SENSITIVITY_ENABLED},
        {"Parameter": "MODEL_FAMILY_FEATURE_SENSITIVITY_VARIANTS", "Value": safe_string(MODEL_FAMILY_FEATURE_SENSITIVITY_VARIANTS)},
        {"Parameter": "RUN_FEATURE_SENSITIVITY_LOGISTIC", "Value": RUN_FEATURE_SENSITIVITY_LOGISTIC},
        {"Parameter": "RUN_FEATURE_SENSITIVITY_TABPFN", "Value": RUN_FEATURE_SENSITIVITY_TABPFN},
        {"Parameter": "RUN_FEATURE_SENSITIVITY_XGBOOST_LIGHT", "Value": RUN_FEATURE_SENSITIVITY_XGBOOST_LIGHT},
        {"Parameter": "FEATURE_SENSITIVITY_XGBOOST_N_ESTIMATORS", "Value": FEATURE_SENSITIVITY_XGBOOST_N_ESTIMATORS},
        {"Parameter": "MULTI_HORIZON_MONTHS", "Value": safe_string(MULTI_HORIZON_MONTHS)},
        {"Parameter": "BENCHMARK_ASSET", "Value": BENCHMARK_ASSET},
        {"Parameter": "OBJECTIVE_SPECIFIC_RETRAINING_ENABLED", "Value": OBJECTIVE_SPECIFIC_RETRAINING_ENABLED},
        {"Parameter": "OBJECTIVE_SPECIFIC_TARGETS", "Value": safe_string(OBJECTIVE_SPECIFIC_TARGETS)},
        {"Parameter": "RUN_OBJECTIVE_SPECIFIC_LOGISTIC", "Value": RUN_OBJECTIVE_SPECIFIC_LOGISTIC},
        {"Parameter": "RUN_OBJECTIVE_SPECIFIC_TABPFN", "Value": RUN_OBJECTIVE_SPECIFIC_TABPFN},
        {"Parameter": "RUN_OBJECTIVE_SPECIFIC_XGBOOST_LIGHT", "Value": RUN_OBJECTIVE_SPECIFIC_XGBOOST_LIGHT},
        {"Parameter": "RUN_OBJECTIVE_SPECIFIC_XGBOOST_SEARCH", "Value": RUN_OBJECTIVE_SPECIFIC_XGBOOST_SEARCH},
        {"Parameter": "OBJECTIVE_SPECIFIC_XGBOOST_N_ESTIMATORS", "Value": OBJECTIVE_SPECIFIC_XGBOOST_N_ESTIMATORS},
        {"Parameter": "OBJECTIVE_SPECIFIC_XGBOOST_SEARCH_ITERATIONS", "Value": OBJECTIVE_SPECIFIC_XGBOOST_SEARCH_ITERATIONS},
        {"Parameter": "CONTEXT_END_DATE", "Value": CONTEXT_END_DATE},
        {"Parameter": "TUNING_END_DATE", "Value": TUNING_END_DATE},
        {"Parameter": "CALIBRATION_END_DATE", "Value": CALIBRATION_END_DATE},
        {"Parameter": "HOLDOUT_START_DATE", "Value": HOLDOUT_START_DATE},
        {"Parameter": "FAST_MODE", "Value": FAST_MODE},
        {"Parameter": "MEMORY_EFFICIENT_MODE", "Value": MEMORY_EFFICIENT_MODE},
        {"Parameter": "MEMORY_EFFICIENT_N_TFM_ESTIMATORS", "Value": MEMORY_EFFICIENT_N_TFM_ESTIMATORS},
        {"Parameter": "SAVE_FULL_PREDICTION_SCORES", "Value": SAVE_FULL_PREDICTION_SCORES},
        {"Parameter": "N_TFM_ESTIMATORS", "Value": N_TFM_ESTIMATORS},
        {"Parameter": "TABPFN_AUTH_PREFLIGHT_ENABLED", "Value": TABPFN_AUTH_PREFLIGHT_ENABLED},
        {"Parameter": "XGBOOST_SEARCH_EARLY_STOPPING_ENABLED", "Value": XGBOOST_SEARCH_EARLY_STOPPING_ENABLED},
        {"Parameter": "XGBOOST_SEARCH_EARLY_STOPPING_ROUNDS", "Value": XGBOOST_SEARCH_EARLY_STOPPING_ROUNDS},
        {"Parameter": "RUN_GPU_XGBOOST", "Value": RUN_GPU_XGBOOST},
        {"Parameter": "RUN_DIRECT_TABPFN", "Value": RUN_DIRECT_TABPFN},
        {"Parameter": "RUN_DIRECT_TABICL", "Value": RUN_DIRECT_TABICL},
        {"Parameter": "RUN_LOGISTIC_REGRESSION_CPU_BENCHMARK", "Value": RUN_LOGISTIC_REGRESSION_CPU_BENCHMARK},
        {"Parameter": "XGBOOST_TUNING_ITERATIONS", "Value": XGBOOST_TUNING_ITERATIONS},
        {"Parameter": "CLASSICAL_TUNING_CV_SPLITS", "Value": CLASSICAL_TUNING_CV_SPLITS},
        {"Parameter": "BOOTSTRAP_ITERATIONS", "Value": BOOTSTRAP_ITERATIONS},
        {"Parameter": "XGBOOST_DEVICE", "Value": XGBOOST_DEVICE},
        {"Parameter": "TABPFN_DEVICE", "Value": safe_string(TABPFN_DEVICE)},
        {"Parameter": "TABICL_DEVICE", "Value": TABICL_DEVICE},
    ]
)
display(configuration_summary)
save_artifact_table(configuration_summary, "configuration_summary.csv", description="Notebook configuration values used for this run.")

method_contract = f"""
# Tactical Asset-Allocation Workflow Contract

Universe: {safe_string(list(ASSET_TICKERS))}
Task: score each asset at monthly signal date t for whether it will be in the top {TARGET_TOP_K} assets by next-month total return inside the configured universe.
Current run scope: RUN_GPU_XGBOOST={RUN_GPU_XGBOOST}; RUN_DIRECT_TABPFN={RUN_DIRECT_TABPFN}; RUN_DIRECT_TABICL={RUN_DIRECT_TABICL}. TabICL is intentionally disabled for this run after prior full-mode RAM failures.
Target construction: next-month asset return is computed from the configured execution-return convention (`{EXECUTION_RETURN_MODE}`); close-to-close and next-open diagnostic returns are both retained for audit. The top-k label is assigned only within the same signal month.
Feature availability policy: features use data available at or before the monthly signal date; forward returns, forward excess returns, benchmark-relative returns, multi-horizon returns, and target labels are excluded from model features. Rolling dollar-volume and high-low spread proxies are included in the predictive matrix as a deliberate P22 feature-set change, not only as downstream portfolio diagnostics. The configured feature variant is `{FEATURE_SET_VARIANT}`; identity-ablated mode removes static ticker, group, and risk-bucket metadata from the main model matrix. Feature-variant sensitivity artifacts compare full, ticker-ablated, identity-ablated, metadata-only, and strict time-series policies using the configured model families. The default selected model-family feature-policy rerun evaluates `{MODEL_FAMILY_FEATURE_SENSITIVITY_VARIANTS}` with enabled families: logistic={RUN_FEATURE_SENSITIVITY_LOGISTIC}, TabPFN={RUN_FEATURE_SENSITIVITY_TABPFN}, fixed GPU XGBoost={RUN_FEATURE_SENSITIVITY_XGBOOST_LIGHT}. FRED series are included in model features only when they have enough non-missing observations in the model-selection window; unavailable optional credit-spread histories are recorded but not imputed into the training set.
Splits: context through {CONTEXT_END_DATE}; model-selection history through {TUNING_END_DATE}; calibration through {CALIBRATION_END_DATE}; holdout from {HOLDOUT_START_DATE}.
Validation: rolling-origin chronological cross-validation on monthly groups with positive-count checks.
Default model families: deterministic allocation rules, GPU XGBoost, direct TabPFN, and guarded direct TabICL. TabICL is disabled in this run while its RAM behavior is investigated, so the TabICL empirical comparison remains unresolved by design. Main and objective-specific XGBoost randomized searches keep chronological validation folds and use fold-level early stopping on each validation block to avoid wasting trees after validation AUC-PR stops improving. Feature-variant sensitivity uses fixed GPU XGBoost and direct TabPFN by default; CPU Logistic Regression is retained only as an optional disabled benchmark path.
Execution convention: scores are formed after month-end close data is available and are evaluated with `{EXECUTION_RETURN_MODE}`. The default next-open convention uses the first adjusted open of the next month through the first adjusted open of the following month. The diagnostic still does not model intraday slippage, taxes, market impact, or mandate constraints.
Portfolio diagnostic: monthly top-k equal-weight allocation from headline model scores, compared with equal-weight, SPY-only, 60/40 SPY/TLT, inverse-volatility, deterministic momentum rules, and inverse-volatility capped top-k score portfolios. Feature-sensitivity score portfolios are saved separately so diagnostic reruns do not pollute the headline allocation table. The main portfolio table uses {TRANSACTION_COST_BPS:.1f} basis points per unit of one-way turnover, with separate transaction-cost, tax-drag, turnover-cap, liquidity-proxy, benchmark-relative, drawdown, ex-ante trailing-covariance risk, and weight-constraint artifacts for stress review. Liquidity and spread proxies are built from rolling dollar volume and high-low range features derived from downloaded OHLCV data.
Alternative objective diagnostics: multi-horizon and benchmark-relative artifacts still evaluate existing one-month scores against alternative labels. This P22 version also performs objective-specific retraining for configured targets `{OBJECTIVE_SPECIFIC_TARGETS}` with enabled families: logistic={RUN_OBJECTIVE_SPECIFIC_LOGISTIC}, TabPFN={RUN_OBJECTIVE_SPECIFIC_TABPFN}, fixed GPU XGBoost={RUN_OBJECTIVE_SPECIFIC_XGBOOST_LIGHT}, target-specific searched GPU XGBoost={RUN_OBJECTIVE_SPECIFIC_XGBOOST_SEARCH}. These reruns are diagnostics and are saved separately from the main one-month allocation table. One-month objective score-to-portfolio diagnostics are separated from multi-horizon objective score-to-return diagnostics; overlapping 3M/6M forward returns are not compounded as monthly portfolio backtests.
Remaining caveats: TabICL is disabled and therefore not empirically resolved; model-family feature sensitivity and objective-specific retraining are focused reruns rather than an exhaustive hyperparameter-search study of every feature policy and model family; objective-specific multi-horizon outputs are horizon-return diagnostics rather than tradeable strategy claims; liquidity, tax, market-impact, ex-ante risk, and constrained-allocation sections are stress proxies rather than production execution or optimization engines; public-data and embedding limitations are documented but not solved here.
Interpretation: educational workflow test; not investment advice; not a claim of market predictability or deployable trading performance.
""".strip()
save_text_artifact("method_contract.md", method_contract, description="Plain-language modeling and evaluation contract.")

print(f"Run mode: {'FAST smoke test' if FAST_MODE else 'full research run'}")
print(f"Asset universe: {safe_string(list(ASSET_TICKERS))}")
print(f"Splits: context <= {CONTEXT_END_DATE}; selection <= {TUNING_END_DATE}; calibration <= {CALIBRATION_END_DATE}; holdout >= {HOLDOUT_START_DATE}")
print(f"HF_TOKEN found: {bool(hf_token)}; TABPFN_TOKEN found: {bool(tabpfn_token)}")
print(f"TabPFN device: {TABPFN_DEVICE}; TabICL device: {TABICL_DEVICE}; XGBoost device: {XGBOOST_DEVICE}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.9/252.9 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.0/73.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.2/240.2 kB 14.7 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.


,Secret,Candidate,Source,Available,Error
0,HF_TOKEN,HF_TOKEN,kaggle_secrets,True,
1,TABPFN_TOKEN,TABPFN_TOKEN,kaggle_secrets,True,


Saved tabpfn_tabicl_tactical_asset_allocation_20260512_v2_outputs/secret_load_summary.csv


tabpfn-v2.6-classifier-v2.6_default.ckpt:   0%|          | 0.00/43.0M [00:00<?, ?B/s]

...

Saved tabpfn_tabicl_tactical_asset_allocation_20260512_v2_outputs/source_provenance_summary.csv


,stage,device_index,device_name,allocated_mb,reserved_mb,max_allocated_mb,free_mb,total_mb
0,after_imports_and_configuration,0,Tesla T4,9.12,22.0,81.92,14757.81,14912.69
1,after_imports_and_configuration,1,Tesla T4,0.00,0.0,40.96,14807.81,14912.69


Saved tabpfn_tabicl_tactical_asset_allocation_20260512_v2_outputs/cuda_memory_after_imports.csv


,Parameter,Value
0,DATA_START_DATE,2006-01-01
1,DATA_END_DATE,2026-05-12
2,FRED_DOWNLOAD_START_DATE,1990-01-01
3,FRED_MIN_SELECTION_OBSERVATIONS,36
4,ASSET_TICKERS,"SPY, QQQ, IWM, TLT, IEF, GLD, EFA, EEM, XLE, XLF"
5,ASSET_COUNT,10
6,TARGET_TOP_K,3
7,PORTFOLIO_TOP_K,3
8,TRANSACTION_COST_BPS,5.0
9,TRANSACTION_COST_SENSITIVITY_BPS,"0.0, 5.0, 10.0, 25.0, 50.0"


Saved tabpfn_tabicl_tactical_asset_allocation_20260512_v2_outputs/configuration_summary.csv
Saved tabpfn_tabicl_tactical_asset_allocation_20260512_v2_outputs/method_contract.md
Run mode: FAST smoke test
Asset universe: SPY, QQQ, IWM, TLT, IEF, GLD, EFA, EEM, XLE, XLF
Splits: context <= 2011-12-31; selection <= 2017-12-31; calibration <= 2019-12-31; holdout >= 2020-01-01
HF_TOKEN found: True; TABPFN_TOKEN found: True
TabPFN device: ['cuda:0', 'cuda:1']; TabICL device: cuda:0; XGBoost device: cuda
CPU times: user 9.71 s, sys: 2.7 s, total: 12.4 s
Wall time: 34.4 s


## 1. Load Public Market, Volatility, and Macro Data

The workflow uses public adjusted ETF prices, Cboe VIX history, and FRED macro series. Some FRED-hosted credit-spread series can be restricted to recent public history; the notebook records each FRED series availability and includes a series in model features only when enough model-selection-window history is present. Public data improves reproducibility, but it is not a substitute for a point-in-time institutional data system. The leakage checklist later records the assumptions that follow from this data choice.

In [2]:
def normalize_date_index(frame, date_column=None):
    out = frame.copy()
    if date_column is not None:
        out[date_column] = pd.to_datetime(out[date_column])
        out = out.set_index(date_column)
    out.index = pd.to_datetime(out.index).tz_localize(None)
    return out.sort_index()


def download_yfinance_ohlcv(ticker):
    frame = yf.download(ticker, start=DATA_START_DATE, end=DATA_END_DATE, auto_adjust=False, progress=False, threads=False)
    if frame.empty:
        raise ValueError(f"yfinance returned no rows for {ticker}.")
    if isinstance(frame.columns, pd.MultiIndex):
        frame.columns = [column[0] for column in frame.columns]
    frame = normalize_date_index(frame)
    required = {"Open", "High", "Low", "Close", "Volume"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{ticker} data missing required columns: {sorted(missing)}")
    if "Adj Close" not in frame.columns:
        frame["Adj Close"] = frame["Close"]
    return frame[["Open", "High", "Low", "Close", "Adj Close", "Volume"]].copy()


price_frames = {}
price_source_rows = []
for ticker in tqdm(list(ASSET_TICKERS), desc="Downloading ETF OHLCV", unit="ticker"):
    frame = download_yfinance_ohlcv(ticker)
    price_frames[ticker] = frame
    price_source_rows.append(
        {
            "source": "yfinance",
            "symbol": ticker,
            "rows": len(frame),
            "start": frame.index.min().date().isoformat(),
            "end": frame.index.max().date().isoformat(),
            "description": ASSET_TICKERS[ticker],
        }
    )

adj_close = pd.concat({ticker: frame["Adj Close"] for ticker, frame in price_frames.items()}, axis=1).sort_index()
volume = pd.concat({ticker: frame["Volume"] for ticker, frame in price_frames.items()}, axis=1).sort_index()
open_px = pd.concat({ticker: frame["Open"] for ticker, frame in price_frames.items()}, axis=1).sort_index()
high_px = pd.concat({ticker: frame["High"] for ticker, frame in price_frames.items()}, axis=1).sort_index()
low_px = pd.concat({ticker: frame["Low"] for ticker, frame in price_frames.items()}, axis=1).sort_index()
close_px = pd.concat({ticker: frame["Close"] for ticker, frame in price_frames.items()}, axis=1).sort_index()


def load_vix_history():
    vix = pd.read_csv(VIX_CBOE_CSV_URL)
    date_column = "DATE" if "DATE" in vix.columns else vix.columns[0]
    vix = normalize_date_index(vix, date_column=date_column)
    rename_map = {column: f"vix_{column.lower()}" for column in vix.columns}
    vix = vix.rename(columns=rename_map)
    close_candidates = [column for column in vix.columns if "close" in column.lower()]
    if close_candidates and "vix_close" not in vix.columns:
        vix = vix.rename(columns={close_candidates[0]: "vix_close"})
    return vix


def load_fred_series(series_id, output_name):
    query = urlencode(
        {
            "id": series_id,
            "cosd": FRED_DOWNLOAD_START_DATE,
            "coed": DATA_END_DATE,
            "observation_start": FRED_DOWNLOAD_START_DATE,
        }
    )
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?{query}"
    frame = pd.read_csv(url)
    date_column = "observation_date" if "observation_date" in frame.columns else frame.columns[0]
    frame = normalize_date_index(frame, date_column=date_column)
    value_column = [column for column in frame.columns if column != date_column][0]
    values = pd.to_numeric(frame[value_column].replace(".", np.nan), errors="coerce")
    return values.rename(output_name).to_frame()

vix_history = load_vix_history()
price_source_rows.append(
    {
        "source": "Cboe VIX CSV",
        "symbol": "VIX",
        "rows": len(vix_history),
        "start": vix_history.index.min().date().isoformat(),
        "end": vix_history.index.max().date().isoformat(),
        "description": "Cboe VIX daily history",
    }
)

fred_frames = []
fred_availability_rows = []
selection_cutoff = pd.Timestamp(TUNING_END_DATE)
preholdout_cutoff = pd.Timestamp(CALIBRATION_END_DATE)
for series_id, output_name in tqdm(FRED_SERIES.items(), desc="Downloading FRED series", unit="series"):
    try:
        frame = load_fred_series(series_id, output_name)
        start_value = frame.index.min().date().isoformat() if len(frame) else ""
        end_value = frame.index.max().date().isoformat() if len(frame) else ""
        price_source_rows.append(
            {
                "source": "FRED CSV",
                "symbol": series_id,
                "rows": len(frame),
                "start": start_value,
                "end": end_value,
                "description": output_name,
            }
        )
        selection_non_null = int(frame.loc[frame.index <= selection_cutoff, output_name].notna().sum()) if len(frame) else 0
        preholdout_non_null = int(frame.loc[frame.index <= preholdout_cutoff, output_name].notna().sum()) if len(frame) else 0
        include_in_features = selection_non_null >= FRED_MIN_SELECTION_OBSERVATIONS
        status = "included" if include_in_features else "excluded_insufficient_selection_history"
        fred_availability_rows.append(
            {
                "series_id": series_id,
                "feature_name": output_name,
                "rows": len(frame),
                "start": start_value,
                "end": end_value,
                "selection_non_null": selection_non_null,
                "preholdout_non_null": preholdout_non_null,
                "minimum_selection_observations": FRED_MIN_SELECTION_OBSERVATIONS,
                "include_in_features": include_in_features,
                "status": status,
            }
        )
        if include_in_features:
            fred_frames.append(frame)
        else:
            print(f"FRED series {series_id} excluded from model features: only {selection_non_null} non-null selection-window observations.")
    except Exception as exc:
        model_errors.append({"Model": f"FRED[{series_id}]", "Error": short_error(exc)})
        fred_availability_rows.append(
            {
                "series_id": series_id,
                "feature_name": output_name,
                "rows": 0,
                "start": "",
                "end": "",
                "selection_non_null": 0,
                "preholdout_non_null": 0,
                "minimum_selection_observations": FRED_MIN_SELECTION_OBSERVATIONS,
                "include_in_features": False,
                "status": f"load_failed: {short_error(exc)}",
            }
        )
        print(f"Could not load FRED series {series_id}: {short_error(exc)}")

fred_series_availability_summary = pd.DataFrame(fred_availability_rows)
if len(fred_series_availability_summary) > 0:
    display(fred_series_availability_summary)
    save_artifact_table(fred_series_availability_summary, "fred_series_availability_summary.csv", description="FRED source coverage and feature-inclusion decisions.")

macro_daily = pd.concat(fred_frames, axis=1).sort_index() if fred_frames else pd.DataFrame(index=adj_close.index)
macro_daily = macro_daily.reindex(adj_close.index).ffill()
vix_daily = vix_history.reindex(adj_close.index).ffill()

source_summary = pd.DataFrame(price_source_rows)
display(source_summary)
save_artifact_table(source_summary, "data_source_summary.csv", description="External data sources used by the notebook.")

price_coverage = pd.DataFrame(
    {
        "ticker": adj_close.columns,
        "first_valid_date": [adj_close[column].first_valid_index().date().isoformat() for column in adj_close.columns],
        "last_valid_date": [adj_close[column].last_valid_index().date().isoformat() for column in adj_close.columns],
        "missing_rate": [float(adj_close[column].isna().mean()) for column in adj_close.columns],
    }
)
display(price_coverage)
save_artifact_table(price_coverage, "price_coverage_summary.csv")


FRED series BAMLH0A0HYM2 excluded from model features: only 0 non-null selection-window observations.
FRED series BAMLC0A0CM excluded from model features: only 0 non-null selection-window observations.


,series_id,feature_name,rows,start,end,selection_non_null,preholdout_non_null,minimum_selection_observations,include_in_features,status
0,DGS10,treasury_10y_yield,9484,1990-01-02,2026-05-08,7006,7505,36,True,included
1,DGS2,treasury_2y_yield,9484,1990-01-02,2026-05-08,7006,7505,36,True,included
2,T10Y2Y,yield_curve_10y_2y,9485,1990-01-02,2026-05-11,7006,7505,36,True,included
3,BAMLH0A0HYM2,high_yield_oas,792,2023-05-12,2026-05-08,0,0,36,False,excluded_insufficient_selection_history
4,BAMLC0A0CM,investment_grade_oas,792,2023-05-12,2026-05-08,0,0,36,False,excluded_insufficient_selection_history
5,DFF,fed_funds_rate,13277,1990-01-01,2026-05-08,10227,10957,36,True,included
6,DTB3,t_bill_3m,9484,1990-01-02,2026-05-08,7006,7505,36,True,included


Saved tabpfn_tabicl_tactical_asset_allocation_20260512_v2_outputs/fred_series_availability_summary.csv


,source,symbol,rows,start,end,description
0,yfinance,SPY,5120,2006-01-03,2026-05-11,US large-cap equity
1,yfinance,QQQ,5120,2006-01-03,2026-05-11,US growth equity
2,yfinance,IWM,5120,2006-01-03,2026-05-11,US small-cap equity
3,yfinance,TLT,5120,2006-01-03,2026-05-11,Long-duration Treasury
4,yfinance,IEF,5120,2006-01-03,2026-05-11,Intermediate Treasury
5,yfinance,GLD,5120,2006-01-03,2026-05-11,Gold
6,yfinance,EFA,5120,2006-01-03,2026-05-11,Developed ex-US equity
7,yfinance,EEM,5120,2006-01-03,2026-05-11,Emerging-market equity
8,yfinance,XLE,5120,2006-01-03,2026-05-11,US energy sector
9,yfinance,XLF,5120,2006-01-03,2026-05-11,US financials sector


Saved tabpfn_tabicl_tactical_asset_allocation_20260512_v2_outputs/data_source_summary.csv


,ticker,first_valid_date,last_valid_date,missing_rate
0,SPY,2006-01-03,2026-05-11,0.0
1,QQQ,2006-01-03,2026-05-11,0.0
2,IWM,2006-01-03,2026-05-11,0.0
3,TLT,2006-01-03,2026-05-11,0.0
4,IEF,2006-01-03,2026-05-11,0.0
5,GLD,2006-01-03,2026-05-11,0.0
6,EFA,2006-01-03,2026-05-11,0.0
7,EEM,2006-01-03,2026-05-11,0.0
8,XLE,2006-01-03,2026-05-11,0.0
9,XLF,2006-01-03,2026-05-11,0.0


Saved tabpfn_tabicl_tactical_asset_allocation_20260512_v2_outputs/price_coverage_summary.csv


PosixPath('tabpfn_tabicl_tactical_asset_allocation_20260512_v2_outputs/price_coverage_summary.csv')

## 2. Build a Point-in-Time Monthly Allocation Table

The modeling table has one row per asset and monthly signal date. Features are computed from information available at or before the signal date. The signal date is the month-end close used for the public-data diagnostic; scores should be interpreted as being formed after those close observations are available and then evaluated on the following close-to-close monthly return. The default target is whether the asset is in the top `TARGET_TOP_K` assets for the next month.

In [3]:
def annualized_realized_vol(log_returns, window):
    return log_returns.rolling(window).std() * math.sqrt(252)


def downside_realized_vol(log_returns, window):
    downside = log_returns.where(log_returns < 0.0, 0.0)
    return downside.rolling(window).std() * math.sqrt(252)


def rolling_drawdown(price, window):
    return price / price.rolling(window).max() - 1.0


def rolling_zscore(series, window):
    rolling_mean = series.rolling(window).mean()
    rolling_std = series.rolling(window).std()
    return (series - rolling_mean) / rolling_std.replace(0.0, np.nan)


feature_series = {}
adj_close_aligned = adj_close.ffill()
volume_aligned = volume.reindex(adj_close_aligned.index).ffill()
high_aligned = high_px.reindex(adj_close_aligned.index).ffill()
low_aligned = low_px.reindex(adj_close_aligned.index).ffill()
close_aligned = close_px.reindex(adj_close_aligned.index).ffill()
asset_log_returns = {}
asset_simple_returns = {}

for ticker in tqdm(adj_close_aligned.columns, desc="Building daily asset features", unit="ticker"):
    price = adj_close_aligned[ticker]
    log_return = np.log(price).diff()
    simple_return = price.pct_change()
    asset_log_returns[ticker] = log_return
    asset_simple_returns[ticker] = simple_return

    for window in RETURN_WINDOWS_DAYS:
        feature_series[f"{ticker.lower()}_return_{window}d"] = price.pct_change(window)
    for window in VOL_WINDOWS_DAYS:
        feature_series[f"{ticker.lower()}_rv_{window}d"] = annualized_realized_vol(log_return, window)
        feature_series[f"{ticker.lower()}_downside_rv_{window}d"] = downside_realized_vol(log_return, window)
    for window in DRAWDOWN_WINDOWS_DAYS:
        feature_series[f"{ticker.lower()}_drawdown_{window}d"] = rolling_drawdown(price, window)
    lower = ticker.lower()
    for window in TREND_WINDOWS_DAYS:
        feature_series[f"{lower}_ma_distance_{window}d"] = price / price.rolling(window).mean() - 1.0
    dollar_volume = close_aligned[ticker] * volume_aligned[ticker]
    high_low_range = (high_aligned[ticker] - low_aligned[ticker]) / close_aligned[ticker].replace(0.0, np.nan)
    feature_series[f"{lower}_volume_zscore_63d"] = rolling_zscore(np.log1p(volume_aligned[ticker]), 63)
    feature_series[f"{lower}_avg_dollar_volume_63d"] = dollar_volume.rolling(63, min_periods=21).mean()
    feature_series[f"{lower}_hl_spread_proxy_21d"] = high_low_range.rolling(21, min_periods=10).mean()

# Cross-asset relationships and regime features.
spy_return = asset_simple_returns.get("SPY")
if spy_return is not None:
    for ticker in adj_close_aligned.columns:
        if ticker == "SPY":
            continue
        joined = pd.concat([asset_simple_returns[ticker], spy_return], axis=1).dropna()
        rolling_cov = joined.iloc[:, 0].rolling(126).cov(joined.iloc[:, 1])
        rolling_var = joined.iloc[:, 1].rolling(126).var()
        feature_series[f"{ticker.lower()}_beta_to_spy_126d"] = (rolling_cov / rolling_var.replace(0.0, np.nan)).reindex(adj_close_aligned.index)

if "vix_close" in vix_daily.columns:
    feature_series["vix_close"] = vix_daily["vix_close"]
    feature_series["vix_change_21d"] = vix_daily["vix_close"].diff(21)
    feature_series["vix_zscore_252d"] = rolling_zscore(vix_daily["vix_close"], 252)

for column in macro_daily.columns:
    series = macro_daily[column]
    feature_series[column] = series
    feature_series[f"{column}_change_21d"] = series.diff(21)
    feature_series[f"{column}_zscore_252d"] = rolling_zscore(series, 252)

market_feature_daily = pd.DataFrame(feature_series).sort_index()
monthly_market_features = market_feature_daily.resample("M").last()
monthly_prices = adj_close_aligned.resample("M").last()
adjustment_ratio = (adj_close_aligned / close_px.replace(0.0, np.nan)).replace([np.inf, -np.inf], np.nan)
adj_open_px = (open_px * adjustment_ratio).reindex(adj_close_aligned.index).ffill()
monthly_open_prices = adj_open_px.resample("M").first()
monthly_open_prices_for_returns = monthly_open_prices.copy()
# The yfinance end date is exclusive. If it falls inside the latest resampled month,
# drop that partial month as a signal month. Keep its first adjusted open for next-open
# exit-return measurement because that open is already known after the month begins.
requested_end_period = pd.Timestamp(DATA_END_DATE).to_period("M")
if len(monthly_prices) > 0 and monthly_prices.index.max().to_period("M") == requested_end_period:
    monthly_prices = monthly_prices.iloc[:-1].copy()
    monthly_market_features = monthly_market_features.loc[monthly_market_features.index.isin(monthly_prices.index)].copy()
monthly_close_to_close_forward_returns = monthly_prices.shift(-1) / monthly_prices - 1.0
monthly_next_open_forward_returns = monthly_open_prices_for_returns.shift(-2) / monthly_open_prices_for_returns.shift(-1) - 1.0
if EXECUTION_RETURN_MODE == "close_to_close":
    monthly_forward_returns = monthly_close_to_close_forward_returns
elif EXECUTION_RETURN_MODE == "next_open_to_next_open":
    monthly_forward_returns = monthly_next_open_forward_returns
else:
    raise ValueError(f"Unsupported EXECUTION_RETURN_MODE: {EXECUTION_RETURN_MODE}")
monthly_asset_returns = monthly_prices.pct_change()
multi_horizon_forward_returns = {
    horizon: (1.0 + monthly_forward_returns).rolling(horizon).apply(np.prod, raw=True).shift(-(horizon - 1)) - 1.0
    for horizon in MULTI_HORIZON_MONTHS
}

asset_metadata_rows = []
for ticker in ASSET_TICKERS:
    row = {
        "asset": ticker,
        "asset_risk_bucket": ASSET_RISK_BUCKET.get(ticker, np.nan),
    }
    for asset_ticker in ASSET_TICKERS:
        row[f"asset_is_{asset_ticker.lower()}"] = int(ticker == asset_ticker)
    for group in sorted(set(ASSET_GROUPS.values())):
        row[f"asset_group_{group}"] = int(ASSET_GROUPS.get(ticker) == group)
    asset_metadata_rows.append(row)
asset_metadata = pd.DataFrame(asset_metadata_rows).set_index("asset")

panel_rows = []
for signal_date in tqdm(monthly_market_features.index, desc="Building monthly panel", unit="month"):
    market_row = monthly_market_features.loc[signal_date]
    if signal_date not in monthly_forward_returns.index:
        continue
    forward_row = monthly_forward_returns.loc[signal_date]
    close_to_close_row = monthly_close_to_close_forward_returns.loc[signal_date] if signal_date in monthly_close_to_close_forward_returns.index else pd.Series(dtype=float)
    next_open_row = monthly_next_open_forward_returns.loc[signal_date] if signal_date in monthly_next_open_forward_returns.index else pd.Series(dtype=float)
    same_month_returns = monthly_asset_returns.loc[signal_date] if signal_date in monthly_asset_returns.index else pd.Series(dtype=float)
    benchmark_forward_return = float(forward_row.get(BENCHMARK_ASSET, np.nan)) if BENCHMARK_ASSET in forward_row.index else np.nan
    horizon_rows = {horizon: returns.loc[signal_date] if signal_date in returns.index else pd.Series(dtype=float) for horizon, returns in multi_horizon_forward_returns.items()}
    available_forward = forward_row.dropna()
    if len(available_forward) < max(3, TARGET_TOP_K):
        continue
    top_assets = set(available_forward.sort_values(ascending=False).head(TARGET_TOP_K).index)
    universe_forward_mean = float(available_forward.mean())
    universe_forward_rank = available_forward.rank(ascending=False, method="first")
    for ticker, forward_return in available_forward.items():
        row = {
            "date": pd.Timestamp(signal_date),
            "asset": ticker,
            "target_top_k_next_1m": int(ticker in top_assets),
            "forward_1m_return": float(forward_return),
            "forward_1m_close_to_close_return": float(close_to_close_row.get(ticker, np.nan)),
            "forward_1m_next_open_return": float(next_open_row.get(ticker, np.nan)),
            "forward_1m_excess_return": float(forward_return - universe_forward_mean),
            "forward_1m_rank": float(universe_forward_rank[ticker]),
            "forward_1m_benchmark_return": benchmark_forward_return,
            "forward_1m_vs_benchmark_return": float(forward_return - benchmark_forward_return) if np.isfinite(benchmark_forward_return) else np.nan,
            "target_outperform_benchmark_next_1m": int(forward_return > benchmark_forward_return) if np.isfinite(benchmark_forward_return) else np.nan,
            "universe_forward_1m_return": universe_forward_mean,
            "execution_return_mode": EXECUTION_RETURN_MODE,
            "asset_month_return": float(same_month_returns.get(ticker, np.nan)),
        }
        row.update(asset_metadata.loc[ticker].to_dict())
        lower = ticker.lower()
        row["asset_return_21d"] = market_row.get(f"{lower}_return_21d", np.nan)
        row["asset_return_63d"] = market_row.get(f"{lower}_return_63d", np.nan)
        row["asset_return_126d"] = market_row.get(f"{lower}_return_126d", np.nan)
        row["asset_return_252d"] = market_row.get(f"{lower}_return_252d", np.nan)
        row["asset_rv_21d"] = market_row.get(f"{lower}_rv_21d", np.nan)
        row["asset_rv_63d"] = market_row.get(f"{lower}_rv_63d", np.nan)
        row["asset_rv_126d"] = market_row.get(f"{lower}_rv_126d", np.nan)
        row["asset_downside_rv_63d"] = market_row.get(f"{lower}_downside_rv_63d", np.nan)
        row["asset_drawdown_126d"] = market_row.get(f"{lower}_drawdown_126d", np.nan)
        row["asset_drawdown_252d"] = market_row.get(f"{lower}_drawdown_252d", np.nan)
        row["asset_ma_distance_126d"] = market_row.get(f"{lower}_ma_distance_126d", np.nan)
        row["asset_volume_zscore_63d"] = market_row.get(f"{lower}_volume_zscore_63d", np.nan)
        row["asset_avg_dollar_volume_63d"] = market_row.get(f"{lower}_avg_dollar_volume_63d", np.nan)
        row["asset_hl_spread_proxy_21d"] = market_row.get(f"{lower}_hl_spread_proxy_21d", np.nan)
        row["asset_beta_to_spy_126d"] = market_row.get(f"{lower}_beta_to_spy_126d", 1.0 if ticker == "SPY" else np.nan)
        for horizon, horizon_row in horizon_rows.items():
            horizon_available = horizon_row.dropna()
            horizon_return = horizon_row.get(ticker, np.nan)
            if len(horizon_available) >= max(3, TARGET_TOP_K) and np.isfinite(horizon_return):
                horizon_rank = horizon_available.rank(ascending=False, method="first")
                row[f"forward_{horizon}m_return"] = float(horizon_return)
                row[f"forward_{horizon}m_excess_return"] = float(horizon_return - horizon_available.mean())
                row[f"forward_{horizon}m_rank"] = float(horizon_rank[ticker])
                row[f"target_top_k_next_{horizon}m"] = int(ticker in set(horizon_available.sort_values(ascending=False).head(TARGET_TOP_K).index))
            else:
                row[f"forward_{horizon}m_return"] = np.nan
                row[f"forward_{horizon}m_excess_return"] = np.nan
                row[f"forward_{horizon}m_rank"] = np.nan
                row[f"target_top_k_next_{horizon}m"] = np.nan
        for feature_name, feature_value in market_row.items():
            if feature_name.startswith(f"{lower}_"):
                continue
            row[f"market_{feature_name}"] = feature_value
        panel_rows.append(row)

model_frame = pd.DataFrame(panel_rows).sort_values(["date", "asset"]).reset_index(drop=True)
model_frame = model_frame.replace([np.inf, -np.inf], np.nan)
model_frame = model_frame.dropna(subset=["target_top_k_next_1m", "forward_1m_return"]).reset_index(drop=True)
model_frame["target_top_k_next_1m"] = model_frame["target_top_k_next_1m"].astype(int)
model_frame["month"] = model_frame["date"].dt.to_period("M").astype(str)
model_frame["month_index"] = pd.factorize(model_frame["month"])[0]

if FAST_MODE:
    # Keep the full chronology but reduce the number of rows by using the configured smaller universe.
    model_frame = model_frame.loc[model_frame["asset"].isin(ASSET_TICKERS)].reset_index(drop=True)

split_preview = model_frame.groupby("month").agg(rows=("asset", "size"), positive_rows=("target_top_k_next_1m", "sum"), mean_forward_return=("forward_1m_return", "mean")).reset_index()
model_frame_head = model_frame.head(30)

display(model_frame_head)
display(split_preview.tail(12))
save_artifact_table(model_frame_head, "model_frame_head.csv")
save_artifact_table(split_preview, "monthly_target_preview.csv")

target_definition_summary = pd.DataFrame(
    [
        {"metric": "rows", "value": len(model_frame)},
        {"metric": "months", "value": model_frame["month"].nunique()},
        {"metric": "assets", "value": model_frame["asset"].nunique()},
        {"metric": "positive_rows", "value": int(model_frame["target_top_k_next_1m"].sum())},
        {"metric": "positive_rate", "value": float(model_frame["target_top_k_next_1m"].mean())},
        {"metric": "target_top_k", "value": TARGET_TOP_K},
        {"metric": "benchmark_asset", "value": BENCHMARK_ASSET},
        {"metric": "multi_horizon_months", "value": safe_string(MULTI_HORIZON_MONTHS)},
    ]
)
display(target_definition_summary)
save_artifact_table(target_definition_summary, "target_definition_summary.csv")

execution_return_summary = pd.DataFrame(
    [
        {
            "return_mode": "close_to_close",
            "non_null_rows": int(monthly_close_to_close_forward_returns.stack(dropna=True).shape[0]),
            "first_signal_month": str(monthly_close_to_close_forward_returns.dropna(how="all").index.min().to_period("M")) if len(monthly_close_to_close_forward_returns.dropna(how="all")) else "",
            "last_signal_month": str(monthly_close_to_close_forward_returns.dropna(how="all").index.max().to_period("M")) if len(monthly_close_to_close_forward_returns.dropna(how="all")) else "",
            "is_selected_target_mode": EXECUTION_RETURN_MODE == "close_to_close",
        },
        {
            "return_mode": "next_open_to_next_open",
            "non_null_rows": int(monthly_next_open_forward_returns.stack(dropna=True).shape[0]),
            "first_signal_month": str(monthly_next_open_forward_returns.dropna(how="all").index.min().to_period("M")) if len(monthly_next_open_forward_returns.dropna(how="all")) else "",
            "last_signal_month": str(monthly_next_open_forward_returns.dropna(how="all").index.max().to_period("M")) if len(monthly_next_open_forward_returns.dropna(how="all")) else "",
            "is_selected_target_mode": EXECUTION_RETURN_MODE == "next_open_to_next_open",
        },
    ]
)
display(execution_return_summary)
save_artifact_table(execution_return_summary, "execution_return_summary.csv", description="Return conventions available for target and portfolio diagnostics.")

target_audit_columns = [
    "date", "month", "asset", "target_top_k_next_1m", "forward_1m_return",
    "forward_1m_close_to_close_return", "forward_1m_next_open_return",
    "forward_1m_benchmark_return", "forward_1m_vs_benchmark_return", "target_outperform_benchmark_next_1m",
    "forward_1m_excess_return", "forward_1m_rank", "execution_return_mode",
]
for horizon in MULTI_HORIZON_MONTHS:
    target_audit_columns.extend([f"target_top_k_next_{horizon}m", f"forward_{horizon}m_return", f"forward_{horizon}m_excess_return", f"forward_{horizon}m_rank"])
target_audit_columns = [column for column in target_audit_columns if column in model_frame.columns]
model_target_audit = model_frame[target_audit_columns].copy()
save_artifact_table(model_target_audit, "model_target_audit.csv", description="Per-row target audit with selected execution convention, alternative execution returns, benchmark-relative return, and multi-horizon labels.")

liquidity_feature_summary = (
    model_frame.groupby("asset", observed=True)
    .agg(
        rows=("asset", "size"),
        adv_non_null=("asset_avg_dollar_volume_63d", lambda value: int(value.notna().sum())),
        median_adv_usd=("asset_avg_dollar_volume_63d", "median"),
        min_adv_usd=("asset_avg_dollar_volume_63d", "min"),
        median_hl_spread_proxy_bps=("asset_hl_spread_proxy_21d", lambda value: float(value.median() * 10000.0)),
        max_hl_spread_proxy_bps=("asset_hl_spread_proxy_21d", lambda value: float(value.max() * 10000.0)),
    )
    .reset_index()
)
display(liquidity_feature_summary.round(4))
save_artifact_table(liquidity_feature_summary, "liquidity_feature_summary.csv", description="Availability and scale checks for rolling dollar-volume and high-low spread proxy features.")

multi_horizon_target_summary_rows = []
for horizon in MULTI_HORIZON_MONTHS:
    target_column = f"target_top_k_next_{horizon}m"
    return_column = f"forward_{horizon}m_return"
    if target_column in model_frame.columns:
        valid = model_frame[target_column].notna() & model_frame[return_column].notna()
        multi_horizon_target_summary_rows.append({"horizon_months": horizon, "rows": int(valid.sum()), "positive_rate": float(model_frame.loc[valid, target_column].mean()) if valid.any() else np.nan, "mean_forward_return": float(model_frame.loc[valid, return_column].mean()) if valid.any() else np.nan})
multi_horizon_target_summary = pd.DataFrame(multi_horizon_target_summary_rows)
if len(multi_horizon_target_summary) > 0:
    display(multi_horizon_target_summary.round(4))
    save_artifact_table(multi_horizon_target_summary, "multi_horizon_target_summary.csv")

benchmark_target_summary = pd.DataFrame(
    [
        {
            "benchmark_asset": BENCHMARK_ASSET,
            "rows": int(model_frame["target_outperform_benchmark_next_1m"].notna().sum()),
            "outperform_rate": float(model_frame["target_outperform_benchmark_next_1m"].mean()),
            "mean_vs_benchmark_return": float(model_frame["forward_1m_vs_benchmark_return"].mean()),
        }
    ]
)
display(benchmark_target_summary.round(4))
save_artifact_table(benchmark_target_summary, "benchmark_relative_target_summary.csv")


Building daily asset features:   0%|          | 0/10 [00:00<?, ?ticker/s]

Building monthly panel:   0%|          | 0/244 [00:00<?, ?month/s]

,date,asset,target_top_k_next_1m,forward_1m_return,forward_1m_close_to_close_return,forward_1m_next_open_return,forward_1m_excess_return,forward_1m_rank,forward_1m_benchmark_return,forward_1m_vs_benchmark_return,...,market_spy_drawdown_126d,market_spy_drawdown_252d,market_spy_ma_distance_63d,market_spy_ma_distance_126d,market_spy_ma_distance_200d,market_spy_volume_zscore_63d,market_spy_avg_dollar_volume_63d,market_spy_hl_spread_proxy_21d,month,month_index
0,2006-01-31,EEM,0,-0.029831,-0.038500,-0.029831,-0.020382,9.0,0.006103,-0.035934,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.008240,2006-01,0
1,2006-01-31,EFA,0,-0.000159,-0.007000,-0.000159,0.009290,5.0,0.006103,-0.006262,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.008240,2006-01,0
2,2006-01-31,GLD,0,-0.013559,-0.011111,-0.013559,-0.004109,8.0,0.006103,-0.019661,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.008240,2006-01,0
3,2006-01-31,IEF,0,-0.001123,-0.001036,-0.001123,0.008327,6.0,0.006103,-0.007225,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.008240,2006-01,0
4,2006-01-31,IWM,0,0.002480,0.003179,0.002480,0.011929,4.0,0.006103,-0.003623,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.008240,2006-01,0
5,2006-01-31,QQQ,0,-0.012218,-0.021429,-0.012218,-0.002769,7.0,0.006103,-0.018321,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.008240,2006-01,0
6,2006-01-31,SPY,1,0.006103,0.005726,0.006103,0.015552,3.0,0.006103,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2006-01,0
7,2006-01-31,TLT,1,0.010417,0.011079,0.010417,0.019866,2.0,0.006103,0.004314,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.008240,2006-01,0
8,2006-01-31,XLE,0,-0.082623,-0.092265,-0.082623,-0.073173,10.0,0.006103,-0.088725,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.008240,2006-01,0
9,2006-01-31,XLF,1,0.026018,0.021283,0.026018,0.035468,1.0,0.006103,0.019916,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.008240,2006-01,0


,month,rows,positive_rows,mean_forward_return
231,2025-04,10,3,0.031947
232,2025-05,10,3,0.037142
233,2025-06,10,3,0.003323
234,2025-07,10,3,0.025035
235,2025-08,10,3,0.044396
236,2025-09,10,3,0.017784
237,2025-10,10,3,0.003197
238,2025-11,10,3,0.017093
239,2025-12,10,3,0.030712
240,2026-01,10,3,0.029340


Saved tabpfn_tabicl_tactical_asset_allocation_20260512_v2_outputs/model_frame_head.csv
Saved tabpfn_tabicl_tactical_asset_allocation_20260512_v2_outputs/monthly_target_preview.csv


...

,horizon_months,rows,positive_rate,mean_forward_return
0,3,2410,0.3,0.0226
1,6,2380,0.3,0.0458


Saved tabpfn_tabicl_tactical_asset_allocation_20260512_v2_outputs/multi_horizon_target_summary.csv


,benchmark_asset,rows,outperform_rate,mean_vs_benchmark_return
0,SPY,2430,0.4082,-0.0023


Saved tabpfn_tabicl_tactical_asset_allocation_20260512_v2_outputs/benchmark_relative_target_summary.csv


PosixPath('tabpfn_tabicl_tactical_asset_allocation_20260512_v2_outputs/benchmark_relative_target_summary.csv')

## 3. Chronological Splits and Feature Policy

The model-selection period precedes the calibration and holdout periods. Feature filtering and median imputation are fitted only on the model-selection source window, then reused for calibration and holdout. This avoids using holdout distribution information when preparing model matrices.

In [4]:
date_values = pd.to_datetime(model_frame["date"])
context_mask = date_values <= pd.Timestamp(CONTEXT_END_DATE)
selection_mask = date_values <= pd.Timestamp(TUNING_END_DATE)
calibration_mask = (date_values > pd.Timestamp(TUNING_END_DATE)) & (date_values <= pd.Timestamp(CALIBRATION_END_DATE))
holdout_mask = date_values >= pd.Timestamp(HOLDOUT_START_DATE)
preholdout_mask = date_values <= pd.Timestamp(CALIBRATION_END_DATE)

context_df = model_frame.loc[context_mask].reset_index(drop=True)
selection_df = model_frame.loc[selection_mask].reset_index(drop=True)
calibration_df = model_frame.loc[calibration_mask].reset_index(drop=True)
holdout_df = model_frame.loc[holdout_mask].reset_index(drop=True)
preholdout_df = model_frame.loc[preholdout_mask].reset_index(drop=True)

if len(selection_df) == 0 or len(calibration_df) == 0 or len(holdout_df) == 0:
    raise RuntimeError("One or more required chronological windows are empty. Review split dates and data availability.")

split_summary = pd.DataFrame(
    [
        {"split": "context", "rows": len(context_df), "months": context_df["month"].nunique(), "positive_rows": int(context_df["target_top_k_next_1m"].sum()), "positive_rate": context_df["target_top_k_next_1m"].mean(), "start": context_df["date"].min(), "end": context_df["date"].max()},
        {"split": "model_selection", "rows": len(selection_df), "months": selection_df["month"].nunique(), "positive_rows": int(selection_df["target_top_k_next_1m"].sum()), "positive_rate": selection_df["target_top_k_next_1m"].mean(), "start": selection_df["date"].min(), "end": selection_df["date"].max()},
        {"split": "calibration", "rows": len(calibration_df), "months": calibration_df["month"].nunique(), "positive_rows": int(calibration_df["target_top_k_next_1m"].sum()), "positive_rate": calibration_df["target_top_k_next_1m"].mean(), "start": calibration_df["date"].min(), "end": calibration_df["date"].max()},
        {"split": "preholdout", "rows": len(preholdout_df), "months": preholdout_df["month"].nunique(), "positive_rows": int(preholdout_df["target_top_k_next_1m"].sum()), "positive_rate": preholdout_df["target_top_k_next_1m"].mean(), "start": preholdout_df["date"].min(), "end": preholdout_df["date"].max()},
        {"split": "holdout", "rows": len(holdout_df), "months": holdout_df["month"].nunique(), "positive_rows": int(holdout_df["target_top_k_next_1m"].sum()), "positive_rate": holdout_df["target_top_k_next_1m"].mean(), "start": holdout_df["date"].min(), "end": holdout_df["date"].max()},
    ]
)
display(split_summary)
save_artifact_table(split_summary, "split_summary.csv")

NON_FEATURE_COLUMNS = {
    "date",
    "month",
    "asset",
    "month_index",
    "target_top_k_next_1m",
    "forward_1m_return",
    "forward_1m_close_to_close_return",
    "forward_1m_next_open_return",
    "forward_1m_excess_return",
    "forward_1m_rank",
    "universe_forward_1m_return",
    "execution_return_mode",
}
feature_candidates = [column for column in model_frame.columns if column not in NON_FEATURE_COLUMNS]
feature_candidates = [column for column in feature_candidates if not column.startswith("target_") and not column.startswith("forward_")]
feature_candidates = [column for column in feature_candidates if pd.api.types.is_numeric_dtype(model_frame[column])]

raw_feature_candidates = list(feature_candidates)
ticker_identity_feature_columns = [column for column in raw_feature_candidates if column.startswith("asset_is_")]
group_metadata_feature_columns = [column for column in raw_feature_candidates if column.startswith("asset_group_") or column == "asset_risk_bucket"]
static_identity_feature_columns = sorted(set(ticker_identity_feature_columns + group_metadata_feature_columns))
asset_time_series_prefixes = (
    "asset_return_",
    "asset_rv_",
    "asset_downside_rv_",
    "asset_drawdown_",
    "asset_ma_distance_",
    "asset_volume_",
    "asset_avg_dollar_volume_",
    "asset_hl_spread_proxy_",
    "asset_beta_to_",
    "asset_month_return",
    "market_",
)
strict_time_series_feature_columns = [column for column in raw_feature_candidates if column.startswith(asset_time_series_prefixes)]


def feature_columns_for_variant(variant):
    if variant == "full":
        excluded = []
        columns = list(raw_feature_candidates)
    elif variant == "ticker_ablated":
        excluded = ticker_identity_feature_columns
        columns = [column for column in raw_feature_candidates if column not in set(excluded)]
    elif variant == "identity_ablated":
        excluded = static_identity_feature_columns
        columns = [column for column in raw_feature_candidates if column not in set(excluded)]
    elif variant == "metadata_only":
        columns = list(static_identity_feature_columns)
        excluded = [column for column in raw_feature_candidates if column not in set(columns)]
    elif variant == "strict_time_series":
        columns = list(strict_time_series_feature_columns)
        excluded = [column for column in raw_feature_candidates if column not in set(columns)]
    else:
        raise ValueError(f"Unsupported feature variant: {variant}")
    return columns, excluded


feature_candidates, excluded_variant_feature_columns = feature_columns_for_variant(FEATURE_SET_VARIANT)
feature_variant_summary = pd.DataFrame(
    [
        {"metric": "feature_set_variant", "value": FEATURE_SET_VARIANT},
        {"metric": "raw_numeric_feature_candidates", "value": len(raw_feature_candidates)},
        {"metric": "ticker_identity_feature_columns", "value": len(ticker_identity_feature_columns)},
        {"metric": "group_metadata_feature_columns", "value": len(group_metadata_feature_columns)},
        {"metric": "strict_time_series_feature_columns", "value": len(strict_time_series_feature_columns)},
        {"metric": "excluded_variant_feature_columns", "value": len(excluded_variant_feature_columns)},
        {"metric": "post_variant_feature_candidates", "value": len(feature_candidates)},
    ]
)
display(feature_variant_summary)
save_artifact_table(feature_variant_summary, "feature_variant_summary.csv", description="Feature-family exclusions used for the configured feature-set variant.")
if excluded_variant_feature_columns:
    save_artifact_table(pd.DataFrame({"excluded_feature": sorted(excluded_variant_feature_columns)}), "feature_variant_excluded_columns.csv")

def feature_quality_for_columns(columns):
    if not columns:
        return pd.DataFrame(columns=["feature", "missing_rate_selection", "non_null_selection", "n_unique_selection"])
    return pd.DataFrame(
        {
            "feature": columns,
            "missing_rate_selection": [selection_df[column].isna().mean() for column in columns],
            "non_null_selection": [selection_df[column].notna().sum() for column in columns],
            "n_unique_selection": [selection_df[column].nunique(dropna=True) for column in columns],
        }
    )


def retained_columns_from_quality(quality):
    if len(quality) == 0:
        return []
    return quality.loc[
        (quality["missing_rate_selection"] <= FEATURE_MAX_MISSING_RATE)
        & (quality["non_null_selection"] >= FEATURE_MIN_SELECTION_OBSERVATIONS)
        & (quality["n_unique_selection"] > 1),
        "feature",
    ].tolist()


def fit_feature_transform_policy(columns):
    quality = feature_quality_for_columns(columns)
    retained = retained_columns_from_quality(quality)
    missing_indicators = [column for column in retained if selection_df[column].isna().any()]
    medians = selection_df[retained].median(numeric_only=True).replace([np.inf, -np.inf], np.nan).fillna(0.0) if retained else pd.Series(dtype=float)
    return quality, retained, missing_indicators, medians


def transform_with_feature_policy(frame, retained, medians, missing_indicators):
    if not retained:
        return pd.DataFrame(index=frame.index)
    numeric = frame[retained].copy().replace([np.inf, -np.inf], np.nan)
    indicator_parts = []
    for column in missing_indicators:
        indicator_parts.append(numeric[column].isna().astype(np.float32).rename(f"{column}_is_missing"))
    numeric = numeric.fillna(medians).astype(np.float32)
    if indicator_parts:
        numeric = pd.concat([numeric, pd.concat(indicator_parts, axis=1).astype(np.float32)], axis=1)
    return numeric.astype(np.float32)


feature_variant_policy_rows = []
feature_variant_bundles = {}
for variant in FEATURE_SENSITIVITY_VARIANTS:
    variant_candidates, variant_excluded = feature_columns_for_variant(variant)
    variant_quality, variant_retained, variant_missing_indicators, variant_medians = fit_feature_transform_policy(variant_candidates)
    feature_variant_policy_rows.append(
        {
            "feature_variant": variant,
            "candidate_features": len(variant_candidates),
            "excluded_features": len(variant_excluded),
            "retained_base_features": len(variant_retained),
            "missing_indicator_features": len(variant_missing_indicators),
            "final_model_features": len(variant_retained) + len(variant_missing_indicators),
            "selected_for_main_run": variant == FEATURE_SET_VARIANT,
            "sensitivity_enabled": FEATURE_VARIANT_SENSITIVITY_ENABLED,
        }
    )
    if FEATURE_VARIANT_SENSITIVITY_ENABLED and variant_retained:
        feature_variant_bundles[variant] = {
            "retained_columns": variant_retained,
            "missing_indicator_columns": variant_missing_indicators,
            "feature_medians": variant_medians,
            "X_preholdout": transform_with_feature_policy(preholdout_df, variant_retained, variant_medians, variant_missing_indicators),
            "X_holdout": transform_with_feature_policy(holdout_df, variant_retained, variant_medians, variant_missing_indicators),
        }
feature_variant_policy_summary = pd.DataFrame(feature_variant_policy_rows)
display(feature_variant_policy_summary)
save_artifact_table(feature_variant_policy_summary, "feature_variant_policy_summary.csv", description="Comparable feature-family policies for identity and time-series sensitivity checks.")

selection_feature_quality, retained_feature_columns, missing_indicator_columns, feature_medians = fit_feature_transform_policy(feature_candidates)

if not retained_feature_columns:
    raise RuntimeError("No retained feature columns after the feature policy. Relax feature policy thresholds or review data loading.")


def transform_features(frame):
    return transform_with_feature_policy(frame, retained_feature_columns, feature_medians, missing_indicator_columns)

X_selection = transform_features(selection_df)
y_selection = selection_df["target_top_k_next_1m"].astype(int)
X_calibration = transform_features(calibration_df)
y_calibration = calibration_df["target_top_k_next_1m"].astype(int)
X_holdout = transform_features(holdout_df)
y_holdout = holdout_df["target_top_k_next_1m"].astype(int)
X_preholdout = transform_features(preholdout_df)
y_preholdout = preholdout_df["target_top_k_next_1m"].astype(int)

feature_policy_summary = pd.DataFrame(
    [
        {"metric": "feature_set_variant", "value": FEATURE_SET_VARIANT},
        {"metric": "candidate_numeric_features_after_variant", "value": len(feature_candidates)},
        {"metric": "excluded_variant_features", "value": len(excluded_variant_feature_columns)},
        {"metric": "retained_base_features", "value": len(retained_feature_columns)},
        {"metric": "missing_indicator_features", "value": len(missing_indicator_columns)},
        {"metric": "final_model_features", "value": X_selection.shape[1]},
        {"metric": "feature_max_missing_rate", "value": FEATURE_MAX_MISSING_RATE},
        {"metric": "feature_min_selection_observations", "value": FEATURE_MIN_SELECTION_OBSERVATIONS},
    ]
)
display(feature_policy_summary)
save_artifact_table(feature_policy_summary, "feature_policy_summary.csv")
save_artifact_table(selection_feature_quality.sort_values("missing_rate_selection", ascending=False), "feature_quality_summary.csv")

feature_bundle_summary = pd.DataFrame(
    [
        {"bundle": "selection", "rows": X_selection.shape[0], "features": X_selection.shape[1], "positive_rows": int(y_selection.sum()), "positive_rate": y_selection.mean()},
        {"bundle": "calibration", "rows": X_calibration.shape[0], "features": X_calibration.shape[1], "positive_rows": int(y_calibration.sum()), "positive_rate": y_calibration.mean()},
        {"bundle": "preholdout", "rows": X_preholdout.shape[0], "features": X_preholdout.shape[1], "positive_rows": int(y_preholdout.sum()), "positive_rate": y_preholdout.mean()},
        {"bundle": "holdout", "rows": X_holdout.shape[0], "features": X_holdout.shape[1], "positive_rows": int(y_holdout.sum()), "positive_rate": y_holdout.mean()},
    ]
)
display(feature_bundle_summary)
save_artifact_table(feature_bundle_summary, "feature_bundle_summary.csv")


,split,rows,months,positive_rows,positive_rate,start,end
0,context,720,72,216,0.3,2006-01-31,2011-12-31
1,model_selection,1440,144,432,0.3,2006-01-31,2017-12-31
2,calibration,240,24,72,0.3,2018-01-31,2019-12-31
3,preholdout,1680,168,504,0.3,2006-01-31,2019-12-31
4,holdout,750,75,225,0.3,2020-01-31,2026-03-31


Saved tabpfn_tabicl_tactical_asset_allocation_20260512_v2_outputs/split_summary.csv


,metric,value
0,feature_set_variant,identity_ablated
1,raw_numeric_feature_candidates,252
2,ticker_identity_feature_columns,10
3,group_metadata_feature_columns,9
4,strict_time_series_feature_columns,233
5,excluded_variant_feature_columns,19
6,post_variant_feature_candidates,233


Saved tabpfn_tabicl_tactical_asset_allocation_20260512_v2_outputs/feature_variant_summary.csv
Saved tabpfn_tabicl_tactical_asset_allocation_20260512_v2_outputs/feature_variant_excluded_columns.csv


,feature_variant,candidate_features,excluded_features,retained_base_features,missing_indicator_features,final_model_features,selected_for_main_run,sensitivity_enabled
0,full,252,0,250,226,476,False,True
1,ticker_ablated,242,10,240,226,466,False,True
2,identity_ablated,233,19,233,226,459,True,True
3,metadata_only,19,233,17,0,17,False,True
4,strict_time_series,233,19,233,226,459,False,True


...

,metric,value
0,feature_set_variant,identity_ablated
1,candidate_numeric_features_after_variant,233
2,excluded_variant_features,19
3,retained_base_features,233
4,missing_indicator_features,226
5,final_model_features,459
6,feature_max_missing_rate,0.35
7,feature_min_selection_observations,36


Saved tabpfn_tabicl_tactical_asset_allocation_20260512_v2_outputs/feature_policy_summary.csv
Saved tabpfn_tabicl_tactical_asset_allocation_20260512_v2_outputs/feature_quality_summary.csv


,bundle,rows,features,positive_rows,positive_rate
0,selection,1440,459,432,0.3
1,calibration,240,459,72,0.3
2,preholdout,1680,459,504,0.3
3,holdout,750,459,225,0.3


Saved tabpfn_tabicl_tactical_asset_allocation_20260512_v2_outputs/feature_bundle_summary.csv


PosixPath('tabpfn_tabicl_tactical_asset_allocation_20260512_v2_outputs/feature_bundle_summary.csv')

## 4. Model Registry, Tuning, and Evaluation Helpers

The default learned model path is GPU-accelerated XGBoost with randomized hyperparameter search over broad distributions plus a direct TabPFN 2.6 scorer. TabICL code is retained behind `RUN_DIRECT_TABICL`, but the default run disables it because the full Kaggle run exceeded RAM in that step, so the TabICL empirical comparison remains unresolved by design. Deterministic rules are included because tactical allocation workflows should be compared with simple finance baselines before adding model complexity. Feature-variant sensitivity uses fixed GPU XGBoost and direct TabPFN 2.6 by default; CPU Logistic Regression remains only as a disabled optional benchmark path.

In [5]:
def y_to_numpy(y):
    if hasattr(y, "to_numpy"):
        return y.to_numpy(dtype=np.int32)
    return np.asarray(y, dtype=np.int32)


def to_numpy_float32(X):
    if hasattr(X, "to_numpy"):
        return X.to_numpy(dtype=np.float32)
    return np.asarray(X, dtype=np.float32)


def to_cupy_float32(X):
    return cp.asarray(to_numpy_float32(X))


def take_rows(X, row_indices):
    if hasattr(X, "iloc"):
        return X.iloc[row_indices]
    return X[row_indices]


def safe_metric(metric_fn, y_true, y_score):
    y_true_values = y_to_numpy(y_true)
    y_score_values = np.asarray(y_score, dtype=float)
    if len(np.unique(y_true_values)) < 2:
        return np.nan
    try:
        return float(metric_fn(y_true_values, y_score_values))
    except Exception:
        return np.nan


def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = pd.Series(y_to_numpy(y_true))
    y_proba = pd.Series(np.asarray(y_proba, dtype=float)).clip(0.0, 1.0)
    if y_proba.nunique(dropna=True) <= 1:
        return np.nan
    n_bins = min(n_bins, y_proba.nunique(dropna=True))
    try:
        bins = pd.qcut(y_proba, q=n_bins, duplicates="drop")
    except ValueError:
        return np.nan
    frame = pd.DataFrame({"y_true": y_true, "y_proba": y_proba, "bin": bins})
    grouped = frame.groupby("bin", observed=True)
    weights = grouped.size() / len(frame)
    observed = grouped["y_true"].mean()
    predicted = grouped["y_proba"].mean()
    return float((weights * (observed - predicted).abs()).sum())


def calibration_bin_table(y_true, y_proba, n_bins=10):
    y_true = pd.Series(y_to_numpy(y_true))
    y_proba = pd.Series(np.asarray(y_proba, dtype=float)).clip(0.0, 1.0)
    n_bins = min(n_bins, max(1, y_proba.nunique(dropna=True)))
    try:
        bins = pd.qcut(y_proba, q=n_bins, duplicates="drop")
    except ValueError:
        bins = pd.cut(y_proba, bins=n_bins, include_lowest=True, duplicates="drop")
    frame = pd.DataFrame({"y_true": y_true, "y_proba": y_proba, "bin": bins})
    return (
        frame.groupby("bin", observed=True)
        .agg(rows=("y_true", "size"), observed_rate=("y_true", "mean"), mean_predicted_probability=("y_proba", "mean"), score_min=("y_proba", "min"), score_max=("y_proba", "max"))
        .reset_index()
    )


def positive_class_proba(model, X):
    proba = model.predict_proba(X)
    proba = cp.asnumpy(proba) if hasattr(proba, "get") else np.asarray(proba)
    if proba.ndim == 1:
        return proba.astype(float)
    if proba.shape[1] == 1:
        return proba[:, 0].astype(float)
    return proba[:, 1].astype(float)


def predict_proba_in_chunks(model, X, chunk_size=PREDICTION_CHUNK_SIZE, desc="Predicting"):
    if len(X) <= chunk_size:
        return positive_class_proba(model, X)
    parts = []
    for start in tqdm(range(0, len(X), chunk_size), desc=desc, unit="chunk", leave=False):
        stop = min(start + chunk_size, len(X))
        parts.append(positive_class_proba(model, take_rows(X, np.arange(start, stop))))
    return np.concatenate(parts)


class GPUXGBClassifier(XGBClassifier):
    def fit(self, X, y, **kwargs):
        fit_kwargs = dict(kwargs)
        if XGBOOST_DEVICE == "cuda":
            X_fit = to_cupy_float32(X)
            y_fit = cp.asarray(y_to_numpy(y))
            if fit_kwargs.get("eval_set") is not None:
                fit_kwargs["eval_set"] = [(to_cupy_float32(eval_X), cp.asarray(y_to_numpy(eval_y))) for eval_X, eval_y in fit_kwargs["eval_set"]]
        else:
            X_fit = to_numpy_float32(X)
            y_fit = y_to_numpy(y)
            if fit_kwargs.get("eval_set") is not None:
                fit_kwargs["eval_set"] = [(to_numpy_float32(eval_X), y_to_numpy(eval_y)) for eval_X, eval_y in fit_kwargs["eval_set"]]
        return super().fit(X_fit, y_fit, **fit_kwargs)

    def predict_proba(self, X, **kwargs):
        X_predict = to_cupy_float32(X) if XGBOOST_DEVICE == "cuda" else to_numpy_float32(X)
        proba = super().predict_proba(X_predict, **kwargs)
        return cp.asnumpy(proba) if hasattr(proba, "get") else np.asarray(proba)

    def predict(self, X, **kwargs):
        X_predict = to_cupy_float32(X) if XGBOOST_DEVICE == "cuda" else to_numpy_float32(X)
        prediction = super().predict(X_predict, **kwargs)
        return cp.asnumpy(prediction) if hasattr(prediction, "get") else np.asarray(prediction)


def make_prefit_calibrator(base_model, X_cal, y_cal, method="sigmoid"):
    try:
        calibrated = CalibratedClassifierCV(estimator=base_model, method=method, cv="prefit")
    except TypeError:
        calibrated = CalibratedClassifierCV(base_estimator=base_model, method=method, cv="prefit")
    calibrated.fit(X_cal, y_cal)
    return calibrated


def make_walk_forward_month_splits(frame, y, n_splits=CLASSICAL_TUNING_CV_SPLITS, min_train_positives=MIN_CV_TRAIN_POSITIVES, min_validation_positives=MIN_CV_VALIDATION_POSITIVES):
    y_values = y_to_numpy(y)
    months = pd.Series(frame["month"].to_numpy())
    unique_months = months.drop_duplicates().to_numpy()
    if len(unique_months) < n_splits + 2:
        n_splits = max(1, len(unique_months) - 2)
    min_train_months = max(24, int(len(unique_months) * 0.35))
    boundaries = np.unique(np.linspace(min_train_months, len(unique_months), n_splits + 1, dtype=int))
    splits = []
    rows = []
    for fold_idx, (validation_start, validation_stop) in enumerate(zip(boundaries[:-1], boundaries[1:]), start=1):
        train_months = set(unique_months[:validation_start])
        validation_months = set(unique_months[validation_start:validation_stop])
        train_idx = np.where(months.isin(train_months).to_numpy())[0]
        validation_idx = np.where(months.isin(validation_months).to_numpy())[0]
        train_pos = int(y_values[train_idx].sum()) if len(train_idx) else 0
        valid_pos = int(y_values[validation_idx].sum()) if len(validation_idx) else 0
        keep = len(train_idx) > 0 and len(validation_idx) > 0 and train_pos >= min_train_positives and valid_pos >= min_validation_positives
        rows.append(
            {
                "fold": fold_idx,
                "train_rows": len(train_idx),
                "validation_rows": len(validation_idx),
                "train_positive_rows": train_pos,
                "validation_positive_rows": valid_pos,
                "train_start": min(train_months) if train_months else "",
                "train_end": max(train_months) if train_months else "",
                "validation_start": min(validation_months) if validation_months else "",
                "validation_end": max(validation_months) if validation_months else "",
                "used": keep,
            }
        )
        if keep:
            splits.append((train_idx, validation_idx))
    if not splits:
        # Fallback to the last 20 percent of months as validation if the positive-count constraints are too strict.
        split_point = max(1, int(len(unique_months) * 0.80))
        train_months = set(unique_months[:split_point])
        validation_months = set(unique_months[split_point:])
        train_idx = np.where(months.isin(train_months).to_numpy())[0]
        validation_idx = np.where(months.isin(validation_months).to_numpy())[0]
        splits = [(train_idx, validation_idx)]
        rows.append({"fold": "fallback_last_month_block", "train_rows": len(train_idx), "validation_rows": len(validation_idx), "train_positive_rows": int(y_values[train_idx].sum()), "validation_positive_rows": int(y_values[validation_idx].sum()), "used": True})
    return splits, pd.DataFrame(rows)


class ProgressRandomizedSearchCV:
    def __init__(self, estimator, param_distributions, n_iter, cv_splits, random_state=SEED, error_score=np.nan, label="model search", early_stopping_rounds=None):
        self.estimator = estimator
        self.param_distributions = param_distributions
        self.n_iter = int(n_iter)
        self.cv = list(cv_splits)
        self.random_state = random_state
        self.error_score = error_score
        self.label = label
        self.early_stopping_rounds = int(early_stopping_rounds) if early_stopping_rounds else None

    def planned_fit_count(self):
        return self.n_iter * len(self.cv)

    def uses_early_stopping(self):
        return bool(self.early_stopping_rounds and self.early_stopping_rounds > 0)

    def _fit_fold_estimator(self, estimator, X, y, train_idx, validation_idx):
        fit_kwargs = {}
        if self.uses_early_stopping() and isinstance(estimator, XGBClassifier):
            estimator.set_params(early_stopping_rounds=self.early_stopping_rounds)
            fit_kwargs["eval_set"] = [(take_rows(X, validation_idx), take_rows(y, validation_idx))]
            fit_kwargs["verbose"] = False
        estimator.fit(take_rows(X, train_idx), take_rows(y, train_idx), **fit_kwargs)
        return estimator

    def cv_results_frame(self):
        if not hasattr(self, "trial_rows_"):
            return pd.DataFrame()
        rows = []
        for row in self.trial_rows_:
            cleaned = dict(row)
            cleaned["params"] = artifact_json(cleaned.get("params", {}))
            cleaned["fold_scores"] = artifact_json(cleaned.get("fold_scores", []))
            cleaned["fold_best_iterations"] = artifact_json(cleaned.get("fold_best_iterations", []))
            rows.append(cleaned)
        return pd.DataFrame(rows)

    def fit(self, X, y):
        parameter_draws = list(ParameterSampler(self.param_distributions, n_iter=self.n_iter, random_state=self.random_state))
        if not parameter_draws:
            raise RuntimeError(f"{self.label}: no hyperparameter draws were generated.")
        if not self.cv:
            raise RuntimeError(f"{self.label}: no cross-validation folds were available.")
        trial_rows = []
        best_score = -np.inf
        best_params = None
        progress = tqdm(parameter_draws, desc=f"Tuning {self.label}", unit="trial")
        for trial_idx, params in enumerate(progress, start=1):
            fold_scores = []
            fold_errors = []
            fold_best_iterations = []
            for fold_idx, (train_idx, validation_idx) in enumerate(self.cv, start=1):
                estimator = clone(self.estimator)
                estimator.set_params(**params)
                try:
                    estimator = self._fit_fold_estimator(estimator, X, y, train_idx, validation_idx)
                    validation_score = positive_class_proba(estimator, take_rows(X, validation_idx))
                    fold_score = safe_metric(average_precision_score, take_rows(y, validation_idx), validation_score)
                    best_iteration = getattr(estimator, "best_iteration", None)
                    if best_iteration is None:
                        best_iteration = getattr(estimator, "best_iteration_", None)
                    fold_best_iterations.append(int(best_iteration) + 1 if best_iteration is not None else np.nan)
                except Exception as exc:
                    fold_score = self.error_score if np.isscalar(self.error_score) else np.nan
                    fold_errors.append(f"fold_{fold_idx}: {short_error(exc)}")
                    fold_best_iterations.append(np.nan)
                fold_scores.append(fold_score)
                del estimator
                if XGBOOST_DEVICE == "cuda":
                    try:
                        cp.get_default_memory_pool().free_all_blocks()
                    except Exception:
                        pass
            fold_scores_array = np.asarray(fold_scores, dtype=float)
            best_iterations_array = np.asarray(fold_best_iterations, dtype=float)
            mean_score = float(np.nanmean(fold_scores_array)) if np.isfinite(fold_scores_array).any() else np.nan
            std_score = float(np.nanstd(fold_scores_array)) if np.isfinite(fold_scores_array).any() else np.nan
            mean_best_iteration = float(np.nanmean(best_iterations_array)) if np.isfinite(best_iterations_array).any() else np.nan
            max_best_iteration = float(np.nanmax(best_iterations_array)) if np.isfinite(best_iterations_array).any() else np.nan
            trial_rows.append({"trial": trial_idx, "params": params, "mean_test_score": mean_score, "std_test_score": std_score, "fold_scores": fold_scores, "mean_best_iteration": mean_best_iteration, "max_best_iteration": max_best_iteration, "fold_best_iterations": fold_best_iterations, "early_stopping_rounds": self.early_stopping_rounds if self.uses_early_stopping() else 0, "errors": "; ".join(fold_errors)})
            if np.isfinite(mean_score) and mean_score > best_score:
                best_score = mean_score
                best_params = params
            progress.set_postfix(best_ap=f"{best_score:.4f}" if np.isfinite(best_score) else "nan")
        if best_params is None:
            raise RuntimeError(f"{self.label}: all randomized-search trials failed or produced NaN scores.")
        self.best_params_ = best_params
        self.best_score_ = float(best_score)
        self.trial_rows_ = trial_rows
        self.cv_results_ = {
            "params": [row["params"] for row in trial_rows],
            "mean_test_score": np.asarray([row["mean_test_score"] for row in trial_rows], dtype=float),
            "std_test_score": np.asarray([row["std_test_score"] for row in trial_rows], dtype=float),
            "mean_best_iteration": np.asarray([row["mean_best_iteration"] for row in trial_rows], dtype=float),
            "max_best_iteration": np.asarray([row["max_best_iteration"] for row in trial_rows], dtype=float),
            "errors": [row["errors"] for row in trial_rows],
        }
        return self


def best_estimator_from_search(search_model):
    estimator = clone(search_model.estimator)
    estimator.set_params(**search_model.best_params_)
    return estimator


cv_splits, cv_split_summary = make_walk_forward_month_splits(selection_df, y_selection)
display(cv_split_summary)
save_artifact_table(cv_split_summary, "cv_split_summary.csv")


,fold,train_rows,validation_rows,train_positive_rows,validation_positive_rows,train_start,train_end,validation_start,validation_end,used
0,1,500,470,150,141,2006-01,2010-02,2010-03,2014-01,True
1,2,970,470,291,141,2006-01,2014-01,2014-02,2017-12,True


Saved tabpfn_tabicl_tactical_asset_allocation_20260512_v2_outputs/cv_split_summary.csv


PosixPath('tabpfn_tabicl_tactical_asset_allocation_20260512_v2_outputs/cv_split_summary.csv')

## 5. Run Allocation Scorers

This section creates deterministic rule scores, runs fixed GPU XGBoost and direct TabPFN 2.6 feature-variant sensitivity checks, tunes the main GPU XGBoost scorer with chronological fold-level early stopping, and evaluates direct TabPFN 2.6. CPU Logistic Regression remains available only as a disabled optional benchmark path. TabICL remains guarded but disabled in the default run, so the TabICL empirical comparison is a known unresolved item. The learned models produce probabilities for the next-month top-k label. Portfolio diagnostics use the scores only for ranking assets inside each month.

In [6]:
def metric_row(model_name, base_model, family, calibration, evaluation_window, y_true, y_score, score_type, fit_seconds=0.0, predict_seconds=0.0, calibration_seconds=0.0, cv_average_precision=np.nan, best_params="", timing_notes=""):
    y_true_values = y_to_numpy(y_true)
    y_score_values = np.asarray(y_score, dtype=float)
    row = {
        "Model": model_name,
        "Base Model": base_model,
        "Family": family,
        "Calibration": calibration,
        "Evaluation Window": evaluation_window,
        "Rows": len(y_true_values),
        "Positive Rows": int(y_true_values.sum()),
        "Positive Rate": float(y_true_values.mean()) if len(y_true_values) else np.nan,
        "Score Type": score_type,
        "Average Precision": safe_metric(average_precision_score, y_true_values, y_score_values),
        "ROC AUC": safe_metric(roc_auc_score, y_true_values, y_score_values),
        "Brier Score": brier_score_loss(y_true_values, np.clip(y_score_values, 0.0, 1.0)) if score_type == "probability" and len(np.unique(y_true_values)) > 1 else np.nan,
        "Log Loss": log_loss(y_true_values, np.clip(y_score_values, 1e-6, 1.0 - 1e-6), labels=[0, 1]) if score_type == "probability" and len(np.unique(y_true_values)) > 1 else np.nan,
        "ECE Quantile 10": expected_calibration_error(y_true_values, y_score_values, n_bins=10) if score_type == "probability" else np.nan,
        "Fit Seconds": fit_seconds,
        "Predict Seconds": predict_seconds,
        "Calibration Seconds": calibration_seconds,
        "Workflow Seconds": fit_seconds + predict_seconds + calibration_seconds,
        "CV/Validation Average Precision": cv_average_precision,
        "Best Params": best_params,
        "Timing Notes": timing_notes,
    }
    return row


def build_prediction_frame(model_name, frame, scores, evaluation_window):
    columns = [
        "date",
        "month",
        "asset",
        "target_top_k_next_1m",
        "forward_1m_return",
        "forward_1m_close_to_close_return",
        "forward_1m_next_open_return",
        "forward_1m_benchmark_return",
        "forward_1m_vs_benchmark_return",
        "target_outperform_benchmark_next_1m",
        "forward_1m_excess_return",
        "forward_1m_rank",
    ]
    for horizon in MULTI_HORIZON_MONTHS:
        columns.extend([f"target_top_k_next_{horizon}m", f"forward_{horizon}m_return", f"forward_{horizon}m_excess_return", f"forward_{horizon}m_rank"])
    available_columns = [column for column in columns if column in frame.columns]
    out = frame[available_columns].copy()
    out["Model"] = model_name
    out["Evaluation Window"] = evaluation_window
    out["score"] = np.asarray(scores, dtype=np.float32)
    out["target_top_k_next_1m"] = out["target_top_k_next_1m"].astype(np.int8)
    return out


def add_model_predictions(model_name, base_model, family, calibration, score_type, eval_payloads, fit_seconds=0.0, calibration_seconds=0.0, cv_average_precision=np.nan, best_params="", timing_notes=""):
    for evaluation_window, frame, y_true, y_score, predict_seconds in eval_payloads:
        y_score_values = np.asarray(y_score, dtype=np.float32)
        predictions[(model_name, evaluation_window)] = y_score_values
        prediction_frames.append(build_prediction_frame(model_name, frame, y_score_values, evaluation_window))
        model_rows.append(metric_row(model_name, base_model, family, calibration, evaluation_window, y_true, y_score_values, score_type, fit_seconds, predict_seconds, calibration_seconds, cv_average_precision, best_params, timing_notes))


def add_rule_score(model_name, score_column, ascending=False):
    for evaluation_window, frame, y_true in [("calibration", calibration_df, y_calibration), ("holdout", holdout_df, y_holdout)]:
        if score_column not in frame.columns:
            print(f"Skipping {model_name}; missing {score_column}.")
            return
        score = frame[score_column].astype(float).to_numpy()
        if ascending:
            score = -score
        add_model_predictions(model_name, "Rule", "deterministic_rule", "none", "raw_score", [(evaluation_window, frame, y_true, score, 0.0)], timing_notes="deterministic rule score; not a calibrated probability")


# Deterministic allocation rule scores.
prior_score = np.repeat(y_preholdout.mean(), len(holdout_df))
add_model_predictions("Dummy[Preholdout prior probability]", "Dummy", "baseline", "none", "probability", [("holdout", holdout_df, y_holdout, prior_score, 0.0)], timing_notes="constant prior from pre-holdout label frequency")
add_rule_score("Rule[12M momentum top-k]", "asset_return_252d", ascending=False)
add_rule_score("Rule[6M momentum top-k]", "asset_return_126d", ascending=False)
add_rule_score("Rule[Low volatility top-k]", "asset_rv_63d", ascending=True)
add_rule_score("Rule[Risk-adjusted momentum top-k]", "asset_ma_distance_126d", ascending=False)


# Feature-variant sensitivity is handled by evaluate_model_family_feature_sensitivity() below.
# The previous CPU Logistic Regression sensitivity path is intentionally disabled by default.


def make_light_xgboost_estimator(y, n_estimators):
    positive = max(int(np.asarray(y).sum()), 1)
    negative = max(int(len(y) - np.asarray(y).sum()), 1)
    return GPUXGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method=XGBOOST_TREE_METHOD,
        device=XGBOOST_DEVICE,
        random_state=SEED,
        n_jobs=1,
        verbosity=0,
        n_estimators=n_estimators,
        max_depth=3,
        learning_rate=0.03,
        subsample=0.85,
        colsample_bytree=0.75,
        min_child_weight=5.0,
        reg_lambda=10.0,
        scale_pos_weight=negative / positive,
    )


def append_sensitivity_result(rows, model_name, base_model, family, feature_variant, y_true, score, fit_seconds, predict_seconds, notes):
    result = metric_row(
        model_name=model_name,
        base_model=base_model,
        family=family,
        calibration="none",
        evaluation_window="holdout",
        y_true=y_true,
        y_score=score,
        score_type="probability",
        fit_seconds=fit_seconds,
        predict_seconds=predict_seconds,
        timing_notes=notes,
    )
    result["Feature Variant"] = feature_variant
    rows.append(result)


def evaluate_model_family_feature_sensitivity():
    if not MODEL_FAMILY_FEATURE_SENSITIVITY_ENABLED or not FEATURE_VARIANT_SENSITIVITY_ENABLED:
        return pd.DataFrame()
    rows = []
    requested_variants = [variant for variant in MODEL_FAMILY_FEATURE_SENSITIVITY_VARIANTS if variant in feature_variant_bundles]
    if not requested_variants:
        return pd.DataFrame()
    for variant in tqdm(requested_variants, desc="Model-family feature sensitivity", unit="variant"):
        bundle = feature_variant_bundles[variant]
        X_train = bundle["X_preholdout"]
        X_test = bundle["X_holdout"]
        if RUN_FEATURE_SENSITIVITY_LOGISTIC:
            try:
                model_name = f"FeatureSensitivity[{variant}][LogisticRegression]"
                model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=5000, class_weight="balanced", random_state=SEED))
                start = time.perf_counter()
                model.fit(X_train, y_preholdout)
                fit_seconds = time.perf_counter() - start
                predict_start = time.perf_counter()
                score = positive_class_proba(model, X_test)
                predict_seconds = time.perf_counter() - predict_start
                append_sensitivity_result(rows, model_name, "Logistic Regression", "feature_policy_sensitivity", variant, y_holdout, score, fit_seconds, predict_seconds, "fixed CPU Logistic Regression rerun for selected feature-policy sensitivity")
                del model, score
            except Exception as exc:
                model_errors.append({"Model": f"FeatureSensitivity[{variant}][LogisticRegression]", "Error": short_error(exc)})
        if RUN_FEATURE_SENSITIVITY_XGBOOST_LIGHT:
            try:
                model_name = f"FeatureSensitivity[{variant}][XGBoost light]"
                model = make_light_xgboost_estimator(y_preholdout, FEATURE_SENSITIVITY_XGBOOST_N_ESTIMATORS)
                start = time.perf_counter()
                model.fit(X_train, y_preholdout)
                fit_seconds = time.perf_counter() - start
                predict_start = time.perf_counter()
                score = predict_proba_in_chunks(model, X_test, desc=f"{model_name} holdout prediction")
                predict_seconds = time.perf_counter() - predict_start
                append_sensitivity_result(rows, model_name, "XGBoost", "feature_policy_sensitivity", variant, y_holdout, score, fit_seconds, predict_seconds, "fixed GPU XGBoost rerun; no target-specific hyperparameter search")
                add_model_predictions(model_name, "XGBoost", "feature_policy_sensitivity", variant, "probability", [("holdout", holdout_df, y_holdout, score, predict_seconds)], fit_seconds=fit_seconds, timing_notes="fixed GPU XGBoost feature-policy sensitivity rerun; no target-specific hyperparameter search")
                del model, score
                cleanup_runtime_memory(f"after_{model_name}_prediction")
            except Exception as exc:
                model_errors.append({"Model": f"FeatureSensitivity[{variant}][XGBoost light]", "Error": short_error(exc)})
        if RUN_FEATURE_SENSITIVITY_TABPFN:
            try:
                model_name = f"FeatureSensitivity[{variant}][TabPFN direct]"
                model = make_tabpfn_classifier()
                start = time.perf_counter()
                model.fit(to_numpy_float32(X_train), y_to_numpy(y_preholdout))
                fit_seconds = time.perf_counter() - start
                predict_start = time.perf_counter()
                score = positive_class_proba(model, to_numpy_float32(X_test))
                predict_seconds = time.perf_counter() - predict_start
                append_sensitivity_result(rows, model_name, "TabPFN", "feature_policy_sensitivity", variant, y_holdout, score, fit_seconds, predict_seconds, "direct TabPFN rerun for selected feature-policy sensitivity")
                add_model_predictions(model_name, "TabPFN", "feature_policy_sensitivity", variant, "probability", [("holdout", holdout_df, y_holdout, score, predict_seconds)], fit_seconds=fit_seconds, timing_notes="direct TabPFN feature-policy sensitivity rerun")
                del model, score
                cleanup_runtime_memory(f"after_{model_name}_prediction")
            except Exception as exc:
                model_errors.append({"Model": f"FeatureSensitivity[{variant}][TabPFN direct]", "Error": short_error(exc)})
    summary = pd.DataFrame(rows)
    if len(summary) > 0:
        display(summary.sort_values(["Feature Variant", "Average Precision"], ascending=[True, False]).round(4))
        save_artifact_table(summary, "model_family_feature_sensitivity.csv", description="Selected feature-policy reruns across enabled model families; separate from the main tuned allocation comparison.")
        save_artifact_table(summary, "feature_variant_sensitivity_summary.csv", description="Feature-policy sensitivity using enabled model families; default uses fixed GPU XGBoost and direct TabPFN, not CPU Logistic Regression.")
    return summary


model_family_feature_sensitivity = evaluate_model_family_feature_sensitivity()


def xgboost_param_distributions(y):
    positive = max(int(y.sum()), 1)
    negative = max(int(len(y) - y.sum()), 1)
    scale_pos_weight = negative / positive
    return {
        "n_estimators": randint(400, 3000),
        "max_depth": randint(2, 10),
        "learning_rate": loguniform(0.003, 0.20),
        "subsample": uniform(0.55, 0.45),
        "colsample_bytree": uniform(0.45, 0.55),
        "colsample_bylevel": uniform(0.50, 0.50),
        "min_child_weight": loguniform(0.05, 80.0),
        "gamma": loguniform(1e-5, 20.0),
        "reg_alpha": loguniform(1e-5, 100.0),
        "reg_lambda": loguniform(0.05, 150.0),
        "max_bin": randint(128, 1024),
        "scale_pos_weight": [1.0, math.sqrt(scale_pos_weight), scale_pos_weight],
        "max_delta_step": [0, 1, 3, 5, 10],
    }


def make_xgboost_estimator():
    return GPUXGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method=XGBOOST_TREE_METHOD,
        device=XGBOOST_DEVICE,
        random_state=SEED,
        n_jobs=1,
        verbosity=0,
    )


def evaluate_xgboost_workflow():
    model_name = "XGBoost[GPU allocation scorer]"
    console.rule(model_name)
    search = ProgressRandomizedSearchCV(
        estimator=make_xgboost_estimator(),
        param_distributions=xgboost_param_distributions(y_selection),
        n_iter=XGBOOST_TUNING_ITERATIONS,
        cv_splits=cv_splits,
        random_state=SEED,
        label=model_name,
        early_stopping_rounds=XGBOOST_SEARCH_EARLY_STOPPING_ROUNDS if XGBOOST_SEARCH_EARLY_STOPPING_ENABLED else None,
    )
    print(f"{model_name}: {search.n_iter} randomized parameter draws x {len(search.cv)} chronological folds = {search.planned_fit_count():,} tuning fits; early_stopping_rounds={search.early_stopping_rounds}.")
    search_start = time.perf_counter()
    search.fit(X_selection, y_selection)
    search_seconds = time.perf_counter() - search_start
    best_params = artifact_json(search.best_params_)
    xgboost_search_cv_results = search.cv_results_frame()
    if len(xgboost_search_cv_results) > 0:
        display(xgboost_search_cv_results.sort_values("mean_test_score", ascending=False).head(20).round(4))
        save_artifact_table(xgboost_search_cv_results, "xgboost_search_cv_results.csv", description="Main XGBoost randomized-search trial diagnostics, including fold scores and early-stopping best iterations.")
    print(f"Best validation Average Precision: {search.best_score_:.4f}")
    print(f"Best params: {best_params}")

    final_model = best_estimator_from_search(search)
    final_start = time.perf_counter()
    final_model.fit(X_preholdout, y_preholdout)
    final_fit_seconds = time.perf_counter() - final_start
    predict_start = time.perf_counter()
    holdout_score = predict_proba_in_chunks(final_model, X_holdout, desc=f"{model_name} holdout prediction")
    holdout_predict_seconds = time.perf_counter() - predict_start
    add_model_predictions(model_name, "XGBoost", "gpu_boosted_tree", "none", "probability", [("holdout", holdout_df, y_holdout, holdout_score, holdout_predict_seconds)], fit_seconds=search_seconds + final_fit_seconds, cv_average_precision=search.best_score_, best_params=best_params, timing_notes="randomized search on monthly rolling-origin folds; final model fit on all pre-holdout rows")
    del final_model
    cleanup_runtime_memory("after_xgboost_final_prediction")

    if EVALUATE_CALIBRATION_VARIANTS:
        base_model = best_estimator_from_search(search)
        base_start = time.perf_counter()
        base_model.fit(X_selection, y_selection)
        base_fit_seconds = time.perf_counter() - base_start
        predict_start = time.perf_counter()
        base_calibration_score = predict_proba_in_chunks(base_model, X_calibration, desc=f"{model_name} calibration-base calibration prediction")
        base_holdout_score = predict_proba_in_chunks(base_model, X_holdout, desc=f"{model_name} calibration-base holdout prediction")
        base_predict_seconds = time.perf_counter() - predict_start
        add_model_predictions(f"{model_name} Calibration Base", "XGBoost", "gpu_boosted_tree_calibration_base", "none_calibration_base", "probability", [("calibration", calibration_df, y_calibration, base_calibration_score, base_predict_seconds / 2.0), ("holdout", holdout_df, y_holdout, base_holdout_score, base_predict_seconds / 2.0)], fit_seconds=search_seconds + base_fit_seconds, cv_average_precision=search.best_score_, best_params=best_params, timing_notes="base model excludes calibration-window labels")
        if y_calibration.nunique() > 1:
            calibration_start = time.perf_counter()
            calibrated_model = make_prefit_calibrator(base_model, X_calibration, y_calibration, method="sigmoid")
            calibration_seconds = time.perf_counter() - calibration_start
            predict_start = time.perf_counter()
            calibrated_calibration_score = predict_proba_in_chunks(calibrated_model, X_calibration, desc=f"{model_name} calibrated calibration prediction")
            calibrated_holdout_score = predict_proba_in_chunks(calibrated_model, X_holdout, desc=f"{model_name} calibrated holdout prediction")
            calibrated_predict_seconds = time.perf_counter() - predict_start
            add_model_predictions(f"{model_name} Calibrated", "XGBoost", "gpu_boosted_tree_calibrated", "sigmoid", "probability", [("calibration", calibration_df, y_calibration, calibrated_calibration_score, calibrated_predict_seconds / 2.0), ("holdout", holdout_df, y_holdout, calibrated_holdout_score, calibrated_predict_seconds / 2.0)], fit_seconds=search_seconds + base_fit_seconds, calibration_seconds=calibration_seconds, cv_average_precision=search.best_score_, best_params=best_params, timing_notes="sigmoid calibration fitted on calibration window")
            del calibrated_model, calibrated_calibration_score, calibrated_holdout_score
        del base_model, base_calibration_score, base_holdout_score
    del search
    cleanup_runtime_memory("after_xgboost_workflow")


def tabicl_batch_size():
    if CUDA_DEVICE_COUNT == 0:
        return 256
    return 1024 if len(X_preholdout) < 5000 else 512


def evaluate_direct_tfm(model_name, model_factory, package_name):
    console.rule(model_name)
    try:
        start = time.perf_counter()
        final_model = model_factory()
        print(f"{model_name}: fitting final direct model on {len(X_preholdout):,} pre-holdout rows.")
        final_model.fit(to_numpy_float32(X_preholdout), y_to_numpy(y_preholdout))
        final_fit_seconds = time.perf_counter() - start
        predict_start = time.perf_counter()
        final_holdout_score = positive_class_proba(final_model, to_numpy_float32(X_holdout))
        final_predict_seconds = time.perf_counter() - predict_start
        add_model_predictions(model_name, package_name, "direct_tfm", "none", "probability", [("holdout", holdout_df, y_holdout, final_holdout_score, final_predict_seconds)], fit_seconds=final_fit_seconds, timing_notes="direct TFM classifier fitted on all pre-holdout rows; no task-specific weight training claim is made")
        del final_model, final_holdout_score
        cleanup_runtime_memory(f"after_{model_name}_final_prediction")

        if EVALUATE_CALIBRATION_VARIANTS:
            base_model = model_factory()
            base_start = time.perf_counter()
            print(f"{model_name}: fitting calibration-base direct model on {len(X_selection):,} selection rows.")
            base_model.fit(to_numpy_float32(X_selection), y_to_numpy(y_selection))
            base_fit_seconds = time.perf_counter() - base_start
            predict_start = time.perf_counter()
            base_calibration_score = positive_class_proba(base_model, to_numpy_float32(X_calibration))
            base_holdout_score = positive_class_proba(base_model, to_numpy_float32(X_holdout))
            base_predict_seconds = time.perf_counter() - predict_start
            add_model_predictions(f"{model_name} Calibration Base", package_name, "direct_tfm_calibration_base", "none_calibration_base", "probability", [("calibration", calibration_df, y_calibration, base_calibration_score, base_predict_seconds / 2.0), ("holdout", holdout_df, y_holdout, base_holdout_score, base_predict_seconds / 2.0)], fit_seconds=base_fit_seconds, timing_notes="direct TFM context excludes calibration-window labels")
            del base_model, base_calibration_score, base_holdout_score
            cleanup_runtime_memory(f"after_{model_name}_calibration_base_prediction")
        cleanup_runtime_memory(f"after_{model_name}_workflow")
    except Exception as exc:
        error = short_error(exc)
        model_errors.append({"Model": model_name, "Error": error})
        print(f"{model_name} failed: {error}")


if RUN_GPU_XGBOOST:
    try:
        evaluate_xgboost_workflow()
    except Exception as exc:
        error = short_error(exc)
        model_errors.append({"Model": "XGBoost[GPU allocation scorer]", "Error": error})
        print(f"XGBoost workflow failed: {error}")

if RUN_LOGISTIC_REGRESSION_CPU_BENCHMARK:
    try:
        logistic_name = "LogisticRegression[CPU optional benchmark]"
        logistic = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=5000, class_weight="balanced", random_state=SEED))
        start = time.perf_counter()
        logistic.fit(X_preholdout, y_preholdout)
        fit_seconds = time.perf_counter() - start
        predict_start = time.perf_counter()
        score = positive_class_proba(logistic, X_holdout)
        predict_seconds = time.perf_counter() - predict_start
        add_model_predictions(logistic_name, "Logistic Regression", "optional_cpu", "none", "probability", [("holdout", holdout_df, y_holdout, score, predict_seconds)], fit_seconds=fit_seconds, timing_notes="optional CPU benchmark; disabled by default")
    except Exception as exc:
        model_errors.append({"Model": "LogisticRegression[CPU optional benchmark]", "Error": short_error(exc)})

if RUN_DIRECT_TABPFN:
    def make_tabpfn_direct():
        return make_tabpfn_classifier()
    evaluate_direct_tfm("TabPFN[Direct allocation scorer]", make_tabpfn_direct, "TabPFN")

if RUN_DIRECT_TABICL:
    def make_tabicl_direct():
        from tabicl import TabICLClassifier
        return TabICLClassifier(n_estimators=N_TFM_ESTIMATORS, device=TABICL_DEVICE, batch_size=tabicl_batch_size(), random_state=SEED, checkpoint_version=TABICL_CHECKPOINT_VERSION)
    evaluate_direct_tfm("TabICL[Direct allocation scorer]", make_tabicl_direct, "TabICL")

allocation_summary = pd.DataFrame(model_rows)
if len(allocation_summary) == 0:
    raise RuntimeError("No model rows were produced.")
allocation_summary = allocation_summary.sort_values(["Evaluation Window", "Average Precision"], ascending=[True, False], na_position="last").reset_index(drop=True)
performance_columns = ["Model", "Base Model", "Calibration", "Evaluation Window", "Rows", "Positive Rate", "Score Type", "Average Precision", "ROC AUC", "Brier Score", "Log Loss", "ECE Quantile 10", "CV/Validation Average Precision", "Workflow Seconds"]
display(allocation_summary[performance_columns].round(4))
save_artifact_table(allocation_summary, "allocation_summary.csv")

if model_errors:
    model_error_summary = pd.DataFrame(model_errors)
    display(model_error_summary)
    save_artifact_table(model_error_summary, "model_error_summary.csv")

best_params = allocation_summary.loc[allocation_summary["Best Params"].astype(str).str.len() > 0, ["Model", "CV/Validation Average Precision", "Best Params"]].drop_duplicates(subset=["Model"])
if len(best_params) > 0:
    display(best_params)
    save_artifact_table(best_params, "best_params.csv")

timing_notes = allocation_summary.loc[allocation_summary["Timing Notes"].astype(str).str.len() > 0, ["Model", "Calibration", "Evaluation Window", "Timing Notes"]].drop_duplicates()
if len(timing_notes) > 0:
    save_artifact_table(timing_notes, "timing_notes.csv")

all_prediction_frame = pd.concat(prediction_frames, ignore_index=True) if prediction_frames else pd.DataFrame()
prediction_frames.clear()
cleanup_runtime_memory("after_prediction_frame_assembly")
if len(all_prediction_frame) > 0 and SAVE_FULL_PREDICTION_SCORES:
    save_artifact_table(all_prediction_frame, "model_prediction_scores.csv", description="Per-row model scores for calibration and holdout windows.")
elif len(all_prediction_frame) > 0:
    compact_prediction_scores = all_prediction_frame.loc[
        all_prediction_frame["Evaluation Window"].eq("holdout"),
        ["month", "asset", "target_top_k_next_1m", "forward_1m_return", "Model", "score"],
    ].copy()
    save_artifact_table(compact_prediction_scores, "model_prediction_scores_holdout_compact.csv", description="Compact holdout-only model scores for memory-efficient runs.")

if len(all_prediction_frame) > 0:
    execution_audit_columns = [
        "Evaluation Window", "Model", "date", "month", "asset", "score",
        "target_top_k_next_1m", "forward_1m_return",
        "forward_1m_close_to_close_return", "forward_1m_next_open_return",
        "forward_1m_excess_return", "forward_1m_rank",
    ]
    execution_audit_columns = [column for column in execution_audit_columns if column in all_prediction_frame.columns]
    save_artifact_table(all_prediction_frame[execution_audit_columns].copy(), "model_prediction_execution_audit.csv", description="Per-model score audit retaining selected and alternative execution-convention returns.")


Model-family feature sensitivity:   0%|          | 0/5 [00:00<?, ?variant/s]

,Model,Base Model,Family,Calibration,Evaluation Window,Rows,Positive Rows,Positive Rate,Score Type,Average Precision,...,Log Loss,ECE Quantile 10,Fit Seconds,Predict Seconds,Calibration Seconds,Workflow Seconds,CV/Validation Average Precision,Best Params,Timing Notes,Feature Variant
1,FeatureSensitivity[full][TabPFN direct],TabPFN,feature_policy_sensitivity,none,holdout,750,225,0.3,probability,0.3748,...,0.6031,0.0456,1.8861,4.2061,0.0,6.0922,NaN,,direct TabPFN rerun for selected feature-polic...,full
0,FeatureSensitivity[full][XGBoost light],XGBoost,feature_policy_sensitivity,none,holdout,750,225,0.3,probability,0.3285,...,0.6474,0.1209,3.2938,0.1688,0.0,3.4626,NaN,,fixed GPU XGBoost rerun; no target-specific hy...,full
5,FeatureSensitivity[identity_ablated][TabPFN di...,TabPFN,feature_policy_sensitivity,none,holdout,750,225,0.3,probability,0.3650,...,0.6029,0.0488,1.4779,4.0884,0.0,5.5663,NaN,,direct TabPFN rerun for selected feature-polic...,identity_ablated
4,FeatureSensitivity[identity_ablated][XGBoost l...,XGBoost,feature_policy_sensitivity,none,holdout,750,225,0.3,probability,0.3258,...,0.6491,0.1195,0.7616,0.0246,0.0,0.7862,NaN,,fixed GPU XGBoost rerun; no target-specific hy...,identity_ablated
7,FeatureSensitivity[metadata_only][TabPFN direct],TabPFN,feature_policy_sensitivity,none,holdout,750,225,0.3,probability,0.3570,...,0.6027,0.0443,0.3431,0.2209,0.0,0.5640,NaN,,direct TabPFN rerun for selected feature-polic...,metadata_only
6,FeatureSensitivity[metadata_only][XGBoost light],XGBoost,feature_policy_sensitivity,none,holdout,750,225,0.3,probability,0.3551,...,0.6797,0.1923,0.3929,0.0219,0.0,0.4149,NaN,,fixed GPU XGBoost rerun; no target-specific hy...,metadata_only
9,FeatureSensitivity[strict_time_series][TabPFN ...,TabPFN,feature_policy_sensitivity,none,holdout,750,225,0.3,probability,0.3650,...,0.6029,0.0488,1.7568,4.0345,0.0,5.7913,NaN,,direct TabPFN rerun for selected feature-polic...,strict_time_series
8,FeatureSensitivity[strict_time_series][XGBoost...,XGBoost,feature_policy_sensitivity,none,holdout,750,225,0.3,probability,0.3258,...,0.6491,0.1195,0.7519,0.0250,0.0,0.7769,NaN,,fixed GPU XGBoost rerun; no target-specific hy...,strict_time_series
3,FeatureSensitivity[ticker_ablated][TabPFN direct],TabPFN,feature_policy_sensitivity,none,holdout,750,225,0.3,probability,0.3743,...,0.6028,0.0690,1.7218,4.1135,0.0,5.8353,NaN,,direct TabPFN rerun for selected feature-polic...,ticker_ablated
2,FeatureSensitivity[ticker_ablated][XGBoost light],XGBoost,feature_policy_sensitivity,none,holdout,750,225,0.3,probability,0.3236,...,0.6491,0.1286,0.7697,0.0252,0.0,0.7949,NaN,,fixed GPU XGBoost rerun; no target-specific hy...,ticker_ablated


Saved tabpfn_tabicl_tactical_asset_allocation_20260512_v2_outputs/model_family_feature_sensitivity.csv
Saved tabpfn_tabicl_tactical_asset_allocation_20260512_v2_outputs/feature_variant_sensitivity_summary.csv


───────────────────────────────────────── XGBoost[GPU allocation scorer] ──────────────────────────────────────────

XGBoost[GPU allocation scorer]: 10 randomized parameter draws x 2 chronological folds = 20 tuning fits; early_stopping_rounds=75.


...

TabPFN[Direct allocation scorer]: fitting final direct model on 1,680 pre-holdout rows.
TabPFN[Direct allocation scorer]: fitting calibration-base direct model on 1,440 selection rows.


,Model,Base Model,Calibration,Evaluation Window,Rows,Positive Rate,Score Type,Average Precision,ROC AUC,Brier Score,Log Loss,ECE Quantile 10,CV/Validation Average Precision,Workflow Seconds
0,XGBoost[GPU allocation scorer] Calibration Base,XGBoost,none_calibration_base,calibration,240,0.3,probability,0.3665,0.5604,0.2435,0.7976,0.1832,0.3542,8.5984
1,XGBoost[GPU allocation scorer] Calibrated,XGBoost,sigmoid,calibration,240,0.3,probability,0.3665,0.5604,0.2083,0.6070,0.0585,0.3542,8.5847
2,Rule[Low volatility top-k],Rule,none,calibration,240,0.3,raw_score,0.3253,0.4822,NaN,NaN,NaN,NaN,0.0000
3,TabPFN[Direct allocation scorer] Calibration Base,TabPFN,none_calibration_base,calibration,240,0.3,probability,0.3119,0.5291,0.2109,0.6146,0.0730,NaN,4.6526
4,Rule[12M momentum top-k],Rule,none,calibration,240,0.3,raw_score,0.2657,0.4519,NaN,NaN,NaN,NaN,0.0000
5,Rule[6M momentum top-k],Rule,none,calibration,240,0.3,raw_score,0.2406,0.3915,NaN,NaN,NaN,NaN,0.0000
6,Rule[Risk-adjusted momentum top-k],Rule,none,calibration,240,0.3,raw_score,0.2405,0.3854,NaN,NaN,NaN,NaN,0.0000
7,FeatureSensitivity[full][TabPFN direct],TabPFN,full,holdout,750,0.3,probability,0.3748,0.6022,0.2065,0.6031,0.0456,NaN,6.0922
8,FeatureSensitivity[ticker_ablated][TabPFN direct],TabPFN,ticker_ablated,holdout,750,0.3,probability,0.3743,0.6003,0.2064,0.6028,0.0690,NaN,5.8353
9,FeatureSensitivity[identity_ablated][TabPFN di...,TabPFN,identity_ablated,holdout,750,0.3,probability,0.3650,0.5829,0.2063,0.6029,0.0488,NaN,5.5663


Saved tabpfn_tabicl_tactical_asset_allocation_20260512_v2_outputs/allocation_summary.csv


,Model,CV/Validation Average Precision,Best Params
0,XGBoost[GPU allocation scorer] Calibration Base,0.354182,"{""colsample_bylevel"": 0.8022086896389087, ""col..."
1,XGBoost[GPU allocation scorer] Calibrated,0.354182,"{""colsample_bylevel"": 0.8022086896389087, ""col..."
22,XGBoost[GPU allocation scorer],0.354182,"{""colsample_bylevel"": 0.8022086896389087, ""col..."


Saved tabpfn_tabicl_tactical_asset_allocation_20260512_v2_outputs/best_params.csv
Saved tabpfn_tabicl_tactical_asset_allocation_20260512_v2_outputs/timing_notes.csv
Saved tabpfn_tabicl_tactical_asset_allocation_20260512_v2_outputs/model_prediction_scores.csv
Saved tabpfn_tabicl_tactical_asset_allocation_20260512_v2_outputs/model_prediction_execution_audit.csv


## 6. Ranking, Portfolio, Calibration, Runtime, and Uncertainty Diagnostics

A tactical allocation score should be evaluated in several views: row-level classification quality, within-month rank quality, portfolio behavior after transaction costs, runtime, and probability diagnostics. Constant-score baselines are useful for classification calibration checks but do not define a cross-sectional top-k allocation without an arbitrary tie-break rule, so they are excluded from score-driven portfolio diagnostics. Deterministic rule portfolios that are already represented as named benchmark portfolios are not duplicated in the portfolio table. The portfolio backtest here is a diagnostic of score utility, not a trading recommendation.

In [7]:
MODEL_DISPLAY_NAMES = {
    "Rule[12M momentum top-k]": "12M momentum",
    "Rule[6M momentum top-k]": "6M momentum",
    "Rule[Low volatility top-k]": "Low volatility",
    "Rule[Risk-adjusted momentum top-k]": "Risk-adjusted momentum",
    "XGBoost[GPU allocation scorer]": "XGBoost final",
    "XGBoost[GPU allocation scorer] Calibration Base": "XGBoost calibration base",
    "XGBoost[GPU allocation scorer] Calibrated": "XGBoost calibrated",
    "TabPFN[Direct allocation scorer]": "TabPFN direct",
    "TabPFN[Direct allocation scorer] Calibration Base": "TabPFN calibration base",
    "TabICL[Direct allocation scorer]": "TabICL direct",
    "TabICL[Direct allocation scorer] Calibration Base": "TabICL calibration base",
    "Dummy[Preholdout prior probability]": "Preholdout prior",
    "Equal weight universe": "Equal weight",
    "SPY only": "SPY only",
    "60/40 SPY/TLT": "60/40 SPY/TLT",
    "12M momentum top-k portfolio": "12M momentum",
    "Low volatility top-k portfolio": "Low volatility",
}


DUPLICATE_DETERMINISTIC_PORTFOLIO_MODELS = {
    "Rule[12M momentum top-k]": "12M momentum top-k portfolio",
    "Rule[Low volatility top-k]": "Low volatility top-k portfolio",
}


def model_display_name(model_name):
    return MODEL_DISPLAY_NAMES.get(model_name, model_name)


def is_feature_sensitivity_model(model_name):
    return str(model_name).startswith("FeatureSensitivity[")


def exclude_feature_sensitivity(frame, model_column="Model"):
    if len(frame) == 0 or model_column not in frame.columns:
        return frame.copy()
    return frame.loc[~frame[model_column].astype(str).map(is_feature_sensitivity_model)].copy()


def month_index_to_timestamp(index):
    values = pd.Index(index).astype(str)
    try:
        return pd.PeriodIndex(values, freq="M").to_timestamp()
    except Exception:
        return pd.to_datetime(values, errors="coerce")


def selected_publication_rows(summary, evaluation_window="holdout"):
    window = summary.loc[(summary["Evaluation Window"] == evaluation_window) & summary["Average Precision"].notna()].copy()
    window = exclude_feature_sensitivity(window)
    selected = window.loc[window["Model"].isin(PUBLICATION_MODEL_ORDER)].copy()
    if len(selected) == 0:
        selected = window.copy()
    order_map = {name: idx for idx, name in enumerate(PUBLICATION_MODEL_ORDER)}
    selected["publication_order"] = selected["Model"].map(order_map).fillna(999).astype(int)
    return selected.sort_values(["publication_order", "Average Precision"], ascending=[True, False]).drop(columns=["publication_order"])


headline_allocation_summary = exclude_feature_sensitivity(allocation_summary)
feature_sensitivity_allocation_summary = allocation_summary.loc[allocation_summary["Model"].astype(str).map(is_feature_sensitivity_model)].copy()
if len(headline_allocation_summary) > 0:
    display(headline_allocation_summary[performance_columns].round(4))
    save_artifact_table(headline_allocation_summary, "headline_allocation_summary.csv", description="Allocation summary for headline models only; feature-policy sensitivity reruns are excluded.")
if len(feature_sensitivity_allocation_summary) > 0:
    display(feature_sensitivity_allocation_summary.round(4))
    save_artifact_table(feature_sensitivity_allocation_summary, "feature_sensitivity_allocation_summary.csv", description="Allocation metrics for feature-policy sensitivity reruns, separated from headline model summaries.")


def monthly_rank_quality(score_frame, top_k=TARGET_TOP_K, month_column="month", score_column="score"):
    rows = []
    required_columns = ["Model", "Evaluation Window", month_column, score_column, "forward_1m_return", "target_top_k_next_1m"]
    optional_columns = ["asset"] if "asset" in score_frame.columns else []
    working = score_frame[required_columns + optional_columns].copy()
    working[score_column] = pd.to_numeric(working[score_column], errors="coerce")
    working["forward_1m_return"] = pd.to_numeric(working["forward_1m_return"], errors="coerce")
    working = working.dropna(subset=[score_column, "forward_1m_return"])
    group_columns = ["Model", "Evaluation Window", month_column]
    for (model_name, evaluation_window, month), numeric_group in working.groupby(group_columns, sort=False):
        if len(numeric_group) < max(2, top_k):
            continue

        score = numeric_group[score_column].to_numpy(dtype=np.float64, copy=False)
        realized = numeric_group["forward_1m_return"].to_numpy(dtype=np.float64, copy=False)
        unique_score_count = np.unique(score[np.isfinite(score)]).size
        universe_mean = float(np.nanmean(realized))

        if unique_score_count <= 1:
            ic = np.nan
            top_k_hit_rate = np.nan
            selected_mean_return = np.nan
            active_return = np.nan
        else:
            if "asset" in numeric_group.columns:
                sort_values = numeric_group.sort_values([score_column, "asset"], ascending=[False, True]).head(top_k)
                selected_returns = sort_values["forward_1m_return"].to_numpy(dtype=np.float64, copy=False)
                selected_targets = sort_values["target_top_k_next_1m"].to_numpy(dtype=np.float64, copy=False)
            else:
                top_indices = np.argsort(-score, kind="mergesort")[:top_k]
                selected_returns = realized[top_indices]
                selected_targets = numeric_group["target_top_k_next_1m"].to_numpy(dtype=np.float64, copy=False)[top_indices]
            ic = spearmanr(score, realized, nan_policy="omit").correlation if np.unique(realized[np.isfinite(realized)]).size > 1 else np.nan
            top_k_hit_rate = float(np.nanmean(selected_targets))
            selected_mean_return = float(np.nanmean(selected_returns))
            active_return = selected_mean_return - universe_mean

        rows.append(
            {
                "Model": model_name,
                "Evaluation Window": evaluation_window,
                "month": month,
                "rows": len(numeric_group),
                "top_k": top_k,
                "spearman_ic": float(ic) if ic is not None else np.nan,
                "top_k_hit_rate": top_k_hit_rate,
                "selected_mean_forward_return": selected_mean_return,
                "universe_mean_forward_return": universe_mean,
                "active_forward_return": active_return,
            }
        )
    return pd.DataFrame(rows)


monthly_rank_summary = monthly_rank_quality(all_prediction_frame, top_k=TARGET_TOP_K) if len(all_prediction_frame) else pd.DataFrame()
if len(monthly_rank_summary) > 0:
    monthly_rank_aggregate = (
        monthly_rank_summary.groupby(["Model", "Evaluation Window"])
        .agg(
            months=("month", "nunique"),
            mean_spearman_ic=("spearman_ic", "mean"),
            median_spearman_ic=("spearman_ic", "median"),
            mean_top_k_hit_rate=("top_k_hit_rate", "mean"),
            mean_selected_forward_return=("selected_mean_forward_return", "mean"),
            mean_universe_forward_return=("universe_mean_forward_return", "mean"),
            mean_active_forward_return=("active_forward_return", "mean"),
        )
        .reset_index()
        .sort_values(["Evaluation Window", "mean_active_forward_return"], ascending=[True, False])
    )
    display(monthly_rank_aggregate.round(4))
    save_artifact_table(monthly_rank_summary, "monthly_rank_summary.csv")
    save_artifact_table(monthly_rank_aggregate, "monthly_rank_aggregate.csv")


def evaluate_alternative_objective_score_quality(score_frame):
    if len(score_frame) == 0:
        return pd.DataFrame()
    objective_columns = [("benchmark_outperform_1m", "target_outperform_benchmark_next_1m")]
    for horizon in MULTI_HORIZON_MONTHS:
        objective_columns.append((f"top_k_next_{horizon}m", f"target_top_k_next_{horizon}m"))
    rows = []
    holdout_scores = score_frame.loc[score_frame["Evaluation Window"] == "holdout"].copy()
    for model_name, model_group in holdout_scores.groupby("Model", sort=False):
        score = pd.to_numeric(model_group["score"], errors="coerce")
        for objective_name, target_column in objective_columns:
            if target_column not in model_group.columns:
                continue
            target = pd.to_numeric(model_group[target_column], errors="coerce")
            valid = target.notna() & score.notna()
            if valid.sum() < 10:
                continue
            y_values = target.loc[valid].astype(int).to_numpy()
            score_values = score.loc[valid].to_numpy(dtype=float)
            rows.append(
                {
                    "Model": model_name,
                    "Objective": objective_name,
                    "Target Column": target_column,
                    "Rows": int(valid.sum()),
                    "Positive Rate": float(y_values.mean()),
                    "Average Precision": safe_metric(average_precision_score, y_values, score_values),
                    "ROC AUC": safe_metric(roc_auc_score, y_values, score_values),
                }
            )
    return pd.DataFrame(rows)


alternative_objective_score_quality = evaluate_alternative_objective_score_quality(all_prediction_frame)
if len(alternative_objective_score_quality) > 0:
    display(alternative_objective_score_quality.round(4))
    save_artifact_table(alternative_objective_score_quality, "alternative_objective_score_quality.csv", description="Holdout score quality when existing one-month scores are evaluated against benchmark-relative and multi-horizon targets.")




def objective_target_specs():
    specs = [
        {
            "Objective": "benchmark_outperform_1m",
            "Target Column": "target_outperform_benchmark_next_1m",
            "Return Column": "forward_1m_vs_benchmark_return",
            "Portfolio Return Column": "forward_1m_vs_benchmark_return",
            "Horizon Months": 1,
            "Portfolio Diagnostic Caveat": "benchmark-relative active-return diagnostic; not a benchmark-relative optimizer",
        }
    ]
    for horizon in MULTI_HORIZON_MONTHS:
        specs.append(
            {
                "Objective": f"top_k_next_{horizon}m",
                "Target Column": f"target_top_k_next_{horizon}m",
                "Return Column": f"forward_{horizon}m_return",
                "Portfolio Return Column": f"forward_{horizon}m_return",
                "Horizon Months": horizon,
                "Portfolio Diagnostic Caveat": f"{horizon}-month overlapping forward-return diagnostic; not a monthly executable backtest",
            }
        )
    return [spec for spec in specs if spec["Objective"] in OBJECTIVE_SPECIFIC_TARGETS]


OBJECTIVE_SPEC_BY_NAME = {spec["Objective"]: spec for spec in objective_target_specs()}


def objective_cv_average_precision(spec):
    target_column = spec["Target Column"]
    if not RUN_OBJECTIVE_SPECIFIC_XGBOOST_LIGHT or target_column not in selection_df.columns:
        return np.nan
    fold_scores = []
    for fold_number, (train_idx, valid_idx) in enumerate(cv_splits, start=1):
        train_idx = np.asarray(train_idx, dtype=int)
        valid_idx = np.asarray(valid_idx, dtype=int)
        train_frame = selection_df.iloc[train_idx].copy()
        valid_frame = selection_df.iloc[valid_idx].copy()
        validation_start = pd.to_datetime(valid_frame["date"]).min()
        y_train_fold = train_frame[target_column]
        y_valid_fold = valid_frame[target_column]
        train_keep = y_train_fold.notna().to_numpy() & objective_label_resolved_mask(train_frame, spec, validation_start)
        valid_keep = y_valid_fold.notna().to_numpy()
        if train_keep.sum() == 0 or valid_keep.sum() == 0:
            continue
        X_train_fold = X_selection.iloc[train_idx[train_keep]]
        X_valid_fold = X_selection.iloc[valid_idx[valid_keep]]
        y_train_fold = y_train_fold.iloc[train_keep].astype(int).reset_index(drop=True)
        y_valid_fold = y_valid_fold.iloc[valid_keep].astype(int).reset_index(drop=True)
        if y_train_fold.nunique() < 2 or y_valid_fold.nunique() < 2:
            continue
        if int(y_train_fold.sum()) < MIN_CV_TRAIN_POSITIVES or int(y_valid_fold.sum()) < MIN_CV_VALIDATION_POSITIVES:
            continue
        try:
            model = make_light_xgboost_estimator(y_train_fold, OBJECTIVE_SPECIFIC_XGBOOST_N_ESTIMATORS)
            model.fit(X_train_fold, y_train_fold)
            score = predict_proba_in_chunks(model, X_valid_fold, desc=f"Objective CV {target_column} fold {fold_number}")
            fold_score = safe_metric(average_precision_score, y_valid_fold, score)
            if np.isfinite(fold_score):
                fold_scores.append(fold_score)
            del model, score
            cleanup_runtime_memory(f"after_objective_cv_{target_column}_fold_{fold_number}")
        except Exception as exc:
            model_errors.append({"Model": f"ObjectiveCV[{target_column}][fold_{fold_number}]", "Error": short_error(exc)})
            cleanup_runtime_memory(f"after_objective_cv_{target_column}_fold_{fold_number}_error")
    return float(np.nanmean(fold_scores)) if fold_scores else np.nan


def objective_selection_search_bundle(spec):
    target_column = spec["Target Column"]
    if target_column not in selection_df.columns:
        return pd.DataFrame(), pd.Series(dtype=int), [], pd.DataFrame()
    eligible_mask = selection_df[target_column].notna().to_numpy()
    eligible_original_idx = np.flatnonzero(eligible_mask)
    if len(eligible_original_idx) == 0:
        return pd.DataFrame(), pd.Series(dtype=int), [], pd.DataFrame()
    original_to_position = {int(original_idx): position for position, original_idx in enumerate(eligible_original_idx)}
    objective_cv = []
    split_rows = []
    for fold_number, (train_idx, valid_idx) in enumerate(cv_splits, start=1):
        train_idx = np.asarray(train_idx, dtype=int)
        valid_idx = np.asarray(valid_idx, dtype=int)
        valid_keep = eligible_mask[valid_idx]
        valid_original = valid_idx[valid_keep]
        if len(valid_original) == 0:
            continue
        validation_start = pd.to_datetime(selection_df.iloc[valid_original]["date"]).min()
        train_frame = selection_df.iloc[train_idx].copy()
        train_keep = train_frame[target_column].notna().to_numpy() & objective_label_resolved_mask(train_frame, spec, validation_start)
        train_original = train_idx[train_keep]
        if len(train_original) == 0:
            continue
        train_positions = np.asarray([original_to_position[int(idx)] for idx in train_original if int(idx) in original_to_position], dtype=int)
        valid_positions = np.asarray([original_to_position[int(idx)] for idx in valid_original if int(idx) in original_to_position], dtype=int)
        if len(train_positions) == 0 or len(valid_positions) == 0:
            continue
        y_train_fold = selection_df.iloc[train_original][target_column].astype(int)
        y_valid_fold = selection_df.iloc[valid_original][target_column].astype(int)
        if y_train_fold.nunique() < 2 or y_valid_fold.nunique() < 2:
            continue
        if int(y_train_fold.sum()) < MIN_CV_TRAIN_POSITIVES or int(y_valid_fold.sum()) < MIN_CV_VALIDATION_POSITIVES:
            continue
        objective_cv.append((train_positions, valid_positions))
        split_rows.append(
            {
                "Objective": spec["Objective"],
                "Fold": fold_number,
                "Train Rows": len(train_positions),
                "Validation Rows": len(valid_positions),
                "Train Positive Rows": int(y_train_fold.sum()),
                "Validation Positive Rows": int(y_valid_fold.sum()),
                "Validation Start": validation_start,
                "Training Labels Resolved Before Validation": True,
            }
        )
    X_objective_selection = X_selection.iloc[eligible_original_idx].reset_index(drop=True)
    y_objective_selection = selection_df.iloc[eligible_original_idx][target_column].astype(int).reset_index(drop=True)
    return X_objective_selection, y_objective_selection, objective_cv, pd.DataFrame(split_rows)


def run_objective_xgboost_search(spec):
    X_objective_selection, y_objective_selection, objective_cv, split_summary = objective_selection_search_bundle(spec)
    if len(X_objective_selection) == 0 or y_objective_selection.nunique() < 2 or len(objective_cv) == 0:
        raise RuntimeError(f"insufficient objective-specific CV data for {spec['Objective']}")
    model_name = f"ObjectiveSpecific[{spec['Objective']}][XGBoost searched]"
    search = ProgressRandomizedSearchCV(
        estimator=make_xgboost_estimator(),
        param_distributions=xgboost_param_distributions(y_objective_selection),
        n_iter=OBJECTIVE_SPECIFIC_XGBOOST_SEARCH_ITERATIONS,
        cv_splits=objective_cv,
        random_state=SEED,
        label=model_name,
        early_stopping_rounds=XGBOOST_SEARCH_EARLY_STOPPING_ROUNDS if XGBOOST_SEARCH_EARLY_STOPPING_ENABLED else None,
    )
    print(f"{model_name}: {search.n_iter} randomized draws x {len(objective_cv)} objective chronological folds = {search.planned_fit_count():,} tuning fits; early_stopping_rounds={search.early_stopping_rounds}.")
    search_start = time.perf_counter()
    search.fit(X_objective_selection, y_objective_selection)
    search_seconds = time.perf_counter() - search_start
    search_results = search.cv_results_frame()
    if len(search_results) > 0:
        display(search_results.sort_values("mean_test_score", ascending=False).head(10).round(4))
        save_artifact_table(search_results, f"objective_specific_xgboost_search_cv_results_{spec['Objective']}.csv", description=f"Objective-specific XGBoost randomized-search trial diagnostics for {spec['Objective']}, including fold scores and early-stopping best iterations.")
    if len(split_summary) > 0:
        split_summary = split_summary.copy()
        split_summary["Best CV Average Precision"] = search.best_score_
        split_summary["Early Stopping Rounds"] = search.early_stopping_rounds if search.uses_early_stopping() else 0
        save_artifact_table(split_summary, f"objective_specific_xgboost_search_cv_splits_{spec['Objective']}.csv", description=f"Chronological objective-specific XGBoost search folds for {spec['Objective']}.")
    return search, search_seconds, artifact_json(search.best_params_)


def append_objective_result(
    rows,
    model_name,
    base_model,
    family,
    spec,
    y_true,
    score,
    fit_seconds,
    predict_seconds,
    notes,
    evaluation_window="holdout",
    calibration="none",
    calibration_seconds=0.0,
    cv_average_precision=np.nan,
    best_params="",
):
    result = metric_row(
        model_name=model_name,
        base_model=base_model,
        family=family,
        calibration=calibration,
        evaluation_window=evaluation_window,
        y_true=y_true,
        y_score=score,
        score_type="probability",
        fit_seconds=fit_seconds,
        predict_seconds=predict_seconds,
        calibration_seconds=calibration_seconds,
        cv_average_precision=cv_average_precision,
        best_params=best_params,
        timing_notes=notes,
    )
    result["Objective"] = spec["Objective"]
    result["Target Column"] = spec["Target Column"]
    result["Return Column"] = spec["Return Column"]
    result["Portfolio Return Column"] = spec.get("Portfolio Return Column", spec["Return Column"])
    result["Horizon Months"] = spec["Horizon Months"]
    result["Portfolio Diagnostic Caveat"] = spec.get("Portfolio Diagnostic Caveat", "objective-specific diagnostic")
    result["Training Label Resolution Cutoff"] = HOLDOUT_START_DATE
    rows.append(result)


def append_objective_predictions(frames, model_name, spec, frame, mask, score, evaluation_window="holdout"):
    pred = frame.loc[mask].copy().reset_index(drop=True)
    score_values = np.asarray(score, dtype=np.float32)
    if len(pred) != len(score_values):
        raise ValueError(f"Objective prediction length mismatch for {model_name}: rows={len(pred)}, scores={len(score_values)}")
    pred["Objective"] = spec["Objective"]
    pred["Target Column"] = spec["Target Column"]
    pred["Return Column"] = spec["Return Column"]
    pred["Portfolio Return Column"] = spec.get("Portfolio Return Column", spec["Return Column"])
    pred["Evaluation Window"] = evaluation_window
    pred["Model"] = model_name
    pred["score"] = score_values
    frames.append(pred)


def objective_label_resolved_mask(frame, spec, cutoff_date):
    if spec is None or cutoff_date is None:
        return np.ones(len(frame), dtype=bool)
    horizon = int(spec.get("Horizon Months", 1))
    signal_dates = pd.to_datetime(frame["date"])
    resolution_dates = (signal_dates.dt.to_period("M") + horizon).dt.to_timestamp(how="end")
    return (resolution_dates < pd.Timestamp(cutoff_date)).to_numpy()


def objective_subset(X, frame, target_column, spec=None, resolved_before=None):
    mask = frame[target_column].notna().to_numpy()
    if resolved_before is not None:
        mask = mask & objective_label_resolved_mask(frame, spec, resolved_before)
    X_subset = X.loc[mask].reset_index(drop=True)
    y_subset = frame.loc[mask, target_column].astype(int).reset_index(drop=True)
    return X_subset, y_subset, mask


def has_objective_rows(X, y):
    return len(X) > 0 and y.nunique() >= 2


def evaluate_objective_specific_retraining():
    if not OBJECTIVE_SPECIFIC_RETRAINING_ENABLED:
        return pd.DataFrame(), pd.DataFrame()
    rows = []
    pred_frames = []
    for spec in tqdm(objective_target_specs(), desc="Objective-specific reruns", unit="objective"):
        target_column = spec["Target Column"]
        required_frames = [selection_df, calibration_df, preholdout_df, holdout_df]
        if any(target_column not in frame.columns for frame in required_frames):
            model_errors.append({"Model": f"ObjectiveSpecific[{spec['Objective']}]", "Error": f"missing target column {target_column}"})
            continue
        X_sel, y_sel, sel_mask = objective_subset(X_selection, selection_df, target_column, spec=spec, resolved_before=calibration_df["date"].min())
        X_cal, y_cal, cal_mask = objective_subset(X_calibration, calibration_df, target_column, spec=spec, resolved_before=HOLDOUT_START_DATE)
        X_train, y_train, train_mask = objective_subset(X_preholdout, preholdout_df, target_column, spec=spec, resolved_before=HOLDOUT_START_DATE)
        X_test, y_test, test_mask = objective_subset(X_holdout, holdout_df, target_column)
        if not has_objective_rows(X_train, y_train) or not has_objective_rows(X_test, y_test):
            model_errors.append({"Model": f"ObjectiveSpecific[{spec['Objective']}]", "Error": "insufficient class diversity or rows for objective-specific holdout rerun"})
            continue
        cv_ap = objective_cv_average_precision(spec)
        if RUN_OBJECTIVE_SPECIFIC_LOGISTIC:
            try:
                model_name = f"ObjectiveSpecific[{spec['Objective']}][LogisticRegression]"
                model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=5000, class_weight="balanced", random_state=SEED))
                start = time.perf_counter()
                model.fit(X_train, y_train)
                fit_seconds = time.perf_counter() - start
                predict_start = time.perf_counter()
                score = positive_class_proba(model, X_test)
                predict_seconds = time.perf_counter() - predict_start
                append_objective_result(rows, model_name, "Logistic Regression", "objective_specific", spec, y_test, score, fit_seconds, predict_seconds, "objective-specific CPU Logistic Regression; disabled by default")
                append_objective_predictions(pred_frames, model_name, spec, holdout_df, test_mask, score, evaluation_window="holdout")
                del model, score
            except Exception as exc:
                model_errors.append({"Model": f"ObjectiveSpecific[{spec['Objective']}][LogisticRegression]", "Error": short_error(exc)})
        if RUN_OBJECTIVE_SPECIFIC_XGBOOST_LIGHT:
            try:
                model_name = f"ObjectiveSpecific[{spec['Objective']}][XGBoost light]"
                model = make_light_xgboost_estimator(y_train, OBJECTIVE_SPECIFIC_XGBOOST_N_ESTIMATORS)
                start = time.perf_counter()
                model.fit(X_train, y_train)
                fit_seconds = time.perf_counter() - start
                predict_start = time.perf_counter()
                score = predict_proba_in_chunks(model, X_test, desc=f"{model_name} holdout prediction")
                predict_seconds = time.perf_counter() - predict_start
                append_objective_result(rows, model_name, "XGBoost", "objective_specific", spec, y_test, score, fit_seconds, predict_seconds, "objective-specific fixed GPU XGBoost; training labels restricted to those resolved before holdout; chronological fixed-model CV AP saved; no target-specific hyperparameter search", cv_average_precision=cv_ap)
                append_objective_predictions(pred_frames, model_name, spec, holdout_df, test_mask, score, evaluation_window="holdout")
                del model, score
                cleanup_runtime_memory(f"after_{model_name}_prediction")
            except Exception as exc:
                model_errors.append({"Model": f"ObjectiveSpecific[{spec['Objective']}][XGBoost light]", "Error": short_error(exc)})
                cleanup_runtime_memory(f"after_objective_xgboost_{spec['Objective']}_error")
            if EVALUATE_CALIBRATION_VARIANTS and has_objective_rows(X_sel, y_sel) and has_objective_rows(X_cal, y_cal):
                try:
                    base_name = f"ObjectiveSpecific[{spec['Objective']}][XGBoost light] Calibration Base"
                    base_model = make_light_xgboost_estimator(y_sel, OBJECTIVE_SPECIFIC_XGBOOST_N_ESTIMATORS)
                    base_start = time.perf_counter()
                    base_model.fit(X_sel, y_sel)
                    base_fit_seconds = time.perf_counter() - base_start
                    predict_start = time.perf_counter()
                    base_cal_score = predict_proba_in_chunks(base_model, X_cal, desc=f"{base_name} calibration prediction")
                    base_holdout_score = predict_proba_in_chunks(base_model, X_test, desc=f"{base_name} holdout prediction")
                    base_predict_seconds = time.perf_counter() - predict_start
                    append_objective_result(rows, base_name, "XGBoost", "objective_specific_calibration_base", spec, y_cal, base_cal_score, base_fit_seconds, base_predict_seconds / 2.0, "objective-specific base model excludes calibration-window labels", evaluation_window="calibration", calibration="none_calibration_base", cv_average_precision=cv_ap)
                    append_objective_result(rows, base_name, "XGBoost", "objective_specific_calibration_base", spec, y_test, base_holdout_score, base_fit_seconds, base_predict_seconds / 2.0, "objective-specific base model excludes calibration-window labels", evaluation_window="holdout", calibration="none_calibration_base", cv_average_precision=cv_ap)
                    append_objective_predictions(pred_frames, base_name, spec, calibration_df, cal_mask, base_cal_score, evaluation_window="calibration")
                    append_objective_predictions(pred_frames, base_name, spec, holdout_df, test_mask, base_holdout_score, evaluation_window="holdout")
                    calibration_start = time.perf_counter()
                    calibrated_model = make_prefit_calibrator(base_model, X_cal, y_cal, method="sigmoid")
                    calibration_seconds = time.perf_counter() - calibration_start
                    predict_start = time.perf_counter()
                    calibrated_cal_score = predict_proba_in_chunks(calibrated_model, X_cal, desc=f"{base_name} sigmoid calibration prediction")
                    calibrated_holdout_score = predict_proba_in_chunks(calibrated_model, X_test, desc=f"{base_name} sigmoid holdout prediction")
                    calibrated_predict_seconds = time.perf_counter() - predict_start
                    calibrated_name = f"ObjectiveSpecific[{spec['Objective']}][XGBoost light] Calibrated"
                    append_objective_result(rows, calibrated_name, "XGBoost", "objective_specific_calibrated", spec, y_cal, calibrated_cal_score, base_fit_seconds, calibrated_predict_seconds / 2.0, "sigmoid calibration fitted on objective-specific calibration window", evaluation_window="calibration", calibration="sigmoid", calibration_seconds=calibration_seconds, cv_average_precision=cv_ap)
                    append_objective_result(rows, calibrated_name, "XGBoost", "objective_specific_calibrated", spec, y_test, calibrated_holdout_score, base_fit_seconds, calibrated_predict_seconds / 2.0, "sigmoid calibration fitted on objective-specific calibration window", evaluation_window="holdout", calibration="sigmoid", calibration_seconds=calibration_seconds, cv_average_precision=cv_ap)
                    append_objective_predictions(pred_frames, calibrated_name, spec, calibration_df, cal_mask, calibrated_cal_score, evaluation_window="calibration")
                    append_objective_predictions(pred_frames, calibrated_name, spec, holdout_df, test_mask, calibrated_holdout_score, evaluation_window="holdout")
                    del calibrated_model, calibrated_cal_score, calibrated_holdout_score
                    del base_model, base_cal_score, base_holdout_score
                    cleanup_runtime_memory(f"after_{base_name}_calibration")
                except Exception as exc:
                    model_errors.append({"Model": f"ObjectiveSpecific[{spec['Objective']}][XGBoost light][calibration]", "Error": short_error(exc)})
                    cleanup_runtime_memory(f"after_objective_xgboost_{spec['Objective']}_calibration_error")
        if RUN_OBJECTIVE_SPECIFIC_XGBOOST_SEARCH:
            try:
                search, search_seconds, best_params = run_objective_xgboost_search(spec)
                model_name = f"ObjectiveSpecific[{spec['Objective']}][XGBoost searched]"
                model = best_estimator_from_search(search)
                start = time.perf_counter()
                model.fit(X_train, y_train)
                final_fit_seconds = time.perf_counter() - start
                predict_start = time.perf_counter()
                score = predict_proba_in_chunks(model, X_test, desc=f"{model_name} holdout prediction")
                predict_seconds = time.perf_counter() - predict_start
                append_objective_result(rows, model_name, "XGBoost", "objective_specific_searched", spec, y_test, score, search_seconds + final_fit_seconds, predict_seconds, "objective-specific GPU XGBoost with target-specific chronological randomized search; training labels restricted to outcomes resolved before validation or holdout", cv_average_precision=search.best_score_, best_params=best_params)
                append_objective_predictions(pred_frames, model_name, spec, holdout_df, test_mask, score, evaluation_window="holdout")
                del model, score
                cleanup_runtime_memory(f"after_{model_name}_prediction")
                if EVALUATE_CALIBRATION_VARIANTS and has_objective_rows(X_sel, y_sel) and has_objective_rows(X_cal, y_cal):
                    base_name = f"ObjectiveSpecific[{spec['Objective']}][XGBoost searched] Calibration Base"
                    base_model = best_estimator_from_search(search)
                    base_start = time.perf_counter()
                    base_model.fit(X_sel, y_sel)
                    base_fit_seconds = time.perf_counter() - base_start
                    predict_start = time.perf_counter()
                    base_cal_score = predict_proba_in_chunks(base_model, X_cal, desc=f"{base_name} calibration prediction")
                    base_holdout_score = predict_proba_in_chunks(base_model, X_test, desc=f"{base_name} holdout prediction")
                    base_predict_seconds = time.perf_counter() - predict_start
                    append_objective_result(rows, base_name, "XGBoost", "objective_specific_searched_calibration_base", spec, y_cal, base_cal_score, search_seconds + base_fit_seconds, base_predict_seconds / 2.0, "searched objective-specific base model excludes calibration-window labels", evaluation_window="calibration", calibration="none_calibration_base", cv_average_precision=search.best_score_, best_params=best_params)
                    append_objective_result(rows, base_name, "XGBoost", "objective_specific_searched_calibration_base", spec, y_test, base_holdout_score, search_seconds + base_fit_seconds, base_predict_seconds / 2.0, "searched objective-specific base model excludes calibration-window labels", evaluation_window="holdout", calibration="none_calibration_base", cv_average_precision=search.best_score_, best_params=best_params)
                    append_objective_predictions(pred_frames, base_name, spec, calibration_df, cal_mask, base_cal_score, evaluation_window="calibration")
                    append_objective_predictions(pred_frames, base_name, spec, holdout_df, test_mask, base_holdout_score, evaluation_window="holdout")
                    calibration_start = time.perf_counter()
                    calibrated_model = make_prefit_calibrator(base_model, X_cal, y_cal, method="sigmoid")
                    calibration_seconds = time.perf_counter() - calibration_start
                    predict_start = time.perf_counter()
                    calibrated_cal_score = predict_proba_in_chunks(calibrated_model, X_cal, desc=f"{base_name} sigmoid calibration prediction")
                    calibrated_holdout_score = predict_proba_in_chunks(calibrated_model, X_test, desc=f"{base_name} sigmoid holdout prediction")
                    calibrated_predict_seconds = time.perf_counter() - predict_start
                    calibrated_name = f"ObjectiveSpecific[{spec['Objective']}][XGBoost searched] Calibrated"
                    append_objective_result(rows, calibrated_name, "XGBoost", "objective_specific_searched_calibrated", spec, y_cal, calibrated_cal_score, search_seconds + base_fit_seconds, calibrated_predict_seconds / 2.0, "sigmoid calibration fitted on objective-specific calibration window after target-specific search", evaluation_window="calibration", calibration="sigmoid", calibration_seconds=calibration_seconds, cv_average_precision=search.best_score_, best_params=best_params)
                    append_objective_result(rows, calibrated_name, "XGBoost", "objective_specific_searched_calibrated", spec, y_test, calibrated_holdout_score, search_seconds + base_fit_seconds, calibrated_predict_seconds / 2.0, "sigmoid calibration fitted on objective-specific calibration window after target-specific search", evaluation_window="holdout", calibration="sigmoid", calibration_seconds=calibration_seconds, cv_average_precision=search.best_score_, best_params=best_params)
                    append_objective_predictions(pred_frames, calibrated_name, spec, calibration_df, cal_mask, calibrated_cal_score, evaluation_window="calibration")
                    append_objective_predictions(pred_frames, calibrated_name, spec, holdout_df, test_mask, calibrated_holdout_score, evaluation_window="holdout")
                    del calibrated_model, calibrated_cal_score, calibrated_holdout_score
                    del base_model, base_cal_score, base_holdout_score
                    cleanup_runtime_memory(f"after_{base_name}_calibration")
                del search
                cleanup_runtime_memory(f"after_objective_xgboost_search_{spec['Objective']}")
            except Exception as exc:
                model_errors.append({"Model": f"ObjectiveSpecific[{spec['Objective']}][XGBoost searched]", "Error": short_error(exc)})
                cleanup_runtime_memory(f"after_objective_xgboost_search_{spec['Objective']}_error")
        if RUN_OBJECTIVE_SPECIFIC_TABPFN:
            try:
                model_name = f"ObjectiveSpecific[{spec['Objective']}][TabPFN direct]"
                model = make_tabpfn_classifier()
                start = time.perf_counter()
                model.fit(to_numpy_float32(X_train), y_to_numpy(y_train))
                fit_seconds = time.perf_counter() - start
                predict_start = time.perf_counter()
                score = positive_class_proba(model, to_numpy_float32(X_test))
                predict_seconds = time.perf_counter() - predict_start
                append_objective_result(rows, model_name, "TabPFN", "objective_specific", spec, y_test, score, fit_seconds, predict_seconds, "objective-specific direct TabPFN rerun with training labels restricted to those resolved before holdout; separate from main one-month target")
                append_objective_predictions(pred_frames, model_name, spec, holdout_df, test_mask, score, evaluation_window="holdout")
                del model, score
                cleanup_runtime_memory(f"after_{model_name}_prediction")
            except Exception as exc:
                model_errors.append({"Model": f"ObjectiveSpecific[{spec['Objective']}][TabPFN direct]", "Error": short_error(exc)})
                cleanup_runtime_memory(f"after_objective_tabpfn_{spec['Objective']}_error")
            if EVALUATE_CALIBRATION_VARIANTS and has_objective_rows(X_sel, y_sel) and has_objective_rows(X_cal, y_cal):
                try:
                    base_name = f"ObjectiveSpecific[{spec['Objective']}][TabPFN direct] Calibration Base"
                    base_model = make_tabpfn_classifier()
                    base_start = time.perf_counter()
                    base_model.fit(to_numpy_float32(X_sel), y_to_numpy(y_sel))
                    base_fit_seconds = time.perf_counter() - base_start
                    predict_start = time.perf_counter()
                    base_cal_score = positive_class_proba(base_model, to_numpy_float32(X_cal))
                    base_holdout_score = positive_class_proba(base_model, to_numpy_float32(X_test))
                    base_predict_seconds = time.perf_counter() - predict_start
                    append_objective_result(rows, base_name, "TabPFN", "objective_specific_calibration_base", spec, y_cal, base_cal_score, base_fit_seconds, base_predict_seconds / 2.0, "objective-specific direct TabPFN context excludes calibration-window labels", evaluation_window="calibration", calibration="none_calibration_base")
                    append_objective_result(rows, base_name, "TabPFN", "objective_specific_calibration_base", spec, y_test, base_holdout_score, base_fit_seconds, base_predict_seconds / 2.0, "objective-specific direct TabPFN context excludes calibration-window labels", evaluation_window="holdout", calibration="none_calibration_base")
                    append_objective_predictions(pred_frames, base_name, spec, calibration_df, cal_mask, base_cal_score, evaluation_window="calibration")
                    append_objective_predictions(pred_frames, base_name, spec, holdout_df, test_mask, base_holdout_score, evaluation_window="holdout")
                    del base_model, base_cal_score, base_holdout_score
                    cleanup_runtime_memory(f"after_{base_name}_prediction")
                except Exception as exc:
                    model_errors.append({"Model": f"ObjectiveSpecific[{spec['Objective']}][TabPFN direct][calibration_base]", "Error": short_error(exc)})
                    cleanup_runtime_memory(f"after_objective_tabpfn_{spec['Objective']}_calibration_base_error")
    summary = pd.DataFrame(rows)
    predictions_out = pd.concat(pred_frames, ignore_index=True) if pred_frames else pd.DataFrame()
    if len(summary) > 0:
        summary = summary.sort_values(["Objective", "Evaluation Window", "Average Precision"], ascending=[True, True, False]).reset_index(drop=True)
        display(summary.round(4))
        save_artifact_table(summary, "objective_specific_retraining_summary.csv", description="Objective-specific model reruns for benchmark-relative and multi-horizon targets with calibration-window rows, fixed-model and target-specific searched XGBoost chronological CV evidence where available, and training labels restricted to outcomes resolved before holdout.")
    if len(predictions_out) > 0:
        save_columns = [
            "Objective", "Target Column", "Return Column", "Portfolio Return Column", "Evaluation Window", "Model", "date", "month", "asset", "score",
            "target_top_k_next_1m", "target_outperform_benchmark_next_1m", "forward_1m_return", "forward_1m_vs_benchmark_return",
        ]
        for horizon in MULTI_HORIZON_MONTHS:
            save_columns.extend([f"target_top_k_next_{horizon}m", f"forward_{horizon}m_return"])
        save_columns = [column for column in save_columns if column in predictions_out.columns]
        display(predictions_out[save_columns].head(25))
        save_artifact_table(predictions_out[save_columns].copy(), "objective_specific_prediction_scores.csv", description="Calibration and holdout scores from objective-specific reruns.")

        objective_rank_frames = []
        for (objective_name, model_name, evaluation_window), group in predictions_out.groupby(["Objective", "Model", "Evaluation Window"], sort=False):
            spec = OBJECTIVE_SPEC_BY_NAME.get(objective_name)
            if spec is None:
                continue
            target_column = spec["Target Column"]
            return_column = spec["Return Column"]
            if target_column not in group.columns or return_column not in group.columns:
                continue
            diagnostic_frame = group.copy()
            diagnostic_frame["Evaluation Window"] = evaluation_window
            diagnostic_frame["target_top_k_next_1m"] = pd.to_numeric(diagnostic_frame[target_column], errors="coerce")
            diagnostic_frame["forward_1m_return"] = pd.to_numeric(diagnostic_frame[return_column], errors="coerce")
            rank_frame = monthly_rank_quality(diagnostic_frame, top_k=TARGET_TOP_K)
            if len(rank_frame) > 0:
                rank_frame.insert(0, "Objective", objective_name)
                rank_frame.insert(1, "Target Column", target_column)
                rank_frame.insert(2, "Return Column", return_column)
                rank_frame.insert(3, "Portfolio Diagnostic Caveat", spec.get("Portfolio Diagnostic Caveat", "objective-specific diagnostic"))
                objective_rank_frames.append(rank_frame)
        objective_rank_summary = pd.concat(objective_rank_frames, ignore_index=True) if objective_rank_frames else pd.DataFrame()
        if len(objective_rank_summary) > 0:
            objective_rank_aggregate = (
                objective_rank_summary.groupby(["Objective", "Model", "Evaluation Window", "Portfolio Diagnostic Caveat"])
                .agg(
                    months=("month", "nunique"),
                    mean_spearman_ic=("spearman_ic", "mean"),
                    median_spearman_ic=("spearman_ic", "median"),
                    mean_top_k_hit_rate=("top_k_hit_rate", "mean"),
                    mean_selected_forward_return=("selected_mean_forward_return", "mean"),
                    mean_universe_forward_return=("universe_mean_forward_return", "mean"),
                    mean_active_forward_return=("active_forward_return", "mean"),
                )
                .reset_index()
                .sort_values(["Objective", "Evaluation Window", "mean_active_forward_return"], ascending=[True, True, False])
            )
            display(objective_rank_aggregate.round(4))
            save_artifact_table(objective_rank_summary, "objective_specific_monthly_rank_summary.csv", description="Monthly rank diagnostics for objective-specific rerun scores against each objective's own target and return column.")
            save_artifact_table(objective_rank_aggregate, "objective_specific_monthly_rank_aggregate.csv", description="Aggregated monthly rank diagnostics for objective-specific rerun scores.")
    return summary, predictions_out


objective_specific_retraining_summary, objective_specific_prediction_scores = evaluate_objective_specific_retraining()
if model_errors:
    model_error_summary = pd.DataFrame(model_errors)
    display(model_error_summary)
    save_artifact_table(model_error_summary, "model_error_summary.csv", description="All model errors observed through main, feature-sensitivity, and objective-specific reruns.")

def has_monthly_rank_signal(score_frame, score_column="score"):
    unique_counts = score_frame.groupby("month")[score_column].nunique(dropna=True)
    return bool(len(unique_counts) > 0 and (unique_counts > 1).all())


def weights_from_score_frame(score_frame, score_column="score", top_k=TARGET_TOP_K):
    months = sorted(score_frame["month"].unique())
    assets = sorted(score_frame["asset"].unique())
    weights = pd.DataFrame(0.0, index=months, columns=assets)
    for month, group in score_frame.groupby("month"):
        sort_columns = [score_column]
        ascending = [False]
        if "asset" in group.columns:
            sort_columns.append("asset")
            ascending.append(True)
        selected = group.sort_values(sort_columns, ascending=ascending).head(min(top_k, len(group)))
        if len(selected) == 0:
            continue
        weights.loc[month, selected["asset"].tolist()] = 1.0 / len(selected)
    return weights


def normalize_with_max_weight(raw_weights, max_weight=RISK_WEIGHTED_TOP_K_MAX_WEIGHT):
    raw = pd.Series(raw_weights, dtype=float).replace([np.inf, -np.inf], np.nan).dropna()
    raw = raw.loc[raw > 0.0]
    if len(raw) == 0:
        return raw
    weights = raw / raw.sum()
    if max_weight is None or not np.isfinite(max_weight) or max_weight <= 0.0:
        return weights
    if max_weight * len(weights) < 1.0:
        max_weight = 1.0 / len(weights)
    capped = pd.Series(0.0, index=weights.index, dtype=float)
    free = weights.copy()
    remaining = 1.0
    for _ in range(len(weights)):
        if len(free) == 0:
            break
        candidate = free / free.sum() * remaining
        over = candidate > max_weight
        if not over.any():
            capped.loc[candidate.index] = candidate
            remaining = 0.0
            break
        capped.loc[candidate.loc[over].index] = max_weight
        remaining = max(0.0, 1.0 - capped.sum())
        free = free.loc[~over]
    if capped.sum() > 0.0:
        capped = capped / capped.sum()
    return capped


def risk_weighted_weights_from_score_frame(score_frame, base_frame, score_column="score", top_k=TARGET_TOP_K):
    months = sorted(score_frame["month"].unique())
    assets = sorted(score_frame["asset"].unique())
    weights = pd.DataFrame(0.0, index=months, columns=assets)
    risk_lookup = base_frame.pivot_table(index="month", columns="asset", values="asset_rv_63d", aggfunc="first")
    for month, group in score_frame.groupby("month"):
        sort_columns = [score_column]
        ascending = [False]
        if "asset" in group.columns:
            sort_columns.append("asset")
            ascending.append(True)
        selected = group.sort_values(sort_columns, ascending=ascending).head(min(top_k, len(group)))
        if len(selected) == 0:
            continue
        selected_assets = selected["asset"].tolist()
        if month in risk_lookup.index:
            risk = risk_lookup.loc[month, selected_assets].astype(float).replace(0.0, np.nan)
        else:
            risk = pd.Series(np.nan, index=selected_assets)
        raw = 1.0 / risk.clip(lower=0.02)
        if raw.replace([np.inf, -np.inf], np.nan).dropna().empty:
            weights.loc[month, selected_assets] = 1.0 / len(selected_assets)
        else:
            normalized = normalize_with_max_weight(raw, RISK_WEIGHTED_TOP_K_MAX_WEIGHT)
            weights.loc[month, normalized.index] = normalized
    return weights


def apply_turnover_cap(weights, turnover_cap):
    if turnover_cap is None or not np.isfinite(turnover_cap):
        return weights.copy()
    capped = weights.copy().fillna(0.0).astype(float)
    if len(capped) == 0:
        return capped
    previous = pd.Series(0.0, index=capped.columns)
    adjusted_rows = []
    for _, target in capped.iterrows():
        target = target.astype(float).fillna(0.0)
        if previous.abs().sum() == 0.0:
            adjusted = target
        else:
            diff = target - previous
            turnover = float(diff.abs().sum())
            if turnover > turnover_cap and turnover > 0.0:
                adjusted = previous + diff * (turnover_cap / turnover)
            else:
                adjusted = target
        adjusted = adjusted.clip(lower=0.0)
        total_weight = float(adjusted.sum())
        if total_weight > 0.0:
            adjusted = adjusted / total_weight
        adjusted_rows.append(adjusted)
        previous = adjusted
    return pd.DataFrame(adjusted_rows, index=capped.index, columns=capped.columns)


def deterministic_weight_frames():
    holdout_base = holdout_df.copy()
    months = sorted(holdout_base["month"].unique())
    assets = sorted(holdout_base["asset"].unique())
    frames = {}

    equal_weight = pd.DataFrame(0.0, index=months, columns=assets)
    for month, group in holdout_base.groupby("month"):
        available = group["asset"].tolist()
        equal_weight.loc[month, available] = 1.0 / len(available)
    frames["Equal weight universe"] = equal_weight

    spy_only = pd.DataFrame(0.0, index=months, columns=assets)
    if "SPY" in assets:
        spy_only["SPY"] = 1.0
        frames["SPY only"] = spy_only

    sixty_forty = pd.DataFrame(0.0, index=months, columns=assets)
    if "SPY" in assets and "TLT" in assets:
        sixty_forty["SPY"] = 0.60
        sixty_forty["TLT"] = 0.40
        frames["60/40 SPY/TLT"] = sixty_forty

    inv_vol = pd.DataFrame(0.0, index=months, columns=assets)
    for month, group in holdout_base.groupby("month"):
        vol = group.set_index("asset")["asset_rv_63d"].astype(float).replace(0.0, np.nan)
        raw = 1.0 / vol.clip(lower=0.02)
        raw = raw.replace([np.inf, -np.inf], np.nan).dropna()
        if len(raw) > 0:
            inv_vol.loc[month, raw.index] = raw / raw.sum()
    frames["Inverse volatility universe"] = inv_vol

    momentum_scores = holdout_base[["month", "asset", "asset_return_252d"]].rename(columns={"asset_return_252d": "score"}).copy()
    frames["12M momentum top-k portfolio"] = weights_from_score_frame(momentum_scores, top_k=TARGET_TOP_K)
    low_vol_scores = holdout_base[["month", "asset", "asset_rv_63d"]].rename(columns={"asset_rv_63d": "score"}).copy()
    low_vol_scores["score"] = -low_vol_scores["score"].astype(float)
    frames["Low volatility top-k portfolio"] = weights_from_score_frame(low_vol_scores, top_k=TARGET_TOP_K)
    return frames


def returns_pivot_for_frame(frame):
    return frame.pivot_table(index="month", columns="asset", values="forward_1m_return", aggfunc="first").sort_index()


def portfolio_metrics_from_weights(weights, returns, transaction_cost_bps=TRANSACTION_COST_BPS):
    weights = weights.reindex(index=returns.index, columns=returns.columns).fillna(0.0)
    returns = returns.reindex_like(weights).fillna(0.0)
    gross_returns = (weights * returns).sum(axis=1)
    turnover = weights.diff().abs().sum(axis=1)
    if len(turnover) > 0:
        turnover.iloc[0] = weights.iloc[0].abs().sum()
    costs = turnover * transaction_cost_bps / 10000.0
    net_returns = gross_returns - costs
    equity = (1.0 + net_returns).cumprod()
    years = len(net_returns) / MONTHS_PER_YEAR
    cagr = equity.iloc[-1] ** (1.0 / years) - 1.0 if years > 0 and equity.iloc[-1] > 0 else np.nan
    ann_vol = net_returns.std(ddof=1) * math.sqrt(MONTHS_PER_YEAR)
    sharpe = net_returns.mean() / net_returns.std(ddof=1) * math.sqrt(MONTHS_PER_YEAR) if net_returns.std(ddof=1) > 0 else np.nan
    downside = net_returns[net_returns < 0.0]
    sortino = net_returns.mean() / downside.std(ddof=1) * math.sqrt(MONTHS_PER_YEAR) if len(downside) > 1 and downside.std(ddof=1) > 0 else np.nan
    drawdown = equity / equity.cummax() - 1.0
    return {
        "Months": len(net_returns),
        "CAGR": cagr,
        "Annualized Volatility": ann_vol,
        "Sharpe": sharpe,
        "Sortino": sortino,
        "Max Drawdown": drawdown.min(),
        "Average Monthly Return": net_returns.mean(),
        "Annualized Turnover": turnover.sum() / years if years > 0 else np.nan,
        "Average Holdings": (weights > 0).sum(axis=1).mean(),
        "Total Transaction Cost": costs.sum(),
        "Final Equity": equity.iloc[-1],
    }, equity, net_returns, turnover


objective_specific_horizon_return_rows = []
if "objective_specific_prediction_scores" in globals() and len(objective_specific_prediction_scores) > 0:
    objective_holdout_scores = objective_specific_prediction_scores.loc[objective_specific_prediction_scores["Evaluation Window"] == "holdout"].copy()
    for (objective_name, model_name), group in objective_holdout_scores.groupby(["Objective", "Model"], sort=False):
        spec = OBJECTIVE_SPEC_BY_NAME.get(objective_name)
        if spec is None:
            continue
        return_column = spec.get("Portfolio Return Column", spec["Return Column"])
        target_column = spec["Target Column"]
        horizon_months = int(spec.get("Horizon Months", 1))
        if return_column not in group.columns or target_column not in group.columns:
            model_errors.append({"Model": f"ObjectiveHorizonDiagnostic[{objective_name}][{model_name}]", "Error": f"missing target or return column for {return_column}"})
            continue
        score_frame = group[["month", "asset", "score", return_column, target_column]].copy()
        if not has_monthly_rank_signal(score_frame[["month", "asset", "score"]]):
            continue
        weights = weights_from_score_frame(score_frame[["month", "asset", "score"]], top_k=TARGET_TOP_K)
        returns_for_objective = group.pivot_table(index="month", columns="asset", values=return_column, aggfunc="first").sort_index()
        targets_for_objective = group.pivot_table(index="month", columns="asset", values=target_column, aggfunc="first").sort_index()
        aligned_weights = weights.reindex(index=returns_for_objective.index, columns=returns_for_objective.columns).fillna(0.0)
        selected_horizon_return = (aligned_weights * returns_for_objective).sum(axis=1)
        universe_horizon_return = returns_for_objective.mean(axis=1)
        active_horizon_return = selected_horizon_return - universe_horizon_return
        selected_target_rate = (aligned_weights * targets_for_objective.fillna(0.0)).sum(axis=1)
        turnover = aligned_weights.diff().abs().sum(axis=1)
        if len(turnover) > 0:
            turnover.iloc[0] = aligned_weights.iloc[0].abs().sum()
        average_holdings = (aligned_weights > 0).sum(axis=1)
        for month in returns_for_objective.index:
            objective_specific_horizon_return_rows.append(
                {
                    "Strategy": f"{objective_name} | {model_name}",
                    "Objective": objective_name,
                    "Model": model_name,
                    "month": month,
                    "Horizon Months": horizon_months,
                    "Return Column": return_column,
                    "Target Column": target_column,
                    "Selected Horizon Return": float(selected_horizon_return.loc[month]) if pd.notna(selected_horizon_return.loc[month]) else np.nan,
                    "Universe Horizon Return": float(universe_horizon_return.loc[month]) if pd.notna(universe_horizon_return.loc[month]) else np.nan,
                    "Active Horizon Return": float(active_horizon_return.loc[month]) if pd.notna(active_horizon_return.loc[month]) else np.nan,
                    "Selected Target Rate": float(selected_target_rate.loc[month]) if pd.notna(selected_target_rate.loc[month]) else np.nan,
                    "Monthly Turnover": float(turnover.loc[month]) if pd.notna(turnover.loc[month]) else np.nan,
                    "Average Holdings": float(average_holdings.loc[month]) if pd.notna(average_holdings.loc[month]) else np.nan,
                    "Diagnostic Type": "one_month_score_to_return" if horizon_months == 1 else "overlapping_horizon_score_to_return",
                    "Portfolio Diagnostic Caveat": spec.get("Portfolio Diagnostic Caveat", "objective-specific diagnostic"),
                }
            )

objective_specific_horizon_return_monthly = pd.DataFrame(objective_specific_horizon_return_rows)
if len(objective_specific_horizon_return_monthly) > 0:
    def horizon_sharpe_proxy(values, horizon_months):
        values = pd.to_numeric(values, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
        if len(values) < 2 or values.std(ddof=1) <= 0:
            return np.nan
        return float(values.mean() / values.std(ddof=1) * math.sqrt(MONTHS_PER_YEAR / max(int(horizon_months), 1)))

    summary_rows = []
    group_columns = ["Objective", "Model", "Strategy", "Horizon Months", "Return Column", "Target Column", "Diagnostic Type", "Portfolio Diagnostic Caveat"]
    for keys, group in objective_specific_horizon_return_monthly.groupby(group_columns, sort=False):
        row = dict(zip(group_columns, keys))
        horizon_months = int(row["Horizon Months"])
        selected_returns = pd.to_numeric(group["Selected Horizon Return"], errors="coerce")
        universe_returns = pd.to_numeric(group["Universe Horizon Return"], errors="coerce")
        active_returns = pd.to_numeric(group["Active Horizon Return"], errors="coerce")
        row.update(
            {
                "Observation Months": int(group["month"].nunique()),
                "Mean Selected Horizon Return": float(selected_returns.mean()),
                "Median Selected Horizon Return": float(selected_returns.median()),
                "Mean Universe Horizon Return": float(universe_returns.mean()),
                "Mean Active Horizon Return": float(active_returns.mean()),
                "Median Active Horizon Return": float(active_returns.median()),
                "Selected Horizon Return Volatility": float(selected_returns.std(ddof=1)),
                "Active Horizon Return Volatility": float(active_returns.std(ddof=1)),
                "Selected Horizon Sharpe Proxy": horizon_sharpe_proxy(selected_returns, horizon_months),
                "Active Horizon Sharpe Proxy": horizon_sharpe_proxy(active_returns, horizon_months),
                "Mean Selected Target Rate": float(pd.to_numeric(group["Selected Target Rate"], errors="coerce").mean()),
                "Mean Monthly Turnover": float(pd.to_numeric(group["Monthly Turnover"], errors="coerce").mean()),
                "Max Monthly Turnover": float(pd.to_numeric(group["Monthly Turnover"], errors="coerce").max()),
                "Mean Holdings": float(pd.to_numeric(group["Average Holdings"], errors="coerce").mean()),
                "Compounds Overlapping Returns": False,
            }
        )
        summary_rows.append(row)
    objective_specific_portfolio_summary = pd.DataFrame(summary_rows).sort_values(["Objective", "Active Horizon Sharpe Proxy"], ascending=[True, False]).reset_index(drop=True)
    display(objective_specific_portfolio_summary.round(4))
    save_artifact_table(objective_specific_horizon_return_monthly, "objective_specific_horizon_return_monthly.csv", description="Monthly selected, universe, and active horizon returns from objective-specific rerun scores. Multi-horizon rows are overlapping-return diagnostics and are not compounded as a portfolio backtest.")
    save_artifact_table(objective_specific_portfolio_summary, "objective_specific_portfolio_summary.csv", description="Horizon-aware objective-specific score-to-return diagnostics. 3M/6M rows report horizon returns and Sharpe proxies rather than compounded portfolio CAGR.")
    objective_specific_1m_monthly = objective_specific_horizon_return_monthly.loc[objective_specific_horizon_return_monthly["Horizon Months"] == 1].copy()
    if len(objective_specific_1m_monthly) > 0:
        objective_specific_1m_returns = objective_specific_1m_monthly.pivot_table(index="month", columns="Strategy", values="Selected Horizon Return", aggfunc="first").sort_index()
        objective_specific_1m_returns.index.name = "month"
        save_artifact_table(objective_specific_1m_returns.reset_index(), "objective_specific_portfolio_monthly_returns.csv", description="One-month objective-specific selected return streams only. Multi-horizon diagnostics are saved in objective_specific_horizon_return_monthly.csv.")

holdout_returns = returns_pivot_for_frame(holdout_df)
portfolio_rows = []
equity_curves = pd.DataFrame(index=holdout_returns.index)
monthly_strategy_returns = pd.DataFrame(index=holdout_returns.index)
turnover_curves = pd.DataFrame(index=holdout_returns.index)
weight_frames = {}

for name, weights in deterministic_weight_frames().items():
    metrics, equity, net_returns, turnover = portfolio_metrics_from_weights(weights, holdout_returns)
    metrics.update({"Strategy": name, "Source": "deterministic"})
    portfolio_rows.append(metrics)
    equity_curves[name] = equity
    monthly_strategy_returns[name] = net_returns
    turnover_curves[name] = turnover
    weight_frames[name] = weights

all_holdout_predictions = all_prediction_frame.loc[all_prediction_frame["Evaluation Window"] == "holdout"].copy()
selected_holdout_predictions = exclude_feature_sensitivity(all_holdout_predictions)
feature_sensitivity_holdout_predictions = all_holdout_predictions.loc[all_holdout_predictions["Model"].astype(str).map(is_feature_sensitivity_model)].copy()
portfolio_exclusion_rows = []
score_portfolio_groups = []
for model_name, score_frame in selected_holdout_predictions.groupby("Model"):
    duplicate_strategy = DUPLICATE_DETERMINISTIC_PORTFOLIO_MODELS.get(model_name)
    if duplicate_strategy in weight_frames:
        portfolio_exclusion_rows.append(
            {
                "Model": model_name,
                "Reason": f"excluded from score-driven portfolio because it duplicates deterministic benchmark {duplicate_strategy}",
                "Holdout Months": score_frame["month"].nunique(),
            }
        )
        continue
    if not has_monthly_rank_signal(score_frame):
        portfolio_exclusion_rows.append(
            {
                "Model": model_name,
                "Reason": "excluded from score-driven portfolio because at least one holdout month has no cross-sectional score variation",
                "Holdout Months": score_frame["month"].nunique(),
            }
        )
        continue
    score_portfolio_groups.append((model_name, score_frame))

if portfolio_exclusion_rows:
    portfolio_exclusion_summary = pd.DataFrame(portfolio_exclusion_rows)
    display(portfolio_exclusion_summary)
    save_artifact_table(portfolio_exclusion_summary, "portfolio_exclusion_summary.csv")
else:
    portfolio_exclusion_summary = pd.DataFrame(columns=["Model", "Reason", "Holdout Months"])

for model_name, score_frame in tqdm(score_portfolio_groups, desc="Building score-driven portfolios", unit="model"):
    weights = weights_from_score_frame(score_frame, top_k=TARGET_TOP_K)
    metrics, equity, net_returns, turnover = portfolio_metrics_from_weights(weights, holdout_returns)
    metrics.update({"Strategy": model_name, "Source": "model_score"})
    portfolio_rows.append(metrics)
    equity_curves[model_name] = equity
    monthly_strategy_returns[model_name] = net_returns
    turnover_curves[model_name] = turnover
    weight_frames[model_name] = weights

    risk_strategy_name = f"{model_name} | inverse-vol capped top-k"
    risk_weights = risk_weighted_weights_from_score_frame(score_frame, holdout_df, top_k=TARGET_TOP_K)
    risk_metrics, risk_equity, risk_net_returns, risk_turnover = portfolio_metrics_from_weights(risk_weights, holdout_returns)
    risk_metrics.update({"Strategy": risk_strategy_name, "Source": "model_score_inverse_vol_capped", "Max Asset Weight Cap": RISK_WEIGHTED_TOP_K_MAX_WEIGHT})
    portfolio_rows.append(risk_metrics)
    equity_curves[risk_strategy_name] = risk_equity
    monthly_strategy_returns[risk_strategy_name] = risk_net_returns
    turnover_curves[risk_strategy_name] = risk_turnover
    weight_frames[risk_strategy_name] = risk_weights

portfolio_summary = pd.DataFrame(portfolio_rows).sort_values(["Sharpe", "Final Equity"], ascending=[False, False]).reset_index(drop=True)
display(portfolio_summary.round(4))
save_artifact_table(portfolio_summary, "portfolio_summary.csv", description="Holdout monthly allocation diagnostics after transaction costs.")
save_artifact_table(equity_curves.reset_index(names="month"), "portfolio_equity_curves.csv")
save_artifact_table(monthly_strategy_returns.reset_index(names="month"), "portfolio_monthly_returns.csv")
save_artifact_table(turnover_curves.reset_index(names="month"), "portfolio_turnover.csv")

feature_sensitivity_portfolio_rows = []
feature_sensitivity_portfolio_monthly_returns = pd.DataFrame(index=holdout_returns.index)
if len(feature_sensitivity_holdout_predictions) > 0:
    for model_name, score_frame in tqdm(list(feature_sensitivity_holdout_predictions.groupby("Model")), desc="Building feature-sensitivity portfolios", unit="model"):
        if not has_monthly_rank_signal(score_frame):
            continue
        weights = weights_from_score_frame(score_frame, top_k=TARGET_TOP_K)
        metrics, _, net_returns, turnover = portfolio_metrics_from_weights(weights, holdout_returns)
        metrics.update({"Strategy": model_name, "Source": "feature_sensitivity_model_score", "Diagnostic Scope": "feature_policy_sensitivity"})
        feature_sensitivity_portfolio_rows.append(metrics)
        feature_sensitivity_portfolio_monthly_returns[model_name] = net_returns
        risk_strategy_name = f"{model_name} | inverse-vol capped top-k"
        risk_weights = risk_weighted_weights_from_score_frame(score_frame, holdout_df, top_k=TARGET_TOP_K)
        risk_metrics, _, risk_net_returns, _ = portfolio_metrics_from_weights(risk_weights, holdout_returns)
        risk_metrics.update({"Strategy": risk_strategy_name, "Source": "feature_sensitivity_inverse_vol_capped", "Diagnostic Scope": "feature_policy_sensitivity", "Max Asset Weight Cap": RISK_WEIGHTED_TOP_K_MAX_WEIGHT})
        feature_sensitivity_portfolio_rows.append(risk_metrics)
        feature_sensitivity_portfolio_monthly_returns[risk_strategy_name] = risk_net_returns
feature_sensitivity_portfolio_summary = pd.DataFrame(feature_sensitivity_portfolio_rows)
if len(feature_sensitivity_portfolio_summary) > 0:
    feature_sensitivity_portfolio_summary = feature_sensitivity_portfolio_summary.sort_values(["Sharpe", "Final Equity"], ascending=[False, False]).reset_index(drop=True)
    display(feature_sensitivity_portfolio_summary.round(4))
    save_artifact_table(feature_sensitivity_portfolio_summary, "feature_sensitivity_portfolio_summary.csv", description="Separate portfolio diagnostics for feature-policy sensitivity reruns. These rows are intentionally excluded from the headline portfolio_summary.csv.")
    feature_sensitivity_portfolio_monthly_returns.index.name = "month"
    save_artifact_table(feature_sensitivity_portfolio_monthly_returns.reset_index(), "feature_sensitivity_portfolio_monthly_returns.csv", description="Monthly returns for feature-policy sensitivity score portfolios, saved separately from headline portfolio returns.")

# Store weights in long format for offline inspection.
weight_rows = []
for strategy, weights in weight_frames.items():
    long_weights = weights.reset_index(names="month").melt(id_vars="month", var_name="asset", value_name="weight")
    long_weights["Strategy"] = strategy
    weight_rows.append(long_weights)
portfolio_weights_long = pd.concat(weight_rows, ignore_index=True) if weight_rows else pd.DataFrame()
if len(portfolio_weights_long) > 0:
    save_artifact_table(portfolio_weights_long, "portfolio_weights_long.csv")

portfolio_cost_sensitivity_rows = []
for cost_bps in TRANSACTION_COST_SENSITIVITY_BPS:
    for strategy, weights in weight_frames.items():
        metrics, _, _, _ = portfolio_metrics_from_weights(weights, holdout_returns, transaction_cost_bps=cost_bps)
        metrics.update({"Strategy": strategy, "Transaction Cost Bps": cost_bps})
        portfolio_cost_sensitivity_rows.append(metrics)
portfolio_transaction_cost_sensitivity = pd.DataFrame(portfolio_cost_sensitivity_rows)
if len(portfolio_transaction_cost_sensitivity) > 0:
    portfolio_transaction_cost_sensitivity = portfolio_transaction_cost_sensitivity.sort_values(["Transaction Cost Bps", "Sharpe", "Final Equity"], ascending=[True, False, False]).reset_index(drop=True)
    display(portfolio_transaction_cost_sensitivity.round(4))
    save_artifact_table(portfolio_transaction_cost_sensitivity, "portfolio_transaction_cost_sensitivity.csv", description="Holdout portfolio diagnostics under alternative one-way transaction cost assumptions.")

portfolio_turnover_constraint_rows = []
portfolio_turnover_constraint_monthly_returns = pd.DataFrame(index=holdout_returns.index)
for turnover_cap in TURNOVER_CONSTRAINT_CAPS:
    for strategy, weights in weight_frames.items():
        capped_weights = apply_turnover_cap(weights, turnover_cap)
        metrics, _, net_returns, turnover = portfolio_metrics_from_weights(capped_weights, holdout_returns)
        metrics.update({"Strategy": strategy, "Turnover Cap": turnover_cap, "Constraint Source": "post_score_turnover_cap"})
        portfolio_turnover_constraint_rows.append(metrics)
        portfolio_turnover_constraint_monthly_returns[f"{strategy} | turnover_cap_{turnover_cap:g}"] = net_returns
portfolio_turnover_constraint_summary = pd.DataFrame(portfolio_turnover_constraint_rows)
if len(portfolio_turnover_constraint_summary) > 0:
    portfolio_turnover_constraint_summary = portfolio_turnover_constraint_summary.sort_values(["Turnover Cap", "Sharpe", "Final Equity"], ascending=[True, False, False]).reset_index(drop=True)
    display(portfolio_turnover_constraint_summary.round(4))
    save_artifact_table(portfolio_turnover_constraint_summary, "portfolio_turnover_constraint_summary.csv", description="Holdout portfolio diagnostics after applying monthly turnover caps to target weights.")
    save_artifact_table(portfolio_turnover_constraint_monthly_returns.reset_index(names="month"), "portfolio_turnover_constraint_monthly_returns.csv")

portfolio_tax_drag_rows = []
for tax_bps in TAX_DRAG_SENSITIVITY_BPS_PER_TURNOVER:
    for strategy, weights in weight_frames.items():
        metrics, _, _, _ = portfolio_metrics_from_weights(weights, holdout_returns, transaction_cost_bps=TRANSACTION_COST_BPS + tax_bps)
        metrics.update({"Strategy": strategy, "Base Transaction Cost Bps": TRANSACTION_COST_BPS, "Tax Drag Bps Per Turnover": tax_bps})
        portfolio_tax_drag_rows.append(metrics)
portfolio_tax_drag_sensitivity = pd.DataFrame(portfolio_tax_drag_rows)
if len(portfolio_tax_drag_sensitivity) > 0:
    portfolio_tax_drag_sensitivity = portfolio_tax_drag_sensitivity.sort_values(["Tax Drag Bps Per Turnover", "Sharpe", "Final Equity"], ascending=[True, False, False]).reset_index(drop=True)
    display(portfolio_tax_drag_sensitivity.round(4))
    save_artifact_table(portfolio_tax_drag_sensitivity, "portfolio_tax_drag_sensitivity.csv", description="Illustrative tax/friction drag sensitivity expressed as additional basis points per one-way turnover.")

portfolio_weight_constraint_rows = []
for strategy, weights in weight_frames.items():
    aligned = weights.reindex(index=holdout_returns.index, columns=holdout_returns.columns).fillna(0.0)
    holdings = (aligned.abs() > 1e-8).sum(axis=1)
    leverage = aligned.abs().sum(axis=1)
    max_weight = aligned.max(axis=1)
    turnover = turnover_curves[strategy] if strategy in turnover_curves.columns else aligned.diff().abs().sum(axis=1)
    portfolio_weight_constraint_rows.append(
        {
            "Strategy": strategy,
            "Average Holdings": float(holdings.mean()),
            "Max Holdings": int(holdings.max()) if len(holdings) else 0,
            "Average Max Asset Weight": float(max_weight.mean()),
            "Max Asset Weight": float(max_weight.max()),
            "Average Gross Leverage": float(leverage.mean()),
            "Max Gross Leverage": float(leverage.max()),
            "Average Monthly Turnover": float(turnover.mean()),
            "Max Monthly Turnover": float(turnover.max()),
            "Months Above 50pct Turnover": int((turnover > 0.5).sum()),
            "Months Above 100pct Turnover": int((turnover > 1.0).sum()),
        }
    )
portfolio_weight_constraint_summary = pd.DataFrame(portfolio_weight_constraint_rows).sort_values(["Average Monthly Turnover", "Max Asset Weight"]).reset_index(drop=True)
if len(portfolio_weight_constraint_summary) > 0:
    display(portfolio_weight_constraint_summary.round(4))
    save_artifact_table(portfolio_weight_constraint_summary, "portfolio_weight_constraint_summary.csv", description="Weight, concentration, leverage, and turnover constraint diagnostics for each strategy.")


def drawdown_duration(drawdown):
    longest = 0
    current = 0
    for value in drawdown.fillna(0.0):
        if value < 0.0:
            current += 1
            longest = max(longest, current)
        else:
            current = 0
    return longest


portfolio_drawdown_rows = []
for strategy in monthly_strategy_returns.columns:
    returns = pd.to_numeric(monthly_strategy_returns[strategy], errors="coerce").dropna()
    if len(returns) == 0:
        continue
    equity = (1.0 + returns).cumprod()
    drawdown = equity / equity.cummax() - 1.0
    portfolio_drawdown_rows.append(
        {
            "Strategy": strategy,
            "Max Drawdown": float(drawdown.min()),
            "Average Drawdown": float(drawdown.mean()),
            "Ulcer Index": float(np.sqrt(np.mean(np.square(np.minimum(drawdown, 0.0))))),
            "Longest Drawdown Months": int(drawdown_duration(drawdown)),
            "Final Equity": float(equity.iloc[-1]),
        }
    )
portfolio_drawdown_summary = pd.DataFrame(portfolio_drawdown_rows).sort_values(["Max Drawdown", "Ulcer Index"], ascending=[False, True]).reset_index(drop=True)
if len(portfolio_drawdown_summary) > 0:
    display(portfolio_drawdown_summary.round(4))
    save_artifact_table(portfolio_drawdown_summary, "portfolio_drawdown_summary.csv", description="Drawdown-focused portfolio diagnostics for objectives beyond average return and Sharpe.")

all_monthly_asset_returns = returns_pivot_for_frame(model_df).reindex(columns=holdout_returns.columns).sort_index()
portfolio_ex_ante_risk_rows = []
for strategy, weights in weight_frames.items():
    aligned = weights.reindex(index=holdout_returns.index, columns=holdout_returns.columns).fillna(0.0).astype(float)
    strategy_net_returns = monthly_strategy_returns[strategy] if strategy in monthly_strategy_returns.columns else pd.Series(index=holdout_returns.index, dtype=float)
    for month, weight_row in aligned.iterrows():
        history = all_monthly_asset_returns.loc[all_monthly_asset_returns.index < month].tail(EX_ANTE_RISK_LOOKBACK_MONTHS)
        history = history.dropna(axis=1, thresh=EX_ANTE_RISK_MIN_MONTHS).dropna(how="all")
        gross_weight_sum = float(weight_row.abs().sum())
        if len(history) < EX_ANTE_RISK_MIN_MONTHS or gross_weight_sum <= 0.0:
            continue
        available_weights = weight_row.reindex(history.columns).fillna(0.0)
        available_weights = available_weights.loc[available_weights.abs() > 1e-10]
        if len(available_weights) == 0:
            continue
        history = history.reindex(columns=available_weights.index).fillna(0.0).astype(float)
        covariance = history.cov().replace([np.inf, -np.inf], np.nan).fillna(0.0)
        if covariance.empty:
            continue
        w = available_weights.reindex(covariance.index).fillna(0.0).to_numpy(dtype=float)
        covariance_values = covariance.to_numpy(dtype=float)
        monthly_variance = float(w @ covariance_values @ w)
        if monthly_variance < 0.0 and monthly_variance > -1e-12:
            monthly_variance = 0.0
        if monthly_variance < 0.0:
            continue
        ex_ante_monthly_vol = math.sqrt(monthly_variance)
        portfolio_history = history.mul(available_weights, axis=1).sum(axis=1)
        benchmark_beta = np.nan
        if BENCHMARK_ASSET in all_monthly_asset_returns.columns:
            benchmark_history = all_monthly_asset_returns.loc[history.index, BENCHMARK_ASSET]
            pair = pd.concat([portfolio_history, benchmark_history], axis=1, keys=["portfolio", "benchmark"]).dropna()
            benchmark_variance = pair["benchmark"].var(ddof=1) if len(pair) >= 3 else np.nan
            if pd.notna(benchmark_variance) and benchmark_variance > 0.0:
                benchmark_beta = pair["portfolio"].cov(pair["benchmark"]) / benchmark_variance
        realized_gross_return = float((weight_row * holdout_returns.loc[month]).sum()) if month in holdout_returns.index else np.nan
        realized_net_return = float(strategy_net_returns.loc[month]) if month in strategy_net_returns.index and pd.notna(strategy_net_returns.loc[month]) else np.nan
        portfolio_ex_ante_risk_rows.append(
            {
                "Strategy": strategy,
                "month": month,
                "Lookback Months Used": int(len(history)),
                "Configured Lookback Months": EX_ANTE_RISK_LOOKBACK_MONTHS,
                "Weight Coverage": float(available_weights.abs().sum() / gross_weight_sum),
                "Ex-Ante Monthly Volatility": float(ex_ante_monthly_vol),
                "Ex-Ante Annualized Volatility": float(ex_ante_monthly_vol * math.sqrt(MONTHS_PER_YEAR)),
                "Ex-Ante 95pct Monthly VaR Proxy": float(1.645 * ex_ante_monthly_vol),
                "Ex-Ante Beta To Benchmark": float(benchmark_beta) if np.isfinite(benchmark_beta) else np.nan,
                "Realized Gross Return": realized_gross_return,
                "Realized Net Return": realized_net_return,
            }
        )

portfolio_ex_ante_risk_monthly = pd.DataFrame(portfolio_ex_ante_risk_rows)
if len(portfolio_ex_ante_risk_monthly) > 0:
    risk_summary_rows = []
    for strategy, group in portfolio_ex_ante_risk_monthly.groupby("Strategy", sort=False):
        predicted_ann_vol = pd.to_numeric(group["Ex-Ante Annualized Volatility"], errors="coerce")
        realized_net = pd.to_numeric(group["Realized Net Return"], errors="coerce").dropna()
        realized_ann_vol = float(realized_net.std(ddof=1) * math.sqrt(MONTHS_PER_YEAR)) if len(realized_net) >= 2 else np.nan
        average_predicted_ann_vol = float(predicted_ann_vol.mean())
        risk_summary_rows.append(
            {
                "Strategy": strategy,
                "Risk Observation Months": int(group["month"].nunique()),
                "Average Ex-Ante Annualized Volatility": average_predicted_ann_vol,
                "Median Ex-Ante Annualized Volatility": float(predicted_ann_vol.median()),
                "Max Ex-Ante Annualized Volatility": float(predicted_ann_vol.max()),
                "Average Ex-Ante Monthly VaR Proxy": float(pd.to_numeric(group["Ex-Ante 95pct Monthly VaR Proxy"], errors="coerce").mean()),
                "Average Ex-Ante Beta To Benchmark": float(pd.to_numeric(group["Ex-Ante Beta To Benchmark"], errors="coerce").mean()),
                "Average Weight Coverage": float(pd.to_numeric(group["Weight Coverage"], errors="coerce").mean()),
                "Realized Net Annualized Volatility": realized_ann_vol,
                "Realized/Predicted Vol Ratio": float(realized_ann_vol / average_predicted_ann_vol) if average_predicted_ann_vol > 0 and np.isfinite(realized_ann_vol) else np.nan,
                "Risk Model Caveat": "trailing covariance proxy using downloaded monthly returns; not a production ex-ante risk model",
            }
        )
    portfolio_ex_ante_risk_summary = pd.DataFrame(risk_summary_rows).sort_values(["Average Ex-Ante Annualized Volatility", "Strategy"]).reset_index(drop=True)
    display(portfolio_ex_ante_risk_summary.round(4))
    save_artifact_table(portfolio_ex_ante_risk_monthly, "portfolio_ex_ante_risk_monthly.csv", description="Monthly trailing-covariance ex-ante risk proxy for headline portfolio weights.")
    save_artifact_table(portfolio_ex_ante_risk_summary, "portfolio_ex_ante_risk_summary.csv", description="Summary of trailing-covariance ex-ante volatility, VaR, beta, and realized/predicted volatility diagnostics.")

benchmark_strategy_candidates = [strategy for strategy in ["SPY only", "Equal weight universe", "60/40 SPY/TLT"] if strategy in monthly_strategy_returns.columns]
benchmark_relative_rows = []
for benchmark_strategy in benchmark_strategy_candidates:
    benchmark_returns = pd.to_numeric(monthly_strategy_returns[benchmark_strategy], errors="coerce")
    for strategy in monthly_strategy_returns.columns:
        if strategy == benchmark_strategy:
            continue
        strategy_returns = pd.to_numeric(monthly_strategy_returns[strategy], errors="coerce")
        pair = pd.concat([strategy_returns, benchmark_returns], axis=1, keys=["strategy", "benchmark"]).dropna()
        if len(pair) < 3:
            continue
        active = pair["strategy"] - pair["benchmark"]
        tracking_error = active.std(ddof=1) * math.sqrt(MONTHS_PER_YEAR)
        annual_active_return = active.mean() * MONTHS_PER_YEAR
        benchmark_variance = pair["benchmark"].var(ddof=1)
        beta = pair["strategy"].cov(pair["benchmark"]) / benchmark_variance if benchmark_variance > 0 else np.nan
        benchmark_relative_rows.append(
            {
                "Strategy": strategy,
                "Benchmark Strategy": benchmark_strategy,
                "Annualized Active Return": float(annual_active_return),
                "Tracking Error": float(tracking_error),
                "Information Ratio": float(annual_active_return / tracking_error) if tracking_error > 0 else np.nan,
                "Correlation": float(pair["strategy"].corr(pair["benchmark"])),
                "Beta": float(beta) if np.isfinite(beta) else np.nan,
                "Monthly Win Rate vs Benchmark": float((active > 0).mean()),
                "Worst Monthly Active Return": float(active.min()),
            }
        )
portfolio_benchmark_relative_summary = pd.DataFrame(benchmark_relative_rows)
if len(portfolio_benchmark_relative_summary) > 0:
    portfolio_benchmark_relative_summary = portfolio_benchmark_relative_summary.sort_values(["Benchmark Strategy", "Information Ratio"], ascending=[True, False]).reset_index(drop=True)
    display(portfolio_benchmark_relative_summary.round(4))
    save_artifact_table(portfolio_benchmark_relative_summary, "portfolio_benchmark_relative_summary.csv", description="Benchmark-relative active return, tracking error, information ratio, correlation, and beta diagnostics.")

liquidity_columns = ["month", "asset", "asset_avg_dollar_volume_63d", "asset_hl_spread_proxy_21d"]
liquidity_available = all(column in holdout_df.columns for column in liquidity_columns)
liquidity_signal_available = liquidity_available and holdout_df["asset_avg_dollar_volume_63d"].notna().any() and holdout_df["asset_hl_spread_proxy_21d"].notna().any()
if liquidity_signal_available:
    adv = holdout_df.pivot_table(index="month", columns="asset", values="asset_avg_dollar_volume_63d", aggfunc="first").reindex(index=holdout_returns.index, columns=holdout_returns.columns)
    spread = holdout_df.pivot_table(index="month", columns="asset", values="asset_hl_spread_proxy_21d", aggfunc="first").reindex(index=holdout_returns.index, columns=holdout_returns.columns)
    liquidity_rows = []
    for strategy, weights in weight_frames.items():
        aligned = weights.reindex(index=holdout_returns.index, columns=holdout_returns.columns).fillna(0.0)
        trade_weights = aligned.diff().abs()
        if len(trade_weights) > 0:
            trade_weights.iloc[0] = aligned.iloc[0].abs()
        weighted_adv = (aligned.abs() * adv).replace([np.inf, -np.inf], np.nan).sum(axis=1, min_count=1)
        weighted_spread_bps = (aligned.abs() * spread * 10000.0).replace([np.inf, -np.inf], np.nan).sum(axis=1, min_count=1)
        trade_notional = trade_weights * MARKET_IMPACT_NOTIONAL_USD
        participation = (trade_notional / adv.replace(0.0, np.nan)).replace([np.inf, -np.inf], np.nan)
        trade_weight_sum = trade_weights.sum(axis=1).replace(0.0, np.nan)
        turnover_weighted_participation_bps = ((trade_weights * participation).replace([np.inf, -np.inf], np.nan).sum(axis=1, min_count=1) / trade_weight_sum) * 10000.0
        liquidity_rows.append(
            {
                "Strategy": strategy,
                "Average Weighted ADV USD": float(weighted_adv.mean()),
                "Minimum Weighted ADV USD": float(weighted_adv.min()),
                "Average Weighted Spread Proxy Bps": float(weighted_spread_bps.mean()),
                "Max Weighted Spread Proxy Bps": float(weighted_spread_bps.max()),
                "Average Turnover-Weighted Participation Bps": float(turnover_weighted_participation_bps.mean()),
                "Max Turnover-Weighted Participation Bps": float(turnover_weighted_participation_bps.max()),
                "Impact Proxy Notional USD": MARKET_IMPACT_NOTIONAL_USD,
            }
        )
    portfolio_liquidity_impact_proxy = pd.DataFrame(liquidity_rows).sort_values("Average Turnover-Weighted Participation Bps").reset_index(drop=True)
    display(portfolio_liquidity_impact_proxy.round(4))
    save_artifact_table(portfolio_liquidity_impact_proxy, "portfolio_liquidity_impact_proxy.csv", description="Liquidity, spread, and market-impact proxy diagnostics using ADV and high-low spread proxies; not a production execution model.")
else:
    portfolio_liquidity_impact_proxy = pd.DataFrame(
        [
            {
                "Status": "unavailable",
                "Reason": "rolling dollar-volume or high-low spread proxy inputs are missing in holdout_df",
                "ADV Non-Null Rows": int(holdout_df["asset_avg_dollar_volume_63d"].notna().sum()) if "asset_avg_dollar_volume_63d" in holdout_df.columns else 0,
                "Spread Proxy Non-Null Rows": int(holdout_df["asset_hl_spread_proxy_21d"].notna().sum()) if "asset_hl_spread_proxy_21d" in holdout_df.columns else 0,
            }
        ]
    )
    display(portfolio_liquidity_impact_proxy)
    save_artifact_table(portfolio_liquidity_impact_proxy, "portfolio_liquidity_impact_proxy.csv", description="Liquidity proxy availability status.")

# Precision-recall curves.
publication_holdout = selected_publication_rows(allocation_summary, "holdout")
if len(publication_holdout) > 0:
    fig, ax = plt.subplots(figsize=(8, 5.5))
    y_true = y_to_numpy(y_holdout)
    base_rate = y_true.mean()
    for _, row in publication_holdout.iterrows():
        model_name = row["Model"]
        score = predictions.get((model_name, "holdout"))
        if score is None:
            continue
        precision, recall, _ = precision_recall_curve(y_true, np.asarray(score, dtype=float))
        ax.plot(recall, precision, linewidth=1.8, label=f"{model_display_name(model_name)} (AP {row['Average Precision']:.3f})")
    ax.axhline(base_rate, color="gray", linestyle="--", linewidth=1, label=f"Base rate {base_rate:.3f}")
    ax.set_title("Holdout Precision-Recall Curves")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8, frameon=False)
    ax.grid(alpha=0.25)
    fig.subplots_adjust(right=0.62)
    output_path = ARTIFACT_DIR / "publication_precision_recall_curves_holdout.png"
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    register_artifact(output_path, "figure", "Holdout precision-recall curves for selected models.")
    print(f"Saved {output_path}")

# Runtime versus AP.
runtime_ap_summary = allocation_summary.loc[
    allocation_summary["Average Precision"].notna(),
    ["Model", "Base Model", "Calibration", "Evaluation Window", "Average Precision", "ROC AUC", "Brier Score", "Workflow Seconds"],
].sort_values(["Evaluation Window", "Average Precision"], ascending=[True, False]).reset_index(drop=True)
display(runtime_ap_summary.round(4))
save_artifact_table(runtime_ap_summary, "runtime_ap_summary.csv")

holdout_runtime = selected_publication_rows(allocation_summary, "holdout")
holdout_runtime = holdout_runtime.loc[holdout_runtime["Average Precision"].notna()].copy()
if len(holdout_runtime) > 0:
    fig, ax = plt.subplots(figsize=(8.5, 5.5))
    for _, row in holdout_runtime.iterrows():
        label = f"{model_display_name(row['Model'])} ({row['Workflow Seconds']:.1f}s, AP {row['Average Precision']:.3f})"
        ax.scatter(row["Workflow Seconds"], row["Average Precision"], s=60, label=label)
    ax.set_xscale("symlog", linthresh=1.0)
    ax.set_title("Workflow Time versus Holdout Average Precision")
    ax.set_xlabel("Workflow seconds, symlog scale")
    ax.set_ylabel("Average Precision")
    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8, frameon=False)
    ax.grid(alpha=0.25)
    fig.subplots_adjust(right=0.62)
    output_path = ARTIFACT_DIR / "publication_runtime_vs_average_precision_holdout.png"
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    register_artifact(output_path, "figure", "Runtime versus holdout Average Precision for publication models.")
    print(f"Saved {output_path}")

# Portfolio equity curves.
if len(equity_curves.columns) > 0:
    selected_equity_candidates = [
        "Equal weight universe",
        "SPY only",
        "60/40 SPY/TLT",
        "12M momentum top-k portfolio",
        "XGBoost[GPU allocation scorer]",
        "XGBoost[GPU allocation scorer] Calibrated",
        "TabPFN[Direct allocation scorer]",
        "TabICL[Direct allocation scorer]",
    ]
    selected_equity_columns = [column for column in selected_equity_candidates if column in equity_curves.columns]
    if not selected_equity_columns:
        selected_equity_columns = list(equity_curves.columns[:8])
    fig, ax = plt.subplots(figsize=(9.5, 5.5))
    plot_index = month_index_to_timestamp(equity_curves.index)
    for column in selected_equity_columns:
        ax.plot(plot_index, equity_curves[column], linewidth=1.8, label=model_display_name(column))
    ax.set_title("Holdout Tactical Allocation Diagnostic")
    ax.set_xlabel("Year")
    ax.set_ylabel("Growth of 1.0 before taxes")
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.xaxis.set_minor_locator(mdates.MonthLocator(interval=3))
    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8, frameon=False)
    ax.grid(alpha=0.25)
    fig.subplots_adjust(right=0.60, bottom=0.16)
    output_path = ARTIFACT_DIR / "portfolio_equity_curves.png"
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    register_artifact(output_path, "figure", "Holdout portfolio equity curves for selected strategies.")
    print(f"Saved {output_path}")

# Calibration diagnostics.
calibration_quality_view = allocation_summary.loc[
    (allocation_summary["Score Type"] == "probability") & allocation_summary["Average Precision"].notna(),
    ["Model", "Family", "Calibration", "Evaluation Window", "Average Precision", "Brier Score", "Log Loss", "ECE Quantile 10"],
].sort_values(["Evaluation Window", "Average Precision"], ascending=[True, False]).reset_index(drop=True)
display(calibration_quality_view.round(4))
save_artifact_table(calibration_quality_view, "calibration_quality_view.csv")

reliability_rows = []
for (model_name, evaluation_window), score in predictions.items():
    if evaluation_window not in {"calibration", "holdout"}:
        continue
    frame = calibration_df if evaluation_window == "calibration" else holdout_df
    y_true = y_calibration if evaluation_window == "calibration" else y_holdout
    if model_name not in allocation_summary.loc[allocation_summary["Score Type"] == "probability", "Model"].values:
        continue
    table = calibration_bin_table(y_true, score, n_bins=10)
    table.insert(0, "Evaluation Window", evaluation_window)
    table.insert(0, "Model", model_name)
    reliability_rows.append(table)
reliability_summary = pd.concat(reliability_rows, ignore_index=True) if reliability_rows else pd.DataFrame()
if len(reliability_summary) > 0:
    save_artifact_table(reliability_summary, "reliability_summary.csv")

calibration_models = [name for name in CALIBRATION_MODEL_ORDER if (name, "holdout") in predictions]
if calibration_models:
    calibration_points = []
    for model_name in calibration_models:
        score = predictions.get((model_name, "holdout"))
        if score is None:
            continue
        prob_true, prob_pred = calibration_curve(y_to_numpy(y_holdout), np.clip(score, 0.0, 1.0), n_bins=10, strategy="quantile")
        calibration_points.append((model_name, prob_pred, prob_true))
    if calibration_points:
        all_values = np.concatenate([np.asarray(values, dtype=float) for _, prob_pred, prob_true in calibration_points for values in [prob_pred, prob_true]])
        axis_min = max(0.0, float(np.nanmin(all_values)) - 0.05)
        axis_max = min(1.0, float(np.nanmax(all_values)) + 0.05)
        if axis_max - axis_min < 0.25:
            midpoint = (axis_min + axis_max) / 2.0
            axis_min = max(0.0, midpoint - 0.125)
            axis_max = min(1.0, midpoint + 0.125)
        fig, ax = plt.subplots(figsize=(7.5, 5.5))
        for model_name, prob_pred, prob_true in calibration_points:
            ax.plot(prob_pred, prob_true, marker="o", linewidth=1.8, label=model_display_name(model_name))
        ax.plot([axis_min, axis_max], [axis_min, axis_max], color="gray", linestyle="--", linewidth=1, label="Perfect calibration")
        ax.set_xlim(axis_min, axis_max)
        ax.set_ylim(axis_min, axis_max)
        ax.set_title("Holdout Calibration Diagnostics")
        ax.set_xlabel("Mean predicted probability")
        ax.set_ylabel("Observed top-k frequency")
        ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8, frameon=False)
        ax.grid(alpha=0.25)
        fig.subplots_adjust(right=0.62)
        output_path = ARTIFACT_DIR / "publication_calibration_curves_holdout.png"
        fig.savefig(output_path, dpi=180, bbox_inches="tight")
        display(fig)
        plt.close(fig)
        register_artifact(output_path, "figure", "Holdout calibration curves for selected probability models.")
        print(f"Saved {output_path}")

# Month-block bootstrap uncertainty for row metrics and portfolio returns.
def bootstrap_quantiles(values, confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan, np.nan
    alpha = 1.0 - confidence_level
    lower, median, upper = np.quantile(values, [alpha / 2.0, 0.5, 1.0 - alpha / 2.0])
    return lower, median, upper


def portfolio_return_point_metrics(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return {"mean_monthly_return": np.nan, "cagr": np.nan, "annualized_volatility": np.nan, "sharpe": np.nan}
    years = len(values) / MONTHS_PER_YEAR
    ending_value = float(np.prod(1.0 + values))
    cagr = ending_value ** (1.0 / years) - 1.0 if years > 0 and ending_value > 0 else np.nan
    volatility = float(np.std(values, ddof=1) * math.sqrt(MONTHS_PER_YEAR)) if len(values) > 1 else np.nan
    sharpe = float(np.mean(values) / np.std(values, ddof=1) * math.sqrt(MONTHS_PER_YEAR)) if len(values) > 1 and np.std(values, ddof=1) > 0 else np.nan
    return {
        "mean_monthly_return": float(np.mean(values)),
        "cagr": cagr,
        "annualized_volatility": volatility,
        "sharpe": sharpe,
    }


def bootstrap_portfolio_return_metrics(monthly_returns, n_iterations=BOOTSTRAP_ITERATIONS):
    if n_iterations <= 0 or len(monthly_returns) == 0:
        return pd.DataFrame()
    rng = np.random.default_rng(SEED + 17)
    rows = []
    for strategy in tqdm(list(monthly_returns.columns), desc="Bootstrap portfolio returns", unit="strategy"):
        values = pd.to_numeric(monthly_returns[strategy], errors="coerce").dropna().to_numpy(dtype=float)
        if len(values) < 3:
            continue
        point_metrics = portfolio_return_point_metrics(values)
        boot_metrics = {name: [] for name in point_metrics}
        for _ in range(n_iterations):
            sampled = rng.choice(values, size=len(values), replace=True)
            sampled_metrics = portfolio_return_point_metrics(sampled)
            for metric_name, metric_value in sampled_metrics.items():
                boot_metrics[metric_name].append(metric_value)
        for metric_name, metric_values in boot_metrics.items():
            lower, median, upper = bootstrap_quantiles(metric_values)
            rows.append(
                {
                    "Strategy": strategy,
                    "Metric": metric_name,
                    "Point Estimate": point_metrics.get(metric_name, np.nan),
                    "Bootstrap Median": median,
                    "CI Lower": lower,
                    "CI Upper": upper,
                    "Bootstrap Iterations Requested": n_iterations,
                    "Bootstrap Iterations Used": len([value for value in metric_values if np.isfinite(value)]),
                    "Confidence Level": BOOTSTRAP_CONFIDENCE_LEVEL,
                }
            )
    return pd.DataFrame(rows)


publication_portfolio_bootstrap_uncertainty = bootstrap_portfolio_return_metrics(monthly_strategy_returns, n_iterations=BOOTSTRAP_ITERATIONS)
if len(publication_portfolio_bootstrap_uncertainty) > 0:
    display(publication_portfolio_bootstrap_uncertainty.round(4))
    save_artifact_table(publication_portfolio_bootstrap_uncertainty, "publication_portfolio_month_bootstrap_uncertainty.csv")
cleanup_runtime_memory("after_portfolio_bootstrap")


def _rankdata_average(values):
    values = np.asarray(values, dtype=np.float64)
    order = np.argsort(values, kind="mergesort")
    ranks = np.empty(len(values), dtype=np.float64)
    sorted_values = values[order]
    start = 0
    while start < len(values):
        stop = start + 1
        while stop < len(values) and sorted_values[stop] == sorted_values[start]:
            stop += 1
        rank = 0.5 * (start + 1 + stop)
        ranks[order[start:stop]] = rank
        start = stop
    return ranks


def _spearman_corr_fast(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    mask = np.isfinite(a) & np.isfinite(b)
    if mask.sum() < 2:
        return np.nan
    a = a[mask]
    b = b[mask]
    if np.unique(a).size <= 1 or np.unique(b).size <= 1:
        return np.nan
    ra = _rankdata_average(a)
    rb = _rankdata_average(b)
    ra = ra - ra.mean()
    rb = rb - rb.mean()
    denom = np.sqrt(np.sum(ra * ra) * np.sum(rb * rb))
    return float(np.sum(ra * rb) / denom) if denom > 0 else np.nan


def _monthly_rank_arrays(y_values, score_values, return_values, month_codes, top_k=TARGET_TOP_K):
    ic_values = []
    hit_values = []
    for month_code in np.unique(month_codes):
        idx = np.flatnonzero(month_codes == month_code)
        if len(idx) < max(2, top_k):
            continue
        scores = score_values[idx]
        realized = return_values[idx]
        targets = y_values[idx]
        valid = np.isfinite(scores) & np.isfinite(realized)
        if valid.sum() < max(2, top_k):
            continue
        scores = scores[valid]
        realized = realized[valid]
        targets = targets[valid]
        if np.unique(scores).size <= 1:
            continue
        top_idx = np.argsort(-scores, kind="mergesort")[:top_k]
        ic_values.append(_spearman_corr_fast(scores, realized))
        hit_values.append(float(np.mean(targets[top_idx])))
    return (
        float(np.nanmean(ic_values)) if len(ic_values) else np.nan,
        float(np.nanmean(hit_values)) if len(hit_values) else np.nan,
    )


def bootstrap_holdout_row_metrics(score_frame, n_iterations=BOOTSTRAP_ITERATIONS):
    if n_iterations <= 0 or len(score_frame) == 0:
        return pd.DataFrame()
    rng = np.random.default_rng(SEED)
    rows = []
    needed = ["Model", "month", "target_top_k_next_1m", "score", "forward_1m_return"]
    compact = score_frame[needed].copy()
    compact["target_top_k_next_1m"] = compact["target_top_k_next_1m"].astype(np.int8)
    compact["score"] = pd.to_numeric(compact["score"], errors="coerce").astype(np.float32)
    compact["forward_1m_return"] = pd.to_numeric(compact["forward_1m_return"], errors="coerce").astype(np.float32)
    for model_name, model_group in tqdm(list(compact.groupby("Model", sort=False)), desc="Bootstrap row metrics", unit="model"):
        month_values = model_group["month"].astype(str).to_numpy()
        unique_months = pd.unique(month_values)
        month_to_code = {month: idx for idx, month in enumerate(unique_months)}
        month_codes = np.asarray([month_to_code[month] for month in month_values], dtype=np.int16)
        month_indices = [np.flatnonzero(month_codes == idx) for idx in range(len(unique_months))]
        y_values = model_group["target_top_k_next_1m"].to_numpy(dtype=np.int8, copy=False)
        score_values = model_group["score"].to_numpy(dtype=np.float32, copy=False)
        return_values = model_group["forward_1m_return"].to_numpy(dtype=np.float32, copy=False)
        metric_values = {"average_precision": [], "mean_spearman_ic": [], "top_k_hit_rate": []}
        for _ in range(n_iterations):
            sampled_month_codes = rng.integers(0, len(unique_months), size=len(unique_months))
            sampled_idx = np.concatenate([month_indices[code] for code in sampled_month_codes])
            if sampled_idx.size == 0:
                continue
            y_sample = y_values[sampled_idx]
            if np.unique(y_sample).size < 2:
                continue
            score_sample = score_values[sampled_idx]
            return_sample = return_values[sampled_idx]
            block_codes = np.concatenate([np.repeat(block_id, len(month_indices[code])) for block_id, code in enumerate(sampled_month_codes)]).astype(np.int16)
            metric_values["average_precision"].append(float(average_precision_score(y_sample, score_sample)))
            mean_ic, mean_hit = _monthly_rank_arrays(y_sample, score_sample, return_sample, block_codes, top_k=TARGET_TOP_K)
            metric_values["mean_spearman_ic"].append(mean_ic)
            metric_values["top_k_hit_rate"].append(mean_hit)
        point_monthly = monthly_rank_quality(model_group.assign(**{"Evaluation Window": "holdout"}), top_k=TARGET_TOP_K)
        point_estimates = {
            "average_precision": safe_metric(average_precision_score, y_values, score_values),
            "mean_spearman_ic": point_monthly["spearman_ic"].mean() if len(point_monthly) else np.nan,
            "top_k_hit_rate": point_monthly["top_k_hit_rate"].mean() if len(point_monthly) else np.nan,
        }
        for metric_name, values in metric_values.items():
            lower, median, upper = bootstrap_quantiles(values)
            rows.append({"Model": model_name, "Metric": metric_name, "Point Estimate": point_estimates.get(metric_name, np.nan), "Bootstrap Median": median, "CI Lower": lower, "CI Upper": upper, "Bootstrap Iterations Requested": n_iterations, "Bootstrap Iterations Used": len([value for value in values if np.isfinite(value)]), "Confidence Level": BOOTSTRAP_CONFIDENCE_LEVEL})
    return pd.DataFrame(rows)

publication_bootstrap_uncertainty = bootstrap_holdout_row_metrics(selected_holdout_predictions, n_iterations=BOOTSTRAP_ITERATIONS)
if len(publication_bootstrap_uncertainty) > 0:
    display(publication_bootstrap_uncertainty.round(4))
    save_artifact_table(publication_bootstrap_uncertainty, "publication_month_block_bootstrap_uncertainty.csv")
cleanup_runtime_memory("after_row_bootstrap")

if len(allocation_summary) > 0:
    publication_holdout_table = selected_publication_rows(allocation_summary, "holdout")
    save_text_artifact("publication_holdout_summary.txt", publication_holdout_table[performance_columns].round(6).to_string(index=False), description="Plain-text holdout model summary for headline models only; feature-sensitivity diagnostics are saved separately.")
if len(portfolio_summary) > 0:
    save_text_artifact("publication_portfolio_summary.txt", portfolio_summary.round(6).to_string(index=False), description="Plain-text headline portfolio diagnostic summary. Feature-sensitivity portfolios are excluded and saved separately.")
if len(monthly_rank_aggregate) > 0:
    publication_monthly_rank_aggregate = exclude_feature_sensitivity(monthly_rank_aggregate)
    save_text_artifact("publication_monthly_rank_aggregate.txt", publication_monthly_rank_aggregate.round(6).to_string(index=False), description="Plain-text monthly ranking diagnostics for headline models only.")


,Model,Base Model,Calibration,Evaluation Window,Rows,Positive Rate,Score Type,Average Precision,ROC AUC,Brier Score,Log Loss,ECE Quantile 10,CV/Validation Average Precision,Workflow Seconds
0,XGBoost[GPU allocation scorer] Calibration Base,XGBoost,none_calibration_base,calibration,240,0.3,probability,0.3665,0.5604,0.2435,0.7976,0.1832,0.3542,8.5984
1,XGBoost[GPU allocation scorer] Calibrated,XGBoost,sigmoid,calibration,240,0.3,probability,0.3665,0.5604,0.2083,0.6070,0.0585,0.3542,8.5847
2,Rule[Low volatility top-k],Rule,none,calibration,240,0.3,raw_score,0.3253,0.4822,NaN,NaN,NaN,NaN,0.0000
3,TabPFN[Direct allocation scorer] Calibration Base,TabPFN,none_calibration_base,calibration,240,0.3,probability,0.3119,0.5291,0.2109,0.6146,0.0730,NaN,4.6526
4,Rule[12M momentum top-k],Rule,none,calibration,240,0.3,raw_score,0.2657,0.4519,NaN,NaN,NaN,NaN,0.0000
5,Rule[6M momentum top-k],Rule,none,calibration,240,0.3,raw_score,0.2406,0.3915,NaN,NaN,NaN,NaN,0.0000
6,Rule[Risk-adjusted momentum top-k],Rule,none,calibration,240,0.3,raw_score,0.2405,0.3854,NaN,NaN,NaN,NaN,0.0000
11,TabPFN[Direct allocation scorer],TabPFN,none,holdout,750,0.3,probability,0.3650,0.5829,0.2063,0.6029,0.0488,NaN,5.8209
14,Rule[Risk-adjusted momentum top-k],Rule,none,holdout,750,0.3,raw_score,0.3420,0.5289,NaN,NaN,NaN,NaN,0.0000
15,Rule[12M momentum top-k],Rule,none,holdout,750,0.3,raw_score,0.3412,0.5383,NaN,NaN,NaN,NaN,0.0000


Saved tabpfn_tabicl_tactical_asset_allocation_20260512_v2_outputs/headline_allocation_summary.csv


,Model,Base Model,Family,Calibration,Evaluation Window,Rows,Positive Rows,Positive Rate,Score Type,Average Precision,...,Brier Score,Log Loss,ECE Quantile 10,Fit Seconds,Predict Seconds,Calibration Seconds,Workflow Seconds,CV/Validation Average Precision,Best Params,Timing Notes
7,FeatureSensitivity[full][TabPFN direct],TabPFN,feature_policy_sensitivity,full,holdout,750,225,0.3,probability,0.3748,...,0.2065,0.6031,0.0456,1.8861,4.2061,0.0,6.0922,NaN,,direct TabPFN feature-policy sensitivity rerun
8,FeatureSensitivity[ticker_ablated][TabPFN direct],TabPFN,feature_policy_sensitivity,ticker_ablated,holdout,750,225,0.3,probability,0.3743,...,0.2064,0.6028,0.0690,1.7218,4.1135,0.0,5.8353,NaN,,direct TabPFN feature-policy sensitivity rerun
9,FeatureSensitivity[identity_ablated][TabPFN di...,TabPFN,feature_policy_sensitivity,identity_ablated,holdout,750,225,0.3,probability,0.3650,...,0.2063,0.6029,0.0488,1.4779,4.0884,0.0,5.5663,NaN,,direct TabPFN feature-policy sensitivity rerun
10,FeatureSensitivity[strict_time_series][TabPFN ...,TabPFN,feature_policy_sensitivity,strict_time_series,holdout,750,225,0.3,probability,0.3650,...,0.2063,0.6029,0.0488,1.7568,4.0345,0.0,5.7913,NaN,,direct TabPFN feature-policy sensitivity rerun
12,FeatureSensitivity[metadata_only][TabPFN direct],TabPFN,feature_policy_sensitivity,metadata_only,holdout,750,225,0.3,probability,0.3570,...,0.2065,0.6027,0.0443,0.3431,0.2209,0.0,0.5640,NaN,,direct TabPFN feature-policy sensitivity rerun
13,FeatureSensitivity[metadata_only][XGBoost light],XGBoost,feature_policy_sensitivity,metadata_only,holdout,750,225,0.3,probability,0.3551,...,0.2437,0.6797,0.1923,0.3929,0.0219,0.0,0.4149,NaN,,fixed GPU XGBoost feature-policy sensitivity r...
18,FeatureSensitivity[full][XGBoost light],XGBoost,feature_policy_sensitivity,full,holdout,750,225,0.3,probability,0.3285,...,0.2270,0.6474,0.1209,3.2938,0.1688,0.0,3.4626,NaN,,fixed GPU XGBoost feature-policy sensitivity r...
19,FeatureSensitivity[identity_ablated][XGBoost l...,XGBoost,feature_policy_sensitivity,identity_ablated,holdout,750,225,0.3,probability,0.3258,...,0.2275,0.6491,0.1195,0.7616,0.0246,0.0,0.7862,NaN,,fixed GPU XGBoost feature-policy sensitivity r...
20,FeatureSensitivity[strict_time_series][XGBoost...,XGBoost,feature_policy_sensitivity,strict_time_series,holdout,750,225,0.3,probability,0.3258,...,0.2275,0.6491,0.1195,0.7519,0.0250,0.0,0.7769,NaN,,fixed GPU XGBoost feature-policy sensitivity r...
21,FeatureSensitivity[ticker_ablated][XGBoost light],XGBoost,feature_policy_sensitivity,ticker_ablated,holdout,750,225,0.3,probability,0.3236,...,0.2280,0.6491,0.1286,0.7697,0.0252,0.0,0.7949,NaN,,fixed GPU XGBoost feature-policy sensitivity r...


Saved tabpfn_tabicl_tactical_asset_allocation_20260512_v2_outputs/feature_sensitivity_allocation_summary.csv


,Model,Evaluation Window,months,mean_spearman_ic,median_spearman_ic,mean_top_k_hit_rate,mean_selected_forward_return,mean_universe_forward_return,mean_active_forward_return
23,XGBoost[GPU allocation scorer] Calibrated,calibration,24,0.0869,0.1333,0.3472,0.0094,0.0042,0.0052
25,XGBoost[GPU allocation scorer] Calibration Base,calibration,24,0.0869,0.1333,0.3472,0.0094,0.0042,0.0052
15,Rule[Low volatility top-k],calibration,24,-0.0369,0.0000,0.3056,0.0049,0.0042,0.0008
20,TabPFN[Direct allocation scorer] Calibration Base,calibration,24,-0.0519,-0.1033,0.2917,0.0020,0.0042,-0.0022
11,Rule[12M momentum top-k],calibration,24,-0.1111,-0.1636,0.2917,0.0007,0.0042,-0.0035
17,Rule[Risk-adjusted momentum top-k],calibration,24,-0.1646,-0.2545,0.2222,-0.0026,0.0042,-0.0068
13,Rule[6M momentum top-k],calibration,24,-0.1707,-0.1273,0.1806,-0.0067,0.0042,-0.0109
21,TabPFN[Direct allocation scorer] Calibration Base,holdout,75,0.0275,0.0424,0.3644,0.0166,0.0101,0.0066
1,FeatureSensitivity[full][TabPFN direct],holdout,75,0.0682,0.0909,0.3867,0.0162,0.0101,0.0062
18,Rule[Risk-adjusted momentum top-k],holdout,75,0.0227,0.0061,0.3333,0.0147,0.0101,0.0047


...

,Strategy,Average Holdings,Max Holdings,Average Max Asset Weight,Max Asset Weight,Average Gross Leverage,Max Gross Leverage,Average Monthly Turnover,Max Monthly Turnover,Months Above 50pct Turnover,Months Above 100pct Turnover
0,Equal weight universe,10.0,10,0.1000,0.1000,1.0,1.0,0.0133,1.0000,1,0
1,60/40 SPY/TLT,2.0,2,0.6000,0.6000,1.0,1.0,0.0133,1.0000,1,0
2,SPY only,1.0,1,1.0000,1.0000,1.0,1.0,0.0133,1.0000,1,0
3,Inverse volatility universe,10.0,10,0.2392,0.4189,1.0,1.0,0.0869,1.0000,1,0
4,Low volatility top-k portfolio,3.0,3,0.3333,0.3333,1.0,1.0,0.3511,1.3333,35,4
5,12M momentum top-k portfolio,3.0,3,0.3333,0.3333,1.0,1.0,0.3956,1.3333,43,1
6,Rule[6M momentum top-k],3.0,3,0.3333,0.3333,1.0,1.0,0.4756,1.3333,46,7
7,Rule[6M momentum top-k] | inverse-vol capped t...,3.0,3,0.3497,0.3500,1.0,1.0,0.4890,1.4000,46,7
8,TabPFN[Direct allocation scorer] Calibration Base,3.0,3,0.3333,0.3333,1.0,1.0,0.5289,1.3333,52,7
9,TabPFN[Direct allocation scorer] Calibration B...,3.0,3,0.3499,0.3500,1.0,1.0,0.5498,1.4000,52,7


Saved tabpfn_tabicl_tactical_asset_allocation_20260512_v2_outputs/portfolio_weight_constraint_summary.csv


,Strategy,Max Drawdown,Average Drawdown,Ulcer Index,Longest Drawdown Months,Final Equity
0,Rule[Risk-adjusted momentum top-k],-0.1396,-0.0296,0.0471,20,2.7463
1,Rule[Risk-adjusted momentum top-k] | inverse-v...,-0.1399,-0.0305,0.0489,20,2.6992
2,12M momentum top-k portfolio,-0.1475,-0.0206,0.0378,9,2.4728
3,Rule[6M momentum top-k],-0.1530,-0.0354,0.0533,31,2.1671
4,Rule[6M momentum top-k] | inverse-vol capped t...,-0.1623,-0.0382,0.0585,27,2.1605
5,Equal weight universe,-0.1781,-0.0287,0.0511,25,1.9923
6,Inverse volatility universe,-0.1848,-0.0324,0.0563,25,1.6323
7,Low volatility top-k portfolio,-0.2031,-0.0399,0.0665,25,1.4751
8,XGBoost[GPU allocation scorer] | inverse-vol c...,-0.2196,-0.0569,0.0860,28,1.7368
9,XGBoost[GPU allocation scorer],-0.2202,-0.0577,0.0870,28,1.7247


Saved tabpfn_tabicl_tactical_asset_allocation_20260512_v2_outputs/portfolio_drawdown_summary.csv


NameError: name 'model_df' is not defined

## 7. Drift, Leakage, and Reuse Checklist

This checklist records checks that can be performed with public data and the artifacts generated by this notebook. It explicitly records which shortcomings are addressed as diagnostics, which are only stress-tested, and which remain caveats. It does not certify live tradability or point-in-time vendor correctness. A production workflow would need stricter data lineage, corporate-action, execution-cost, tax, liquidity, and mandate-constraint reviews.

In [ ]:
def population_stability_index(expected, actual, n_bins=10):
    expected = pd.Series(expected).replace([np.inf, -np.inf], np.nan).dropna()
    actual = pd.Series(actual).replace([np.inf, -np.inf], np.nan).dropna()
    if len(expected) < n_bins or len(actual) < n_bins:
        return np.nan
    quantiles = np.unique(np.quantile(expected, np.linspace(0.0, 1.0, n_bins + 1)))
    if len(quantiles) <= 2:
        return np.nan
    expected_bins = pd.cut(expected, bins=quantiles, include_lowest=True, duplicates="drop")
    actual_bins = pd.cut(actual, bins=quantiles, include_lowest=True, duplicates="drop")
    expected_pct = expected_bins.value_counts(normalize=True, sort=False).replace(0.0, 1e-6)
    actual_pct = actual_bins.value_counts(normalize=True, sort=False).reindex(expected_pct.index).fillna(1e-6).replace(0.0, 1e-6)
    return float(((actual_pct - expected_pct) * np.log(actual_pct / expected_pct)).sum())

feature_drift_rows = []
for column in tqdm(X_selection.columns, desc="Computing feature drift PSI", unit="feature"):
    selection_values = X_selection[column]
    holdout_values = X_holdout[column]
    feature_drift_rows.append(
        {
            "Feature": column,
            "Selection Mean": selection_values.mean(),
            "Holdout Mean": holdout_values.mean(),
            "Selection Std": selection_values.std(),
            "Holdout Std": holdout_values.std(),
            "PSI": population_stability_index(selection_values, holdout_values, n_bins=10),
        }
    )
feature_drift_summary = pd.DataFrame(feature_drift_rows).sort_values("PSI", ascending=False, na_position="last").reset_index(drop=True)
display(feature_drift_summary.head(20).round(4))
save_artifact_table(feature_drift_summary, "feature_drift_summary.csv")
cleanup_runtime_memory("after_feature_drift")
save_text_artifact("publication_feature_drift_summary.txt", feature_drift_summary.head(30).round(6).to_string(index=False), description="Plain-text top feature drift diagnostics.")

chronological_split_ordered = (
    selection_df["date"].max() < calibration_df["date"].min()
    and calibration_df["date"].max() < holdout_df["date"].min()
)
feature_name_leakage_flags = [column for column in X_selection.columns if column.startswith("target_") or column.startswith("forward_") or "next_" in column.lower()]
price_window_has_forward_gap = holdout_df["date"].max() < model_frame["date"].max() or model_frame["forward_1m_return"].notna().all()

if "fred_series_availability_summary" in globals() and len(fred_series_availability_summary) > 0:
    fred_included_count = int(fred_series_availability_summary["include_in_features"].sum())
    fred_excluded_count = int((~fred_series_availability_summary["include_in_features"]).sum())
    fred_excluded_names = fred_series_availability_summary.loc[~fred_series_availability_summary["include_in_features"], "series_id"].tolist()
    fred_availability_status = "review" if fred_excluded_count else "pass"
    fred_availability_evidence = f"included={fred_included_count}; excluded={fred_excluded_count}; excluded_series={fred_excluded_names}"
else:
    fred_included_count = 0
    fred_excluded_count = 0
    fred_availability_status = "not_available"
    fred_availability_evidence = "fred_series_availability_summary was not available"

leakage_checks = pd.DataFrame(
    [
        {"Check": "Target columns excluded from model features", "Status": "pass" if not feature_name_leakage_flags else "fail", "Evidence": f"flagged_feature_columns={feature_name_leakage_flags[:10]}; final_feature_count={X_selection.shape[1]}"},
        {"Check": "Chronological split order", "Status": "pass" if chronological_split_ordered else "fail", "Evidence": f"selection_max={selection_df['date'].max().date()}, calibration_min={calibration_df['date'].min().date()}, calibration_max={calibration_df['date'].max().date()}, holdout_min={holdout_df['date'].min().date()}"},
        {"Check": "Feature policy fitted before calibration and holdout", "Status": "pass", "Evidence": f"feature medians and retained columns fitted on selection window through {TUNING_END_DATE}"},
        {"Check": "Next-month target uses future prices only as label", "Status": "pass", "Evidence": "forward_1m_return and target_top_k_next_1m are excluded from feature columns and used only for training/evaluation labels"},
        {"Check": "Month-end signal and next-month execution convention", "Status": "documented_assumption", "Evidence": f"features use month-end close observations; selected execution_return_mode={EXECUTION_RETURN_MODE}; close-to-close and next-open returns are both saved for audit"},
        {"Check": "Static identity feature policy", "Status": "pass" if FEATURE_SET_VARIANT in {"ticker_ablated", "identity_ablated"} else "documented_assumption", "Evidence": f"feature_set_variant={FEATURE_SET_VARIANT}; excluded_variant_feature_columns={len(excluded_variant_feature_columns)}"},
        {"Check": "FRED series feature availability", "Status": fred_availability_status, "Evidence": fred_availability_evidence},
        {"Check": "Portfolio top-k policy", "Status": "review", "Evidence": "all score-driven portfolios use fixed top-k selection; no holdout optimization of top-k or transaction cost is performed"},
        {"Check": "Transaction costs included", "Status": "pass", "Evidence": f"transaction_cost_bps={TRANSACTION_COST_BPS}; cost applied to one-way monthly turnover"},
        {"Check": "Turnover-cap sensitivity", "Status": "pass" if len(TURNOVER_CONSTRAINT_CAPS) > 0 else "not_requested", "Evidence": f"turnover_constraint_caps={TURNOVER_CONSTRAINT_CAPS}"},
        {"Check": "Feature-variant sensitivity", "Status": "pass" if FEATURE_VARIANT_SENSITIVITY_ENABLED and len(feature_variant_bundles) > 0 and RUN_FEATURE_SENSITIVITY_XGBOOST_LIGHT else "not_requested", "Evidence": f"variants={FEATURE_SENSITIVITY_VARIANTS}; fitted_variant_bundles={list(feature_variant_bundles)}; gpu_xgboost_enabled={RUN_FEATURE_SENSITIVITY_XGBOOST_LIGHT}; cpu_logistic_enabled={RUN_FEATURE_SENSITIVITY_LOGISTIC}"},
        {"Check": "Selected model-family feature sensitivity", "Status": "pass" if "model_family_feature_sensitivity" in globals() and len(model_family_feature_sensitivity) > 0 else "not_requested", "Evidence": f"variants={MODEL_FAMILY_FEATURE_SENSITIVITY_VARIANTS}; artifact=model_family_feature_sensitivity.csv"},
        {"Check": "Multi-horizon target audit", "Status": "pass" if len(MULTI_HORIZON_MONTHS) > 0 else "not_requested", "Evidence": f"multi_horizon_months={MULTI_HORIZON_MONTHS}; artifact=multi_horizon_target_summary.csv"},
        {"Check": "Benchmark-relative target audit", "Status": "pass", "Evidence": f"benchmark_asset={BENCHMARK_ASSET}; artifact=benchmark_relative_target_summary.csv"},
        {"Check": "Objective-specific retraining", "Status": "pass" if "objective_specific_retraining_summary" in globals() and len(objective_specific_retraining_summary) > 0 else "not_requested", "Evidence": f"objectives={OBJECTIVE_SPECIFIC_TARGETS}; artifacts=objective_specific_retraining_summary.csv, objective_specific_monthly_rank_aggregate.csv, objective_specific_portfolio_summary.csv, objective_specific_horizon_return_monthly.csv"},
        {"Check": "Portfolio benchmark-relative diagnostics", "Status": "pass" if "portfolio_benchmark_relative_summary" in globals() and len(portfolio_benchmark_relative_summary) > 0 else "not_available", "Evidence": "artifact=portfolio_benchmark_relative_summary.csv"},
        {"Check": "Headline portfolio excludes feature-sensitivity reruns", "Status": "pass" if "portfolio_summary" in globals() and len(portfolio_summary) > 0 and not portfolio_summary["Strategy"].astype(str).map(is_feature_sensitivity_model).any() and "feature_sensitivity_portfolio_summary" in globals() else "review", "Evidence": "headline artifact=portfolio_summary.csv; sensitivity artifact=feature_sensitivity_portfolio_summary.csv"},
        {"Check": "Portfolio drawdown and concentration diagnostics", "Status": "pass" if "portfolio_drawdown_summary" in globals() and "portfolio_weight_constraint_summary" in globals() else "not_available", "Evidence": "artifacts=portfolio_drawdown_summary.csv, portfolio_weight_constraint_summary.csv"},
        {"Check": "Ex-ante risk proxy diagnostics", "Status": "pass" if "portfolio_ex_ante_risk_summary" in globals() and len(portfolio_ex_ante_risk_summary) > 0 else "not_available", "Evidence": "artifacts=portfolio_ex_ante_risk_summary.csv, portfolio_ex_ante_risk_monthly.csv"},
        {"Check": "Liquidity, spread, impact, and tax-drag proxy diagnostics", "Status": "pass" if "portfolio_liquidity_impact_proxy" in globals() and "portfolio_tax_drag_sensitivity" in globals() and not ("Status" in portfolio_liquidity_impact_proxy.columns and (portfolio_liquidity_impact_proxy["Status"] == "unavailable").any()) else "not_available", "Evidence": "artifacts=liquidity_feature_summary.csv, portfolio_liquidity_impact_proxy.csv, portfolio_tax_drag_sensitivity.csv"},
        {"Check": "TabICL empirical comparison", "Status": "known_unresolved_by_design" if not RUN_DIRECT_TABICL else "enabled", "Evidence": f"RUN_DIRECT_TABICL={RUN_DIRECT_TABICL}; TabICL code remains guarded but disabled for this run per RAM constraints and user instruction"},
        {"Check": "Public market data point-in-time limitations", "Status": "known_limitation", "Evidence": "yfinance/FRED/Cboe public files are reproducible but not equivalent to a licensed point-in-time production data store"},
        {"Check": "Investment interpretation", "Status": "educational_only", "Evidence": "portfolio section is a diagnostic of score utility, not an investment recommendation"},
    ]
)
display(leakage_checks)
save_artifact_table(leakage_checks, "leakage_checks.csv")

if cuda_memory_snapshots:
    cuda_memory_summary = pd.concat(cuda_memory_snapshots, ignore_index=True)
    save_artifact_table(cuda_memory_summary, "cuda_memory_summary_final.csv")

registered_artifact_manifest = pd.DataFrame(artifact_manifest_rows).drop_duplicates(subset=["path", "kind"], keep="last")
if len(registered_artifact_manifest) > 0:
    display(registered_artifact_manifest)
    save_artifact_table(registered_artifact_manifest, "registered_artifact_manifest.csv")

run_closeout = f"""
# Run Closeout

Artifact directory: {ARTIFACT_DIR}
Run mode: {'FAST_MODE smoke test' if FAST_MODE else 'full research run'}
Rows in model frame: {len(model_frame):,}
Months in model frame: {model_frame['month'].nunique():,}
Asset count: {model_frame['asset'].nunique():,}
Feature count: {X_selection.shape[1]:,}
Feature set variant: {FEATURE_SET_VARIANT}
Execution return mode: {EXECUTION_RETURN_MODE}
FRED series included in features: {fred_included_count:,}
FRED series excluded for insufficient selection history or load failure: {fred_excluded_count:,}
Holdout rows: {len(holdout_df):,}
Holdout months: {holdout_df['month'].nunique():,}
Holdout top-k label rate: {y_holdout.mean():.4%}
Successful model rows: {len(allocation_summary):,}
Model errors: {len(model_errors):,}
TabICL enabled: {RUN_DIRECT_TABICL} (known unresolved by design when False)
Feature-variant sensitivity enabled: {FEATURE_VARIANT_SENSITIVITY_ENABLED}
Selected model-family feature sensitivity enabled: {MODEL_FAMILY_FEATURE_SENSITIVITY_ENABLED}
Objective-specific retraining enabled: {OBJECTIVE_SPECIFIC_RETRAINING_ENABLED}
Multi-horizon audit months: {MULTI_HORIZON_MONTHS}
Benchmark asset: {BENCHMARK_ASSET}

Important caveats: TabICL is disabled in this run and remains an unresolved empirical comparison by design; selected model-family feature sensitivity uses fixed GPU XGBoost and direct TabPFN by default and is saved separately from headline allocation outputs. Objective-specific retraining uses fixed GPU XGBoost, target-specific searched GPU XGBoost, and direct TabPFN by default with calibration-window, monthly rank, chronological CV, and horizon-aware score-to-return diagnostics. XGBoost searches use fold-level early stopping on chronological validation blocks to reduce wasted tree-building while preserving the same validation protocol and holdout boundary. These are focused diagnostics, not exhaustive hyperparameter-search reruns of every target, feature policy, and model family. Liquidity, tax, market-impact, ex-ante risk, constrained-allocation, benchmark-relative portfolio, and multi-month objective outputs are stress or interpretive proxies. Use the CSV/TXT/PNG artifacts for offline review because notebook editor output can be truncated.
""".strip()
save_text_artifact("run_closeout.md", run_closeout, description="Final run-level summary and artifact guidance.")


## 8. Save Output Archive

The final cell writes a manifest and creates a ZIP archive for download from Kaggle.

In [ ]:
artifact_file_rows = []
for artifact_path in tqdm(sorted(ARTIFACT_DIR.glob("**/*")), desc="Building artifact manifest", unit="path"):
    if artifact_path.is_file():
        artifact_file_rows.append(
            {
                "path": str(artifact_path),
                "filename": artifact_path.name,
                "suffix": artifact_path.suffix,
                "size_bytes": artifact_path.stat().st_size,
            }
        )

artifact_file_manifest = pd.DataFrame(artifact_file_rows)
if len(artifact_file_manifest) > 0:
    display(artifact_file_manifest)
    save_artifact_table(artifact_file_manifest, "artifact_file_manifest.csv", description="Files present in the artifact directory at archive time.")

import shutil

archive_base_candidates = [
    Path("/kaggle/working/tabpfn_tabicl_tactical_asset_allocation_20260512_v2_outputs"),
    ARTIFACT_DIR.parent / "tabpfn_tabicl_tactical_asset_allocation_20260512_v2_outputs",
]
for archive_base in archive_base_candidates:
    try:
        archive_base.parent.mkdir(parents=True, exist_ok=True)
        archive_path = shutil.make_archive(str(archive_base), "zip", ARTIFACT_DIR)
        print(f"Created {archive_path}")
        break
    except Exception as exc:
        print(f"Could not create archive at {archive_base}.zip: {short_error(exc)}")

print("Notebook workflow complete. Review CSV, TXT, and PNG artifacts for the run record.")
